Title: PRAGMA: Revolut Foundation Model

URL Source: https://arxiv.org/pdf/2604.08649

Published Time: Mon, 13 Apr 2026 00:06:54 GMT

Number of Pages: 21

Markdown Content:
# PRAGMA: Revolut Foundation Model 

Maxim Ostroukhov 1 Ruslan Mikhailov 1 Vladimir Iashin 1

Artem Sokolov 1 Andrei Akshonov 1 Vitaly Protasov 1 Dmitrii Beloborodov 1

Vince Mullin 2 Roman Y. Enzmann 2 Georgios Kolovos 2 Jason Renders 2

Pavel Nesterov 1 Anton Repushko 1

> 1

Revolut Research 2NVIDIA 

Modern financial systems generate vast quantities of transactional and event-level data that encode rich economic signals. This paper presents PRAGMA, a family of foundation models for multi-source banking event sequences. Our approach pre-trains a Transformer-based architecture with masked modelling on a large-scale, heterogeneous banking event corpus using a self-supervised objective tailored to the discrete, variable-length nature of financial records. The resulting model supports a wide range of downstream tasks such as credit scoring, fraud detection, and lifetime value prediction: strong performance can be achieved by training a simple linear model on top of the extracted embeddings and can be further improved with lightweight fine-tuning. Through extensive evaluation on downstream tasks, we demonstrate that PRAGMA achieves superior performance across multiple domains directly from raw event sequences, providing a general-purpose representation layer for financial applications. 

Disclaimer : We report only relative improvements, as absolute metrics are commercially sensitive. All examples are synthetic and not from real production data. Credit Recurrent Trx. LTV Comms Fraud Product Rec.    

> Task-specific model PRAGMA-S (10 M) PRAGMA-M (100 M) PRAGMA-L (1 B)

Figure 1: A single architecture from 10M to 1B parameters that outperforms task-specific models across tasks. 

## 1 Introduction 

Foundation models are general-purpose models trained at scale on broad data distributions and subsequently adapted to a wide variety of downstream tasks (Bommasani et al., 2021). While such models have transformed natural language processing (Devlin et al., 2019; Brown et al., 2020) and computer vision (Kirillov et al., 2023; Caron et al., 2021), their application to multi-source banking user histories remains comparatively underexplored. Modern banks and fintechs accumulate large volumes of data: event streams spanning card and transfer transactions, product usage, in-app navigation, and customer communications, alongside static generalised profile state such as account tenure and 1

> arXiv:2604.08649v1 [cs.LG] 9 Apr 2026

plan. These event streams encode signals relevant to risk management, product analytics, and operations, but they are difficult to model efficiently with off-the-shelf language-model tokenisation and architectures. While serialising structured records as text and feeding them to a standard Transformer is a viable baseline, it inflates sequence lengths considerably because every field name and delimiter becomes several subword tokens. Moreover, numerical values are split into digit fragments that discard magnitude and ordering, both of which are critical for financial reasoning. Together, these limitations make naive text serialisation impractical for the long, heterogeneous user histories common in banking. Multi-source banking user histories differ from text in three ways. First, each event is a variable-length record with mixed categorical, numerical, and free-text fields. Second, histories are long-tailed in length and irregular in time, with strong daily and weekly cycles. Third, practical deployments must operate under strict privacy and regulatory constraints, which limit what can be reported and which features can be used for certain decisions. Because no single off-the-shelf architecture handles all three challenges simultaneously, practitioners default to building task-specific pipelines with extensive feature engineering, making it hard to share statistical strength across domains and products. Prior work addresses isolated slices of this problem. Tabular Transformers such as TabTransformer and FT-Transformer (Huang et al., 2020; Gorishniy et al., 2021) model fixed-schema rows, while sequential recommender models such as SASRec and BERT4Rec (Kang et al., 2018; Sun et al., 2019) operate on item-like interaction histories. Financial foundation models have largely focused on text or generic time-series tokenisation (Yang et al., 2020; Wu et al., 2023; Yang et al., 2023; Jin et al., 2024; Ansari et al., 2024), while newer transaction-ledger models such as nuFormer and TransactionGPT (Braithwaite et al., 2025; Dou et al., 2025) move closer to our setting. However, these models typically ingest a single event source, omit static profile state, and are evaluated on a narrow set of tasks: nuFormer targets product recommendation, while TransactionGPT focuses on anomaly detection and trajectory gener-ation. The literature still lacks a multi-source encoder backbone with explicit profile state that transfers across a broad range of discriminative banking tasks. In this paper, we present PRAGMA, a family of encoder-style foundation models for multi-source banking user his-tories. PRAGMA is pre-trained with masked modelling on a large-scale corpus of user histories that combines multi-source events with static profile state (§2.1). To handle heterogeneity, we apply a key–value–time tokenisation scheme with type-specific value encoding for numerical, categorical, and textual fields (§2.2). The resulting backbone uses two encoder branches for profile state and events whose outputs are fused by a history encoder (§2.3). We choose an encoder-only, bidirectional design because our primary goal is transferable representations for dis-criminative financial tasks, rather than open-ended generation. Masked modelling enables each token to attend to both past and future context (Devlin et al., 2019), which is particularly useful when reconstructing partially observed event records and learning record-level representations from complete histories. After pre-training, PRAGMA can be adapted efficiently in two complementary ways (§3.1). In the embedding probe setting, we freeze the backbone and train a lightweight head on top of the extracted embeddings. In the LoRA fine-tuning setting, we apply Low-Rank Adaptation (LoRA) (Hu et al., 2022) to update only a small fraction of parameters, enabling fast specialisation while keeping most of the backbone shared across tasks. We evaluate PRAGMA on a suite of internal downstream benchmarks spanning credit scoring, fraud detection, com-munication engagement, recurrent transaction detection, lifetime value prediction, and more (§3.2). Across evaluated domains, PRAGMA consistently outperforms strong task-specific baselines while reducing the need for hand-crafted features (Figure 1). We further describe the engineering choices required to train PRAGMA efficiently on long and highly variable user histories, including sequence packing and dynamic batching (§2.4). Our contributions are as follows: • We introduce PRAGMA, a family of encoder-style foundation models for multi-source banking user histories, scaling from 10 M to 1 B parameters, to our knowledge, the largest published encoder backbone for consumer banking event sequences. The architecture combines a key–value–time tokenisation scheme with a two-branch design in which profile-state and event encoders feed a history encoder for heterogeneous financial records. • We describe an efficient pre-training recipe for long and irregular banking user histories based on masked modelling, sequence packing, and dynamic batching, and show that LoRA fine-tuning of a pre-trained back-bone consistently matches or outperforms full training from scratch. 2Created  24 �04 �07 19:20:18                                                      

> Type communication
> Channel email
> Product stocks_shares_isa
> Interact interacted
> Created 20 �11 �02 12:09:04
> Type topup
> Direction in
> Amount 100.00
> Currency gbp
> Fee 0.00
> ... ...
> Created 20 �11 �02 12:19:54
> Type card_payment
> Direction out
> Amount 14.99
> Currency gbp
> Fee 0.00
> Description metal plan
> MCC 6012
> ... ...
> Created 22 �12 �25 09:13:48
> Type p2p_transfer
> Direction out
> Amount 150.00
> Currency eur
> Fee 0.00
> ... ...
> Created 22 �12 �25 09:13:40
> Type app_event
> View con �rm_p2p_dialog
> Created 22 �12 �25 09:13:35
> Type app_event
> View p2p_amount
> Created 24 �04 �07 19:24:02
> Type trading
> Direction buy
> Symbol swda
> Amount 100
> Price 74.1945
> Currency gbx
> Order type market
> ... ...
> Region uk
> Plan metal
> Balance 6,012.54
> Currency gbp
> Age 25–35
> Device iphone18,2
> ... ...
> Evaluation Point
> Created 25 �07 �22 13:49:29
> Type app_event
> Screen junior_transfer

Figure 2: Event timeline overview . After account creation, users generate a sequence of platform interactions over time, spanning transactions, in-app navigation, and communications. We aggregate the event history up until a designated evaluation point. Alongside these sequential events, we capture contextual attributes that describe the record’s state at that point, e.g., membership plan or service region. Both events and attributes share a uniform representation: a timestamp and a set of key–value pairs (e.g., Type: card_payment , Channel: email ). All values shown are synthetic; the figure is for illustration purposes only. • We evaluate a single pre-trained backbone across six diverse downstream tasks (credit scoring, fraud detec-tion, lifetime value, communication engagement, recurrent transaction detection, and product recommenda-tion), a substantially broader task scope than prior transaction-ledger models, which typically target one or two tasks. PRAGMA consistently outperforms strong task-specific baselines while reducing the need for hand-crafted features. 

## 2 Pre-training 

2.1 Dataset 

Our goal is to build a foundation model that encodes diverse event-level signals and transfers across a wide range of downstream tasks. Our dataset is structured at the record level, where each observation represents a pseudonymised event history associated with an evaluation point. As shown in Figure 2, we consider an event history alongside contextual attributes. This approach enables the model to account for both sequential patterns and time-invariant features like account currency. All data used in this work is fully anonymised and contains no personally identifiable information. We construct our pre-training dataset from 26 M user records spanning 111 countries, accumulating 24 B events that total 207 B tokens. 

2.1.1 Event History 

Standard platform usage generates event streams across various services, e.g., account funding, payments, in-app navigation, or service communications. These aggregated event histories capture population-level patterns that support a range of analytical and predictive tasks. An event is defined by a created timestamp and a set of key–value pairs, e.g., Direction: out . We fetch events from broad source types that can be loosely grouped into transactions, app, trading, and communication, which were selected for their high expected impact on downstream tasks. Event schemas are specific to their source type and incorporate distinct sets of keys, e.g., Symbol key is unique to trading events. Beyond anonymisation, de-identification, and standard eligibility criteria, no additional statistical filtering or 3pre-processing, such as outlier removal or vocabulary pruning, is applied to the event streams, to ensure that the model captures the full heterogeneity found in production. 

2.1.2 Profile State 

In addition to the event history, we incorporate general contextual attributes such as balance quantile, plan, insurance state, and service region. These attributes provide useful context that is otherwise missing from the event history alone. Profile state is a set of descriptive key–value pairs in an event-like format, e.g., Plan: metal , timestamped at the designated evaluation point (or the cut-off date during pre-training). High-activity users often generate tens of thousands of interactions, exceeding computational bounds; we address this via truncation to a fixed context window (§2.3.5). However, truncation risks discarding early historical milestones that carry useful signal, such as account age. We therefore augment profile state with life-long events , key–value pairs that, unlike regular profile attributes, each carry an individual timestamp recording a first occurrence, e.g., Lifelong: first_topup at 20-11-02 12:09:04 . This timestamp is then used to compute the temporal distance to the evaluation point, enabling the model to encode the timing of historical milestones. 

2.1.3 Pre-training Time Range 

Developing a robust and generalisable model requires a delicate balance between maximising historical coverage and maintaining data relevance. Accordingly, determining the optimal temporal range for pre-training involves navigating several trade-offs between event diversity, distribution shift, and computational efficiency. First, simply including every event from the full available dataset is often impractical and sub-optimal. Older events may reflect historical patterns, product features, or system dynamics that are no longer relevant at inference time. Such discrepancies create a distribution mismatch that can degrade performance, as the model may struggle to generalise from obsolete historical examples to the evolving behaviours present in deployment. Additionally, the inclusion of highly heterogeneous events from long time spans can make the pre-training task harder and slow down model con-vergence. Second, downstream applications may require making predictions on events that took place within temporal ranges either much earlier or much later than those used for pre-training. If the model is not exposed to sufficient diversity in both recent and less-common historical patterns, the performance on these out-of-distribution inputs may suffer. Finally, Transformer architectures have a limited effective context span, determined both by model design and hardware constraints. With these considerations in mind, we select a temporal range of 25 months from 2023 to 2025 for pre-training, balancing comprehensive event coverage, recency, distribution consistency, and tractable sequence modelling. 

2.2 Tokenisation 

Unlike standard LLMs that treat everything as text, a financial foundation model needs to preserve the structural nature and heterogeneity of tabular data. We address this challenge by implementing a disentangled embedding space of input tokens. As shown in Figure 3, we represent each data point by three components: a semantic type (key), a value, and a temporal coordinate, following a common standard in tabular event data (Braithwaite et al., 2025). For instance, Channel: email at 24-04-07 19:20:18 maps to a key, a value, and a temporal coordinate, respectively. This ensures that the model distinguishes between the meaning of a field and its value, while also encoding event chronology. Next, we present how the three are tokenised. 

Semantic Type (Key). The semantic type embedding enables the model to learn the meaning of a field and to contextualise the value it holds. We tokenise all semantic types (keys) as single tokens, and both event and profile state semantic types are encoded in a similar way. This results in a vocabulary of ∼60 tokens. 

Value. We cover the diversity of values with three value types: numerical , categorical , and textual . Numerical values are mapped to percentile buckets, where bin boundaries are learned from training data with an extra bucket for zero, allocating one token per bucket. The distinction between categorical and textual is determined by cardinality thresholding: string fields whose number of unique values falls below a predefined threshold are treated as categorical, 4Created  20 �11 �02 12:19:54                                            

> Type card_payment
> Direction out
> Amount 14.99
> Currency gbp
> Fee 0.00
> Description metal plan
> MCC 6012
> ... ...
> 20 �11 �02 12:19:54 Type Direction Amount Currency Fee Description MCC
> card_payment out 14.99 gbp 0.00 6012
> perc_24 perc_0
> category category number category number text category
> 27021 2892 7688 2259 7645 26965
> 6148 5729 9153 11847 9740 8964 739
> Keys
> Values
> 18.95
> log-seconds
> to the last event
> 12 12
> Timestamp
> day
> hour
> week
> day
> mth
> day
> metal plan
> met plan
> 736 1146
> al
> 2687

Figure 3: Tokenisation overview . A raw event record is decomposed into a temporal coordinate, semantic types (keys), and values. Keys are always represented by one token, while values use type-specific tokenisation: numerical values are bucketised by percentile, categorical values map to a single token, and textual values are split into subword tokens. Some keys therefore expand to multiple value tokens, e.g., Description → met , al , plan . Time is encoded both as log-seconds to the last event and as calendar and time features derived from the timestamp. Profile state is encoded similarly to an event record. while higher-cardinality fields are treated as textual. Categorical values are manually selected from all text fields to prevent splitting common values, such as merchant category codes (MCC), into multiple tokens, and are represented as a single token as well. For textual fields, values are tokenised with a BPE-style subword tokeniser (Sennrich et al., 2016) with a reserved [UNK] token for rare unseen fragments. In total, values allocate a vocabulary of ∼28 k tokens. 

Temporal Information. We encode time in two ways. First, we compute the elapsed time since the most recent event, measured in seconds. We then apply a soft logarithmic transformation, 8 · ln(1 + t/ 8) , to compress the dynamic range of life-long events while preserving high-resolution linear granularity for recent events. This prevents aliasing in positional embeddings caused by extreme temporal gaps without sacrificing the precision of local event sequenc-ing. Second, to capture daily and weekly temporal cycles, we additionally decompose each event timestamp into its cyclical constituents: hour of day, day of week, and day of month, and embed them using periodic functions similar to Gorishniy et al. (2022), but with periods fixed to the known calendar cycles rather than learned. Calendar features are applied only to event-history entries, as cyclical patterns are less relevant for one-off life-long events where the log-seconds encoding already captures the relevant temporal signal. 

2.3 Model Architecture 

PRAGMA is an encoder-only Transformer that inputs an event history along with contextual attributes and outputs dense record-level embeddings. It is trained on a large-scale, diverse dataset with a masked modelling (MLM) objec-tive that reconstructs masked input tokens. Once pre-trained, it acts as a backbone for downstream adaptation with small-scale (2–4 % of the model’s parameters) fine-tuning for a variety of tasks. An overview of PRAGMA is shown in Figure 4. PRAGMA is parametrised as a family of models with 10 M, 100 M, and 1 B parameters, enabling selection according to operational budget and constraints. The details of the architecture family are provided in Table 1. All size variants use GELU activations (Hendrycks et al., 2016), pre-norm layer normalisation (Xiong et al., 2020), and dropout of 0.1 (Srivastava et al., 2014). The model consists of three main blocks: Profile State Encoder, Event Encoder, and History Encoder. First, the profile state tokens are processed by the Profile State Encoder. Second, similar to profile state, each event is encoded independently in the Event Encoder. Finally, the outputs of the Profile State and Event Encoders are concatenated and encoded in the History Encoder to form an output. Depending on the stage, the final output is used either in an MLM head during pre-training, a classification head during fine-tuning, or as-is in an embedding probe. 5USR      

> Emb
> USR
> Emb
> Pro �le State
> Encoder
> RoPE
> EVT
> Emb
> EVT
> Emb
> Event Encoder
> EVT
> EVT
> EVT
> EVT
> DATE
> DATE
> DATE
> EVT
> EVT
> USR
> EVT
> USR
> EVT
> EVT
> EVT
> Proj
> History Encoder
> RoPE
> Time to
> Last Event
> Values
> Keys
> Timestamps
> Time since
> Life-long Events
> Values
> Keys
> Event #1 Event #2 Event #3
> Events
> Pro �le State

Figure 4: PRAGMA backbone overview . Each user record is represented as an ordered event history and profile state, where every field is decomposed into a semantic type (key), one or more values, and a temporal coordinate. Keys and values are embedded from a shared lookup table, and value tokens receive within-field positional embeddings. A 

Profile State Encoder maps profile state xa, with time since life-long events ta encoded via RoPE, into a [USR] 

embedding za, while an Event Encoder independently maps the tokens of each event xe into a [EVT] embedding z′

> e

and adds calendar features zt. A History Encoder then contextualises the sequence z = [ za : ze] with time to the last event te encoded via RoPE, producing a representation for a user record zh.

Width Depth Model Params dmodel dffn Profile Event History Heads 

PRAGMA-S 10 M 192 768 1 5 2 3PRAGMA-M 100 M 512 2048 3 16 6 8PRAGMA-L 1 B 1024 4096 9 45 18 16 Table 1: PRAGMA model family . PRAGMA scales across three variants (10 M, 100 M, 1 B parameters) by jointly increasing model width ( dmodel , dffn ), depth of the profile-state, event, and history encoders, and the number of attention heads. 

2.3.1 Token Embedding 

Profile state and event tokens are embedded identically. For multi-valued fields (e.g., Description ), the key token is replicated to match each of its values, yielding n key–value pairs in total. A single shared embedding table E maps each key and value to a d-dimensional vector; the two embeddings are summed and augmented with static sine/cosine positional encodings (PosEmb) (Vaswani et al., 2017): 

x = PosEmb �E(k) + E(v), x ∈ Rn×d. (1) Positions index values within a field, not across fields—e.g., the value eur of Currency receives position 0, while the three value tokens (met, al, plan) of Description receive positions (0, 1, 2) (see Figure 3). We denote user and event embeddings as xa ∈ Rna×d and xe ∈ Rne×d, respectively. Following common practice in encoder-only Transformers (Devlin et al., 2019; Dosovitskiy et al., 2021), a learnable [USR] (or [EVT] ) token is prepended to each sequence (Figure 4). 

2.3.2 Profile State Encoder 

The Profile State Encoder is a bidirectional Transformer. It inputs the profile state tokens xa ∈ Rna×d and correspond-ing temporal coordinates ta ∈ Rna , where each entry holds the log-seconds since the corresponding life-long event (or 0 for non-life-long pairs). We use RoPE (Su et al., 2024) to encode ta. We disentangle this positional embedding 6from the value-level positional embedding discussed in §2.3.1 to avoid the semantic and scale mismatch. The output is a sequence of profile state embeddings za ∈ Rna×d. We pass the first element, which corresponds to the [USR] 

token, to the History Encoder—we refer to it as za ∈ R1×d for simplicity. 

2.3.3 Event Encoder 

The Event Encoder is a bidirectional Transformer, similar to the Profile State Encoder. It inputs an event history 

xe = ( xe, 1, x e, 2, . . . , x e,n e ), where each element has a distinct number of token embeddings ( xe,i ∈ Rni×d), and processes each event independently of all other events in the history. The module outputs a token-level embedding sequence for each event, denoted bze, which is used by the MLM head during pre-training. Similar to the Profile State Encoder, we select the first token corresponding to the [EVT] token for each event as its aggregated representation 

z′ 

> e

∈ Rne×d.The calendar features (hour of day, day of week, and day of month) xt ∈ Rne×3 are converted to sine and cosine radians and embedded with two MLP layers into zt ∈ Rne×d. Next, the embedded calendar features are added to the Event Encoder output: ze = z′ 

> e

+ zt.

2.3.4 History Encoder 

The History Encoder is a bidirectional Transformer, similar to the other two encoders. It inputs the concatenated aggregated representations of profile state and the calendar-augmented events: z = [ za : ze] ∈ R(1+ ne)×d, as well as the corresponding temporal coordinate te ∈ R1+ ne , where each entry holds the log-seconds to the most recent event in the history ( 0 for the za position). Similar to the Profile State Encoder, RoPE is used to encode positional information. The output is a sequence of embeddings zh ∈ R(1+ ne)×d, where zh, 0 corresponds to [USR] and zh, 1, . . . , z h,n e to the [EVT] tokens. zh is used by the MLM head during pre-training and for downstream probes. 

2.3.5 Training Pre-training Objective. PRAGMA is pre-trained with an MLM objective following BERT (Devlin et al., 2019) where a random subset of event input tokens is masked, and the model reconstructs the original tokens. For each masked token, the MLM head receives the concatenation of three d-dimensional vectors: the Event Encoder output at that token’s position within bze, providing local within-event context; the History Encoder output at the corresponding 

[EVT] position zh,i , providing cross-event context; and the History Encoder output at the [USR] position zh, 0,providing user-level context. This 3d-dimensional representation is projected back to d dimensions and matched against the embedding table to produce logits. The training loss is cross-entropy with label smoothing (Szegedy et al., 2016). 

Masking Strategy. The masking strategy combines three sources: standard individual token-level masking (with 15 % probability), event-level masking (10 %) that requires the model to reconstruct an entire event, and semantic-type (key)-level masking (10 %) where all values of the selected keys are masked, training the model to predict values given context and a key. During pre-training, a small fraction of selected positions are replaced with [UNK] rather than 

[MASK] . Because [UNK] positions are excluded from the MLM objective, they receive no gradient and effectively act as a form of input dropout, training the model to recover original values under a stronger corruption scheme and reducing reliance on the presence of [MASK] , which does not occur at inference time. 

Downstream Adaptation. PRAGMA supports two modes of downstream adaptation. In the embedding probe mode , the record-level representation produced by the History Encoder is extracted as a frozen feature vector, and a lightweight linear probe is trained on top. In the LoRA fine-tuning mode , a small fraction ( ∼2–4 %) of model weights (the attention and feed-forward projections) are updated via Low-Rank Adaptation (Hu et al., 2022), keeping the pre-trained backbone mostly frozen and reducing the risk of catastrophic forgetting. 72.4 Training Infrastructure 

Pre-training PRAGMA on 207 B tokens spanning 24 B user events introduces several engineering challenges. The heterogeneous, table-structured nature of the data requires specialised storage, batching, and truncation strategies. We describe each in turn below. 

Data Storage. The pre-training corpus is stored as a two-level structure: a user index (an LMDB-backed key-value store mapping each user to their tokenised profile state and per-user token statistics) and a collection of event shards 

(Parquet files partitioned by event count, so each file contains only users with the same number of events). This layout allows workers to stream event shards independently and look up profile state on demand. 

Batching. Each training sample consists of a complete event history together with its associated profile state tokens. Because event histories vary greatly in length, from a handful of events to thousands, naïve padding-based batching would waste the majority of compute on padding tokens. Sharding records by event count avoids many random-access disk operations during loading and yields uniform-length event sequences within each batch, so the History Encoder operates on a rectangular tensor without ragged or padded dimensions. We employ dynamic batching with a fixed token budget that fits into GPU memory: records from the same shard are greedily packed until the budget is reached. 

Sequence Packing. Within a batch, individual events still vary in their number of tokens. Rather than padding every event to the longest one, we pack all event tokens into a flat buffer and process them with a variable-length (varlen) attention kernel (Dao et al., 2022), so tokens from different events do not attend to each other at this stage. Together with shard-based batching, this eliminates padding overhead along both the event and token axes. Compared to a padded baseline, sequence packing coupled with dynamic batching yields a 2–5× throughput improvement, depending on the sequence length distribution in the dataset. 

Truncation. To bound memory consumption at a fixed context length, we apply two levels of truncation before packing. At the event level , each individual event is truncated to at most 24 tokens, affecting only 0.01 % of events. At the profile state level , the static profile state sequence is truncated to at most 200 tokens. Users with zero events are discarded; users with more than 6,500 events are subsampled by retaining the most recent ones, preserving temporal recency. 

Pre-training Compute. The three model variants were trained with bf16 mixed precision and the Muon optimiser combined with AdamW (Loshchilov et al., 2019; Jordan, 2024; Liu et al., 2025). PRAGMA-S (10 M parameters) and PRAGMA-M (100 M) were trained on 16 × NVIDIA H100 GPUs, and PRAGMA-L (1 B) on 32 × NVIDIA H100 GPUs. The smallest variant converged in approximately 2 days, while the 100 M and 1 B models each required roughly 2 weeks of wall-clock time. 

## 3 Evaluation 

For commercial sensitivity reasons, we do not report absolute downstream metrics and instead express all results as relative changes with respect to a task-specific reference. Throughout the paper, relative performance is computed as 

(x/ baseline − 1) % , where x is the score of the evaluated method. 

3.1 Evaluation Protocol 

We evaluate PRAGMA primarily via embedding probes and Low-Rank Adaptation (LoRA) (Hu et al., 2022) fine-tuning on downstream tasks. 

3.1.1 Embedding Probing 

Embedding probing facilitates rapid iteration during experimentation before committing to LoRA fine-tuning, e.g., to gauge whether a new feature brings the expected gain, to select a checkpoint after a pre-training run for further evaluation, or to determine whether it is worth exploring a task as a downstream target at all. The embeddings are extracted from the History Encoder output ( zh). 8For our probing analysis, we evaluate the [USR] token, the final [EVT] token, and a combination of both, using a standard linear probe. Given a downstream task with predefined train, validation, and test partitions, we first for-ward each record through the frozen encoder to obtain fixed-size representations and then train a linear probe (logistic or linear regression) on the training partition. We observe that probe performance is robust to the choice of hyper-parameters, so fitting a probe typically takes a couple of minutes. Since our architecture is inherently “pre-norm”, the embeddings were standard-scaled prior to probe fitting. We found that training the probe with the L-BFGS opti-miser (Liu et al., 1989) yields the best results and converges quickly. We note that while Gradient Boosted Decision Trees (GBDT) perform well on lower-dimensional embeddings (e.g., 

192 -d), the requirement for per-task hyper-parameter tuning and the increased time-to-fit make them less practical than linear probing for high-velocity model evaluation. 

3.1.2 Downstream Adaptation with LoRA 

To specialise the PRAGMA backbone for downstream tasks, we employ Low-Rank Adaptation (LoRA), which in-troduces a minimal parameter overhead of only 2–4 %. In this setup, the pre-trained weights are fine-tuned for task-specific objectives to bridge the gap between general representation learning and downstream requirements. We apply LoRA to QKV projections and MLP layers within encoder layers, following a common practice (Hu et al., 2022; Dettmers et al., 2023), and default to rank = 8 with α = 8 across all experiments, but also sweep the rank across 

{4, 8, 16 } on smaller datasets. We use the Adam optimiser (Kingma et al., 2015) for LoRA fine-tuning, and training typically uses 1/8 of the wall-clock time used during pre-training, converging in 12 hours to a few days depending on the dataset size. 

3.1.3 Preparing Downstream Datasets 

For each downstream task, we obtain a unique identifier, which typically consists of a profile id and an evaluation point. Next, we gather the event history and profile attributes directly preceding the evaluation point. We follow the pre-defined folds and splits for each downstream task. The downstream dataset collection process mirrors that of the pre-training dataset. 

3.2 Downstream Tasks Credit Scoring. The task is to assess credit risk for retail applications by predicting the probability of default within the first 12 months of use. The downstream dataset spans multiple years and is diverse across records. This task is cast as a binary classification problem with a minority class, and performance is measured with ROC-AUC and PR-AUC offline metrics. 

Communication Engagement. The task is to predict whether a user who abandoned a credit application mid-process will open a re-engagement communication. This action serves as an upper-funnel proxy for resuming the application and eventually originating a loan. A distinguishing aspect of this task is the severely limited sample size, requiring the model to capture nuanced event-level signals from minimal data. This task is formulated as a binary classification problem, and the main offline metrics are ROC-AUC and PR-AUC. 

External Fraud. This task is a representative fraud detection use case formulated as a binary classification problem. Performance is evaluated using precision and recall as the primary offline metrics. 

Product Recommendation. The task is to predict which products a user is likely to adopt in the near future, con-ditioned on receiving a specific communication (e.g., email or push notification). A key challenge lies in modelling conversion propensity across multiple products simultaneously while accounting for the contextual influence of the communication. The task is formulated as a multilabel classification problem, where the model outputs independent probabilities of conversion for each product in the portfolio. Performance is evaluated using mean average precision (mAP) as the primary offline metric. 

Recurrent Transactions. This task focuses on predicting whether a given transaction corresponds to a recurring subscription that will repeat in the following month. A key challenge lies in distinguishing true recurring patterns 9Task Metric Baseline (ref.) PRAGMA 

Credit scoring PR-AUC – +130.2 % ROC-AUC – +12.4 % Comm. engagement PR-AUC – +79.4 % ROC-AUC – +20.4 % External fraud Precision – +16.7 % Recall – +64.7 % Product rec. mAP – +40.5 % Recurrent txns F1 – +5.8 % Lifetime value PR-AUC – +1.8 % ROC-AUC – +2.6 % Table 2: PRAGMA significantly outperforms internal task-specific models while sharing most of the parameters across tasks. The relative performance is computed as (PRAGMA /baseline − 1). The large variant with LoRA fine-tuning is used as PRAGMA. from irregular or one-off payments given limited historical signals. The problem is formulated as a binary classification task, and performance is evaluated using macro-averaged F1-score to account for class imbalance and ensure balanced performance across classes. 

Lifetime Value (LTV). The LTV task is to assess the probability of a user generating positive gross profit, and is formulated as a binary classification problem. A distinguishing aspect of the LTV dataset is that users have shorter event histories, e.g., a couple of weeks, while the prediction horizon is typically 6 months or more. The main offline metrics are ROC-AUC and PR-AUC. 

3.3 Main Results 

The results presented in Table 2 demonstrate that PRAGMA consistently outperforms existing task-specific baselines across nearly all evaluated domains, despite sharing most of its parameters across tasks. The most striking improve-ments are observed in precision-recall metrics for high-impact tasks: PR-AUC increased by 130.2 % in Credit Scoring and 79.4 % in Communication Engagement, suggesting that PRAGMA is exceptionally effective at identifying low-frequency, high-value signals where traditional models struggle. While ROC-AUC gains are more tempered, they remain substantial at +12.4 % and +20.4 % for the same tasks, respectively. Although performance is more compa-rable on tasks like Lifetime Value and Recurrent Transactions, the overall trend confirms that PRAGMA provides a superior universal representation that matches or exceeds the performance of isolated, task-specific models. 

3.3.1 Effect of Model Scale 

The results in Table 3 illustrate the performance impact of scaling the PRAGMA architecture from the Small (S, 10 M) variant to the Medium (M, 100 M) and Large (L, 1 B) variants. We observe that scaling gains are highly task-dependent, with the most significant improvements concentrated in Credit Scoring, where the Large model achieves a +35.2 % boost in PR-AUC and a +5.8 % gain in ROC-AUC over the Small reference. Notably, the scaling behaviour for Communication Engagement is non-monotonic; the Medium variant exhibits a slight ROC-AUC regression ( −1.8 %), while the Large variant recovers to +0.7 %. For more stable metrics like Recurrent Transactions and LTV, performance gains are more modest, typically remaining under +3.5 %. These results suggest that while increasing parameter count generally enhances predictive power, the Small model already provides a highly competitive representation for transactional and lifetime value predictions, offering a potential efficiency sweet spot for those specific production use cases. 10 PRAGMA Task Metric S (ref.) M L

External fraud Precision – +12.0 % +16.4 % Recall – +24.8 % +23.5 % Product rec. mAP – +18.9 % +27.0 % Credit scoring PR-AUC – +16.3 % +35.2 % ROC-AUC – +3.6 % +5.8 % Lifetime value PR-AUC – +1.5 % +3.0 % ROC-AUC – +1.7 % +3.4 % Comm. engagement PR-AUC – +0.1 % +1.6 % ROC-AUC – −1.8 % +0.7 % Recurrent txns F1 – +0.6 % +0.4 % Table 3: Model performance scales with parameter count . The performance is relative to PRAGMA-S fine-tuned with LoRA and computed as (model /PRAGMA-S − 1). 

PRAGMA-M Task Metric Scratch (ref.) LoRA 

Comm. engagement PR-AUC – +18.6 % ROC-AUC – +5.0 % Credit scoring PR-AUC – +13.0 % ROC-AUC – +1.6 % Product rec. mAP – +10.3 % Recurrent txns F1 – +0.6 % Lifetime value PR-AUC – +0.4 % ROC-AUC – +0.3 % Table 4: Performance comparison of LoRA fine-tuning against task-specific models trained from scratch. Rel-ative performance is computed as (LoRA /Scratch − 1). LoRA consistently matches or exceeds the performance of full-parameter training from scratch. 

3.3.2 Effect of Pre-training 

The results in Table 4 validate our approach, demonstrating that LoRA fine-tuning consistently matches or exceeds the performance of full-parameter training from scratch across all evaluated tasks. The largest gains are observed in Communication Engagement, where LoRA achieves +18.6 % in PR-AUC and +5.0 % in ROC-AUC, suggesting that the pre-trained PRAGMA backbone captures rich diverse event patterns that are difficult to learn when training a model from scratch on a single downstream task. Credit Scoring follows a similar pattern, with LoRA yielding a +13.0 % improvement in PR-AUC and a +1.6 % lift in ROC-AUC. Product Recommendation also benefits substantially, with a +10.3 % gain in mAP. For Recurrent Transactions and Lifetime Value, the improvements are more modest (+0.6 % F1,and +0.4 % / +0.3 % PR-AUC / ROC-AUC respectively), indicating that the scratch-trained baselines already capture most of the task-relevant structure for these objectives, and LoRA fine-tuning maintains parity without regression. These findings are particularly significant for production environments, as they confirm that PRAGMA can consolidate multiple independent, high-maintenance models into a single shared system without sacrificing predictive accuracy, while maintaining a significantly smaller trainable parameter footprint. 11 PRAGMA-S PRAGMA-M PRAGMA-L Task Metric Emb. LoRA Emb. LoRA Emb. LoRA 

Product rec. mAP – +57.2 % – +68.4 % – +68.1 % External fraud Precision – +30.8 % – +29.8 % – +23.8 % Recall – +27.4 % – +24.5 % – +13.3 % Comm. engagement PR-AUC – +72.9 % – +49.7 % – +54.1 % ROC-AUC – +16.9 % – +11.2 % – +13.5 % Credit scoring PR-AUC – +18.0 % – +20.4 % – +10.3 % ROC-AUC – +0.2 % – +2.4 % – +1.5 % Recurrent txns F1 – +4.5 % – +3.2 % – +2.3 % Lifetime value PR-AUC – +3.6 % – +2.4 % – +2.9 % ROC-AUC – +4.7 % – +3.4 % – +3.9 % Table 5: Relative improvement of LoRA-tuned models over embedding-only baselines across scales. For each model size (S, M, L), the embedding-only variant is used as the reference (Emb). Performance gains are computed as (LoRA /Emb − 1). 

3.4 Additional Experiments and Ablations 3.4.1 Effect of Low-Rank Adaptation 

As shown in Table 5, across all evaluated tasks and model scales, the LoRA-tuned variants consistently outperform the embedding-only baselines, demonstrating the efficacy of parameter-efficient fine-tuning in capturing task-specific nuances that fixed embeddings may miss. The most substantial improvements are observed in Communication En-gagement, where LoRA delivers a remarkable +72.9 % gain in PR-AUC for the Small model and maintains significant leads in the Medium and Large variants. In Credit Scoring, we see a peak relative improvement of +20.4 % in PR-AUC for the Medium model, suggesting that LoRA layers are particularly effective at this scale for complex classification. Gains in Recurrent Transactions and LTV are more modest, typically ranging from +2.3 % to +4.7 %. 

3.4.2 Effect of Profile State 

Table 6 isolates the contribution of the Profile State Encoder (§2.3) by comparing the full PRAGMA-S model against a variant that removes the profile-state branch entirely, relying solely on event-level representations. The impact is strongly task-dependent. Credit Scoring benefits substantially, with a +31.8 % relative gain in PR-AUC and +4.9 % in ROC-AUC. The outsized PR-AUC improvement indicates that profile state is particularly valuable for identifying the minority default class, where static signals such as account tenure and onboarding characteristics provide discrim-inative context that event sequences alone cannot fully capture. In contrast, Lifetime Value shows more moderate gains of +2.2 % in PR-AUC and +2.0 % in ROC-AUC, suggesting that gross-profit likelihood is largely inferable from transactional patterns over the prediction horizon. Communication Engagement exhibits a slight PR-AUC regression (−3.0 %) alongside a marginal ROC-AUC gain (+1.3 %), indicating that re-engagement propensity is driven almost entirely by pre-drop-off event patterns rather than static user characteristics. These results validate the two-branch design of PRAGMA: the dedicated Profile State Encoder adds significant value for tasks where static profile state is informative, while the architecture degrades gracefully when those signals are less relevant. 

3.4.3 Communication Engagement (Uplift) 

This task moves beyond conversion prediction to optimal treatment selection: the goal is to identify which messaging strategy best re-engages users with abandoned credit applications. The dataset is smaller in scale than our other downstream benchmarks, yet large-scale pre-training proves decisive, significantly outperforming a baseline trained on the limited in-domain data alone. As an uplift task, it also offers a distinct evaluation angle — PRAGMA is used as a frozen feature extractor feeding a meta-learner rather than being fine-tuned, isolating representational quality in the absence of task-specific adaptation. 12 PRAGMA-S Task Metric Event-only (ref.) Full 

External fraud Precision – +46.8 % Recall – +85.6 % Credit scoring PR-AUC – +31.8 % ROC-AUC – +4.9 % Product rec. mAP – +3.5 % Lifetime value PR-AUC – +2.2 % ROC-AUC – +2.0 % Recurrent txns F1 – +2.4 % Comm. engagement PR-AUC – −3.0 % ROC-AUC – +1.3 % Table 6: Profile state contributes substantially to tasks where static user characteristics are discriminative. The relative performance is computed as (Full /Event-only − 1). 

Task Metric Baseline (ref.) PRAGMA 

Comm. engagement (uplift) AUUC – +163.7 % SNIPS – +10.8 % Table 7: Performance comparison of PRAGMA-L against the internal uplift baseline using the same meta-learner framework. The relative performance is computed as (PRAGMA-L /Baseline − 1). Concretely, we adopt a meta-learner framework (Künzel et al., 2019) to estimate heterogeneous treatment effects, requiring the model to capture complex interactions between pre-drop-off event signals, profile state, and treatment assignment. Both PRAGMA and the baseline use the same meta-learner, differing only in the underlying representa-tion. Table 7 summarises results using Area Under the Uplift Curve (AUUC) and SNIPS (Swaminathan et al., 2015). PRAGMA-L’s ability to capture latent event-level patterns translates to highly effective treatment allocation, achieving a relative AUUC increase of 163.7 % over the internal baseline. 

3.4.4 Effect of a Pre-trained Text Encoder 

In the standard PRAGMA architecture, text values are learned jointly with all other tabular features via an embedding lookup table (see §2.3.1). To prevent the model from underfitting sparse, noisy, or highly irregular financial text (e.g., truncated transaction descriptions), we investigate offloading text comprehension to a dedicated, pre-trained text embedding model, e.g., Nemotron-1B-v2 (de Souza P. Moreira et al., 2024). This decoupled approach provides richer, out-of-the-box semantics and frees the primary Event Transformer (§2.3.3) to focus on cross-feature interactions. While we do not use this as the default formulation in our generalized core architecture, we report on it as an optional extension that offers valuable domain-specific insights. 

Implementation Details. The addition of a pre-trained text encoder involves multiple structural changes to the PRAGMA architecture. First, for semantic types (keys) whose values are normally encoded using a custom-trained BPE tokeniser and a trainable embedding lookup table, we instead use the frozen pre-trained model to map the com-plete text string to a single vector, which is then adapted via a one-layer trainable projection (see Figure 5). Second, instead of reconstructing exact token labels for these text fields during MLM optimisation (see §2.3.5), we train PRAGMA to reconstruct the continuous text embedding produced by the pre-trained text encoder with Mean Squared Error (MSE). 13 Nemotron    

> Emb
> PRAGMA + Nemotron PRAGMA
> Projection
> metal plan
> met plan
> 736 1146
> al
> 2687
> metal plan Description
> 4062
> Emb
> Description
> 4062
> Emb

Figure 5: Text embedding with PRAGMA (left) compared to a version with pre-trained Nemotron-1B-v2 text embedding (right) . Instead of our custom trained BPE tokeniser and a trainable embedding lookup table, a pre-trained “frozen” Nemotron maps an entire text value to a single text embedding vector which is projected into the Transformer’s base dimension with a trainable projection. 

PRAGMA-M Task Metric ref. +Nemotron 

Credit scoring PR-AUC – +16.1 % ROC-AUC – +2.8 % Recurrent txns F1 – +0.1 % Lifetime value PR-AUC – +0.8 % ROC-AUC – +0.6 % External fraud Precision – +3.8 % Recall – −0.7 % Product rec. mAP – −6.4 % Table 8: Impact of pre-trained text embeddings on downstream tasks is concentrated in text-heavy domains. 

The performance is estimated relative to a LoRA-tuned PRAGMA-M. 

Results & Discussion. The results are shown in Table 8. Downstream effects track how much label-relevant signal sits in free text versus categorical and behavioural structure. Credit Scoring shows the clearest upside, with +16.1 % relative PR-AUC and +2.8 % ROC-AUC under Nemotron. Product Recommendation instead loses ground: mAP drops by 6.4 % relative, plausibly because sparse text adds little beyond what the structural channels already encode. External Fraud moves modestly and in opposite directions on precision (+3.8 %) versus recall ( −0.7 %), while LTV and Recurrent Transactions stay near flat on the reported metrics. Because this variant also increases PRAGMA-M training latency by about 18 %, we keep it as an opt-in module for text-heavy tasks rather than baking it into the default architecture. 

3.4.5 Limitations in Highly Relational Tasks: Anti-Money Laundering 

We formulate Anti-Money Laundering (AML) as a binary classification task. As shown in Table 9, this is a setting where PRAGMA significantly underperforms the production baseline. We attribute this performance gap to two primary factors. First, the downstream AML dataset is sufficiently large for the baseline model to learn robust task-specific representations without requiring foundation-level pre-training. Second, and more critically, AML detection is inherently relational: the baseline leverages cross-record features that capture network-level signals. Because PRAGMA processes event histories in isolation, the resulting embeddings do not inherently capture the cross-record dependency structures crucial for this task. 14 Task Metric Baseline (ref.) PRAGMA 

Anti-money laundering F0.5 – −47.1 % Table 9: Performance comparison of PRAGMA against baseline for Anti-Money Laundering. The relative performance is computed as (PRAGMA /Baseline − 1) using linear probe on PRAGMA-L embeddings. Performance is evaluated primarily using F0.5 , as it emphasises precision while still accounting for recall. PRAGMA suffers a 47.1 % drop in F0.5 compared to the network-aware baseline, demonstrating that isolated record-level repre-sentations may be insufficient for this highly relational domain. Addressing this limitation remains a key direction for future work. 

## 4 Related Work 

4.1 Transformer 

The landscape of sequence modelling was fundamentally reshaped by the introduction of the Transformer architec-ture (Vaswani et al., 2017), which dispensed with recurrent layers in favour of a parallelisable self-attention mech-anism. Following this, the field branched out into encoder-only models like BERT (Devlin et al., 2019), optimised for discriminative tasks, and decoder-only architectures like GPT-3 (Brown et al., 2020), which catalysed the cur-rent generative AI era through massive scaling and emergent in-context learning. Subsequent research has extended the architecture’s reach via the Vision Transformer (ViT) (Dosovitskiy et al., 2021) for visual perception and the T5 framework (Raffel et al., 2020) for unified text-to-text processing. Recent advancements have prioritised computa-tional efficiency and multimodality, notably through hardware-aware optimisations like FlashAttention (Dao et al., 2022) and the adoption of Mixture-of-Experts (MoE) (Fedus et al., 2022) in models like Mixtral 8×7B (Jiang et al., 2024). In the current paradigm, models such as Gemini 1.5 (Gemini Team, 2024) and GPT-4o (Hurst et al., 2024) have moved beyond compositional architectures to native multimodality, enabling seamless reasoning across diverse data streams. In this landscape, PRAGMA should be understood as an encoder foundation model for heterogeneous tabular event streams. Although motivated by financial transactions, it extends naturally to any domain where entities accumulate irregular, multi-field records over time. It inherits the scalability and bidirectional contextualisation of encoder-only Transformers, adapting them to heterogeneous fields, explicit time signals, and reusable record-level representations. 

4.2 Masked Modelling 

Parallel to the scaling of generative decoders, masked modelling established a dominant paradigm for self-supervised representation learning. This was pioneered by BERT (Devlin et al., 2019), which utilised a Masked Language Mod-elling (MLM) objective to capture bidirectional context, a technique further refined by RoBERTa (Liu et al., 2019) through dynamic masking and optimised training recipes. The success of MLM was later translated to the vision do-main via Masked Image Modelling (MIM), with BEiT (Bao et al., 2021) and Masked Autoencoders (MAE) (He et al., 2022) demonstrating that reconstructing obscured image patches forces the model to learn holistic structural repre-sentations. Recent trends have moved towards cross-modal unification, as seen in Data2Vec (Baevski et al., 2022), and a shift from raw signal reconstruction to latent feature prediction, exemplified by the Joint-Embedding Predictive Architecture (I-JEPA) (Assran et al., 2023). PRAGMA is directly inspired by this line of work, but extends masked modelling from text and images to hetero-geneous financial records. Our objective masks individual tokens, whole events, and semantic types, encouraging the reconstruction of partially observed events and the learning of transferable representations from full transaction histories. 

4.3 Transformers for Tabular Data 

While Gradient Boosted Decision Trees (GBDTs) have historically dominated structured data, the Transformer has spurred a new class of “Tabular Deep Learning” architectures. Early entries like TabTransformer (Huang et al., 15 2020) and FT-Transformer (Gorishniy et al., 2021) focused on modelling inter-feature dependencies through self-attention, demonstrating performance parity with GBDTs on high-dimensional datasets. This was improved by SAINT (Somepalli et al., 2021), which introduced a dual-attention mechanism for both feature and row interactions, and Trompt (Chen et al., 2023), which proposed prompt-tuning to disentangle intrinsic table properties from sample variations. A paradigm shift occurred with TabPFN (Hollmann et al., 2023), a foundation model pre-trained on syn-thetic data to approximate Bayesian inference. Leveraging in-context learning, TabPFN generates predictions via a single forward pass, eliminating the need for iterative training. While the original model was restricted to 1,000 sam-ples, TabPFN-v2 and TabPFN-v2.5 (Hollmann et al., 2025; Grinsztajn et al., 2025) scaled the architecture to handle 100,000 samples and real-world complexities, providing native support for categorical features, missing values, and outliers. Most recently, Mitra (Zhang et al., 2025) has adopted the dual-attention mechanism of SAINT but follows the foundation model paradigm of TabPFN by being pre-trained exclusively on a massive mixture of synthetic priors. PRAGMA is related in spirit to tabular Transformers because it preserves field identity and models cross-field inter-actions with attention, but unlike TabTransformer, FT-Transformer, and SAINT, it does not operate on a fixed-schema single row. Compared with TabPFN-style tabular foundation models trained on synthetic supervised tasks, PRAGMA is pre-trained with self-supervision on real financial ledgers and models variable-length user histories of heterogeneous events with a hierarchical encoder. 

4.4 Modelling for Recommender Systems 

Sequential recommendation models share structural similarities with transaction modelling, as both process ordered event sequences with rich side information. Transformer-based recommenders treat user interaction histories as token sequences: SASRec (Kang et al., 2018) replaced recurrence with self-attention to capture long-range dependencies, and BERT4Rec (Sun et al., 2019) demonstrated that bidirectional context via masked item prediction yields more robust representations. The field later converged with the LLM paradigm: P5 (Geng et al., 2022) cast diverse rec-ommendation tasks into a unified text-to-text framework built on T5, while TALLRec (Bao et al., 2023) introduced instruction tuning to align general-purpose LLMs with recommendation logic. More recent industrial work has shifted from modelling only positive interactions to encoding richer event streams. Generative Recommenders (Zhai et al., 2024) interleave item and action tokens in a causal sequence, scaling to trillions of parameters with power-law quality gains. ARGUS (Khrylchenko et al., 2025) decomposes autoregressive learning into feedback and next-item prediction, scaling recommender Transformers to one billion parameters. The TransAct line of work (Xia et al., 2023; 2025) embeds each user action as a composite of content, action type, and context for CTR prediction, and extends to lifelong action sequences. PRAGMA is close to this literature in its use of ordered event histories and self-supervised pre-training. Unlike recommendation models that often reduce each interaction to an item token, PRAGMA models richer financial events with typed fields, amounts, free text, and temporal coordinates, and is adapted to a broader set of banking tasks beyond ranking. 

4.5 Foundation Models for Finance 

The paradigm of financial foundation models has rapidly matured from specialised text encoders to comprehensive reasoning engines that integrate diverse data modalities. This evolution began with FinBERT (Yang et al., 2020), which adapted the encoder-only architecture to financial corpora, establishing a rigorous baseline for discriminative tasks like sentiment analysis and ESG classification. The field shifted toward massive generative scale with BloombergGPT (Wu et al., 2023), which demonstrated that interleaving proprietary financial datasets with general web corpora yields superior performance on domain-specific benchmarks. To address the accessibility barriers of such massive models, FinGPT (Yang et al., 2023) introduced a data-centric, lightweight adaptation framework, democratising access to financial LLMs via efficient LoRA fine-tuning (Hu et al., 2022) of open-source models. Most recently, research has transcended textual boundaries to address the structured nature of market data; models like Time-LLM (Jin et al., 2024) and Chronos (Ansari et al., 2024) treat numerical time series as token sequences, enabling Transformers to perform zero-shot forecasting. Extending this structural shift to consumer finance, recent foundation models are now being trained directly on massive-scale user transaction ledgers. For instance, nuFormer (Braithwaite et al., 2025) demonstrates that jointly 16 fusing tokenised transaction sequences with traditional tabular features can effectively replace manual feature en-gineering for real-world risk prediction. Concurrently, TransactionGPT (Dou et al., 2025) introduces a specialised 3D-Transformer architecture to explicitly model the multimodal, temporal, and tabular dimensions of billion-scale payment trajectories, achieving state-of-the-art performance in downstream anomaly detection and trajectory genera-tion. PRAGMA differs from text-centric financial foundation models such as FinBERT, BloombergGPT, and FinGPT, which primarily operate on financial language, and from Time-LLM or Chronos, which tokenise numerical time series for forecasting. It is closer to transaction-ledger models such as nuFormer and TransactionGPT, but aims for a reusable encoder backbone over multi-source banking events with explicit profile state and lightweight adaptation across diverse discriminative tasks. 

## 5 Conclusion 

We presented PRAGMA, a family of encoder-style foundation models for multi-source banking user histories. PRAGMA combines a key–value–time tokenisation scheme with two encoder branches for profile state and events whose outputs are fused by a history encoder, and is pre-trained with masked modelling on large-scale, heterogeneous financial records. Across diverse downstream tasks—credit scoring, fraud detection, communication engagement, product recommendation, recurrent transaction detection, lifetime value prediction, and more—a single pre-trained backbone achieves superior performance directly from raw banking event sequences, providing a general-purpose representation layer for financial applications. Our experiments reveal several practical insights. LoRA fine-tuning consistently matches or exceeds full training from scratch while updating only a small fraction of parameters, confirming that the pre-trained representations transfer effectively across tasks. Scaling from 10 M to 1 B parameters yields large gains on harder tasks such as credit scoring, while smaller models already provide competitive representations for tasks such as lifetime value prediction, offer-ing a practical efficiency trade-off. The dedicated profile state encoder proves particularly valuable for tasks where static contextual attributes are informative, such as credit scoring and fraud detection, while the architecture degrades gracefully when those signals are less relevant. We also find that integrating a pre-trained text encoder improves per-formance in text-dense domains but adds training overhead that is not justified for text-sparse tasks. Finally, the AML case study highlights a clear limitation: tasks that depend on cross-record relational structure remain out of reach for a model that processes event histories in isolation. These results suggest that multi-source banking event sequences admit transferable representations in much the same way as text and vision, despite their heterogeneous structure, irregular timing, and operational constraints. Extend-ing the model to capture cross-record interactions for relational tasks such as anti-money laundering is a promising direction for future work. 

Acknowledgments 

We thank Dmitry Mittov, Ian Iakobsen, Aleksandr Pushin, Muhammad Anas, Viacheslav Karpov, Nathalie Skrzypek, Leyla Sultanova, Francisco Sanz Estevez, Nikita Kravchuk, Tadas Krisciunas, Amey Baokar, Hanna Danilovich, Jyoti Prakash Bal, Vitalii Radchenko, Kade Main, Nic Hatia, and other Revoluters for their contributions to this work. 

## References 

Abdul Fatir Ansari, Lorenzo Stella, Caner Turkmen, Xiyuan Zhang, Pedro Mercado, Huibin Shen, Oleksandr Shchur, Syama Sundar Rangapuram, Sebastian Pineda Arango, Shubham Kapoor, et al. Chronos: Learning the language of time series. Transactions on Machine Learning Research , 2024. Mahmoud Assran, Quentin Duval, Ishan Misra, Piotr Bojanowski, Pascal Vincent, Michael Rabbat, Yann LeCun, and Nicolas Ballas. Self-supervised learning from images with a joint-embedding predictive architecture. In Conference on Computer Vision and Pattern Recognition , 2023. Alexei Baevski, Wei-Ning Hsu, Qiantong Xu, Arun Babu, Jiatao Gu, and Michael Auli. Data2vec: A general frame-work for self-supervised learning in speech, vision and language. In International Conference on Machine Learning ,2022. 17 Hangbo Bao, Li Dong, Songhao Piao, and Furu Wei. BEiT: BERT pre-training of image transformers. arXiv preprint arXiv:2106.08254 , 2021. Keqin Bao, Jizhi Zhang, Yang Zhang, Wenjie Wang, Fuli Feng, and Xiangnan He. TALLRec: An effective and efficient tuning framework to align large language model with recommendation. In ACM Conference on Recommender Systems , 2023. Rishi Bommasani, Drew A. Hudson, Ehsan Adeli, Russ Altman, Simran Arora, Sydney von Arx, Michael S. Bernstein, Jeannette Bohg, Antoine Bosselut, Emma Brunskill, Erik Brynjolfsson, S. Buch, Dallas Card, Rodrigo Castellon, Niladri S. Chatterji, Annie S. Chen, Kathleen A. Creel, Jared Davis, Dora Demszky, Chris Donahue et al. On the opportunities and risks of foundation models. ArXiv , 2021. DT Braithwaite, Misael Cavalcanti, R Austin McEver, Hiroto Udagawa, Daniel Silva, Rohan Ramanath, Felipe Mene-ses, Arissa Yoshida, Evan Wingert, Matheus Ramos, et al. Your spending needs attention: Modeling financial habits with transformers. arXiv preprint arXiv:2507.23267 , 2025. Tom Brown, Benjamin Mann, Nick Ryder, Melanie Subbiah, Jared D Kaplan, Prafulla Dhariwal, Arvind Neelakantan, Pranav Shyam, Girish Sastry, Amanda Askell, et al. Language models are few-shot learners. Advances in Neural Information Processing Systems , 2020. Mathilde Caron, Hugo Touvron, Ishan Misra, Hervé Jégou, Julien Mairal, Piotr Bojanowski, and Armand Joulin. Emerging properties in self-supervised vision transformers. In International Conference on Computer Vision , 2021. Kuan-Yu Chen, Ping-Han Chiang, Hsin-Rung Chou, Ting-Wei Chen, and Darby Tien-Hao Chang. Trompt: Towards a better deep neural network for tabular data. In International Conference on Machine Learning , 2023. Tri Dao, Dan Fu, Stefano Ermon, Atri Rudra, and Christopher Ré. FlashAttention: Fast and memory-efficient exact attention with io-awareness. Advances in Neural Information Processing Systems , 2022. Gabriel de Souza P. Moreira, Radek Osmulski, Mengyao Xu, Ronay Ak, Benedikt Schifferer, and Even Oldridge. Nv-retriever: Improving text embedding models with effective hard-negative mining. arXiv preprint arXiv:2407.15831 ,2024. doi: 10.48550/arXiv.2407.15831. Tim Dettmers, Artidoro Pagnoni, Ari Holtzman, and Luke Zettlemoyer. QLoRA: Efficient finetuning of quantized llms. Advances in Neural Information Processing Systems , 2023. Jacob Devlin, Ming-Wei Chang, Kenton Lee, and Kristina Toutanova. Bert: Pre-training of deep bidirectional trans-formers for language understanding. In North American Association for Computational Linguistics - Human Lan-guage Technologies , 2019. Alexey Dosovitskiy, Lucas Beyer, Alexander Kolesnikov, Dirk Weissenborn, Xiaohua Zhai, Thomas Unterthiner, Mostafa Dehghani, Matthias Minderer, Georg Heigold, Sylvain Gelly, Jakob Uszkoreit, and Neil Houlsby. An image is worth 16x16 words: Transformers for image recognition at scale. In International Conference on Learning Representations , 2021. Yingtong Dou, Zhimeng Jiang, Tianyi Zhang, Mingzhi Hu, Zhichao Xu, Shubham Jain, Uday Singh Saini, Xiran Fan, Jiarui Sun, Menghai Pan, et al. Transactiongpt. arXiv preprint arXiv:2511.08939 , 2025. William Fedus, Barret Zoph, and Noam Shazeer. Switch Transformers: Scaling to trillion parameter models with simple and efficient sparsity. Journal of Machine Learning Research , 2022. Gemini Team. Gemini 1.5: Unlocking multimodal understanding across millions of tokens of context. arXiv preprint arXiv:2403.05530 , 2024. Shijie Geng, Shuchang Liu, Zuohui Fu, Yingqiang Ge, and Yongfeng Zhang. Recommendation as language processing (RLP): A unified pretrain, personalized prompt & predict paradigm (P5). In ACM Conference on Recommender Systems , 2022. Yury Gorishniy, Ivan Rubachev, Valentin Khrulkov, and Artem Babenko. Revisiting deep learning models for tabular data. Advances in Neural Information Processing Systems , 2021. 18 Yury Gorishniy, Ivan Rubachev, and Artem Babenko. On embeddings for numerical features in tabular deep learning. 

Advances in Neural Information Processing Systems , 2022. Léo Grinsztajn, Klemens Flöge, Oscar Key, Felix Birkel, Philipp Jund, Brendan Roof, Benjamin Jäger, Dominik Safaric, Simone Alessi, Adrian Hayler, et al. TabPFN-2.5: Advancing the state of the art in tabular foundation models. arXiv preprint arXiv:2511.08667 , 2025. Kaiming He, Xinlei Chen, Saining Xie, Yanghao Li, Piotr Dollár, and Ross Girshick. Masked autoencoders are scalable vision learners. In Computer Vision and Pattern Recognition , 2022. Dan Hendrycks and Kevin Gimpel. Gaussian error linear units (gelus). arXiv preprint arXiv:1606.08415 , 2016. Noah Hollmann, Samuel Müller, Katharina Eggensperger, and Frank Hutter. TabPFN: A transformer that solves small tabular classification problems in a second. In International Conference on Learning Representations , 2023. Noah Hollmann, Samuel Müller, Lennart Purucker, Arjun Krishnakumar, Max Körfer, Shi Bin Hoo, Robin Tibor Schirrmeister, and Frank Hutter. Accurate predictions on small data with a tabular foundation model. Nature , 2025. Edward J Hu, yelong shen, Phillip Wallis, Zeyuan Allen-Zhu, Yuanzhi Li, Shean Wang, Lu Wang, and Weizhu Chen. LoRA: Low-rank adaptation of large language models. In International Conference on Learning Representations ,2022. Xin Huang, Ashish Khetan, Milan Cvitkovic, and Zohar Karnin. TabTransformer: Tabular data modeling using contextual embeddings. arXiv preprint arXiv:2012.06678 , 2020. Aaron Hurst, Adam Lerer, Adam P Goucher, Adam Perelman, Aditya Ramesh, Aidan Clark, AJ Ostrow, Akila Weli-hinda, Alan Hayes, Alec Radford, et al. GPT-4o system card. arXiv preprint arXiv:2410.21276 , 2024. Albert Q Jiang, Alexandre Sablayrolles, Antoine Roux, Arthur Mensch, Blanche Savary, Chris Bamford, Deven-dra Singh Chaplot, Diego de las Casas, Emma Bou Hanna, Florian Bressand, et al. Mixtral of experts. arXiv preprint arXiv:2401.04088 , 2024. Ming Jin, Shiyu Wang, Lintao Ma, Zhixuan Chu, James Y Zhang, Xiaoming Shi, Pin-Yu Chen, Yuxuan Liang, Yuan-Fang Li, Shirui Pan, and Qingsong Wen. Time-LLM: Time series forecasting by reprogramming large language models. In International Conference on Learning Representations , 2024. Keller Jordan. Muon: An optimizer for hidden layers in neural networks, 2024. URL https://kellerjordan. github.io/posts/muon/ .Wang-Cheng Kang and Julian McAuley. Self-attentive sequential recommendation. In International Conference on Data Mining , 2018. Kirill Khrylchenko, Artem Matveev, Sergei Makeev, and Vladimir Baikalov. Scaling recommender transformers to one billion parameters. arXiv preprint arXiv:2507.15994 , 2025. Diederik P Kingma and Jimmy Ba. Adam: A method for stochastic optimization. In International Conference on Learning Representations , 2015. Alexander Kirillov, Eric Mintun, Nikhila Ravi, Hanzi Mao, Chloe Rolland, Laura Gustafson, Tete Xiao, Spencer Whitehead, Alexander C Berg, Wan-Yen Lo, et al. Segment anything. In Computer Vision and Pattern Recognition ,2023. Sören R Künzel, Jasjeet S Sekhon, Peter J Bickel, and Bin Yu. Metalearners for estimating heterogeneous treatment effects using machine learning. Proceedings of the national academy of sciences , 2019. Dong C Liu and Jorge Nocedal. On the limited memory bfgs method for large scale optimization. Mathematical programming , 1989. Jingyuan Liu, Jianlin Su, Xingcheng Yao, Zhejun Jiang, Guokun Lai, Yulun Du, Yidao Qin, Weixin Xu, Enzhe Lu, Junjie Yan, et al. Muon is scalable for LLM training. arXiv preprint arXiv:2502.16982 , 2025. 19 Yinhan Liu, Myle Ott, Naman Goyal, Jingfei Du, Mandar Joshi, Danqi Chen, Omer Levy, Mike Lewis, Luke Zettlemoyer, and Veselin Stoyanov. RoBERTa: A robustly optimized bert pretraining approach. arXiv preprint arXiv:1907.11692 , 2019. Ilya Loshchilov and Frank Hutter. Decoupled weight decay regularization. In International Conference on Learning Representations , 2019. Colin Raffel, Noam Shazeer, Adam Roberts, Katherine Lee, Sharan Narang, Michael Matena, Yanqi Zhou, Wei Li, and Peter J Liu. Exploring the limits of transfer learning with a unified text-to-text transformer. Journal of Machine Learning Research , 2020. Rico Sennrich, Barry Haddow, and Alexandra Birch. Neural machine translation of rare words with subword units. In 

Annual Meeting of the Association for Computational Linguistics , 2016. Gowthami Somepalli, Micah Goldblum, Avi Schwarzschild, C Bayan Bruss, and Tom Goldstein. SAINT: Improved neural networks for tabular data via row attention and contrastive pre-training. arXiv preprint arXiv:2106.01342 ,2021. Nitish Srivastava, Geoffrey Hinton, Alex Krizhevsky, Ilya Sutskever, and Ruslan Salakhutdinov. Dropout: a simple way to prevent neural networks from overfitting. Journal of Machine Learning Research , 2014. Jianlin Su, Murtadha Ahmed, Yu Lu, Shengfeng Pan, Wen Bo, and Yunfeng Liu. RoFormer: Enhanced transformer with rotary position embedding. Neurocomputing , 2024. Fei Sun, Jun Liu, Jian Wu, Changhua Pei, Xiao Lin, Wenwu Ou, and Peng Jiang. BERT4Rec: Sequential recommen-dation with bidirectional encoder representations from transformer. In International Conference on Information and Knowledge Management , 2019. Adith Swaminathan and Thorsten Joachims. The self-normalized estimator for counterfactual learning. In NeurIPS ,2015. Christian Szegedy, Vincent Vanhoucke, Sergey Ioffe, Jon Shlens, and Zbigniew Wojna. Rethinking the inception architecture for computer vision. In Computer Vision and Pattern Recognition , 2016. Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N Gomez, Łukasz Kaiser, and Illia Polosukhin. Attention is all you need. Advances in Neural Information Processing Systems , 2017. Shijie Wu, Ozan Irsoy, Steven Lu, Vadim Dabravolski, Mark Dredze, Sebastian Ghaffari, Binyam Gebre, Abra-ham Ittycheriah, and Gideon Mann. BloombergGPT: A large language model for finance. arXiv preprint arXiv:2303.17564 , 2023. Xue Xia, Pong Eksombatchai, Nikil Pancha, Dhruvil Deven Badani, Po-Wei Wang, Neng Gu, Saurabh Vishwas Joshi, Nazanin Farahpour, Zhiyuan Zhang, and Andrew Zhai. TransAct: Transformer-based realtime user action model for recommendation at Pinterest. In ACM SIGKDD , 2023. Xue Xia, Saurabh Vishwas Joshi, Kousik Rajesh, Kangnan Li, Yangyi Lu, Nikil Pancha, Dhruvil Deven Badani, Jiajing Xu, and Pong Eksombatchai. TransAct V2: Lifelong user action sequence modeling on Pinterest recommendation. 

arXiv preprint arXiv:2506.02267 , 2025. Ruibin Xiong, Yunchang Yang, Di He, Kai Zheng, Shuxin Zheng, Chen Xing, Huishuai Zhang, Yanyan Lan, Liwei Wang, and Tieyan Liu. On layer normalization in the transformer architecture. In International Conference on Machine Learning , 2020. Hongyang Yang, Xiao-Yang Liu, and Christina Dan Wang. FinGPT: Open-source financial large language models. In 

International Joint Conference on Artificial Intelligence (IJCAI) Symposium on Financial Large Language Models ,2023. Yi Yang, Mark Christopher Siy Uy, and Allen Huang. FinBERT: A pretrained language model for financial commu-nications. arXiv preprint arXiv:2006.08097 , 2020. 20 Jiaqi Zhai, Lucy Liao, Xing Liu, Yueming Wang, Rui Li, Xuan Cao, Leon Gao, Zhaojie Gong, Fangda Gu, Michael He, Yinghai Lu, and Yu Shi. Actions speak louder than words: Trillion-parameter sequential transducers for generative recommendations. arXiv preprint arXiv:2402.17152 , 2024. Xiyuan Zhang, Danielle C. Maddix, Junming Yin, Nick Erickson, Abdul Fatir Ansari, Boran Han, Shuai Zhang, Leman Akoglu, Christos Faloutsos, Michael W. Mahoney, Cuixiong Hu, Huzefa Rangwala, George Karypis, and Bernie Wang. Mitra: Mixed synthetic priors for enhancing tabular foundation models. Advances in Neural Information Processing Systems , 2025. 21

In [ ]:
#| hide
from fastpragma.core import *

# fastpragma

> An easy-to-use API for foundation model development, based on the [fastpragma](https://arxiv.org/abs/2604.08649) framework.

## Usage

### Installation

Install latest from the GitHub [repository][repo]:

Install from [conda][conda]:

Install from [pypi][pypi]:

[repo]: https://github.com/risheekkumarb/fastpragma
[docs]: https://risheekkumarb.github.io/fastpragma/
[pypi]: https://pypi.org/project/fastpragma/
[conda]: https://anaconda.org/risheekkumarb/fastpragma

In [ ]:
# Install latest from the GitHub repository:
# $ pip install git+https://github.com/risheekkumarb/fastpragma.git

# or from conda:
# $ conda install -c risheekkumarb fastpragma

# or from pypi:
# $ pip install fastpragma

### Documentation

Documentation can be found hosted on this GitHub [repository][repo]'s [pages][docs]. Additionally you can find package manager specific guidelines on [conda][conda] and [pypi][pypi] respectively.

[repo]: https://github.com/risheekkumarb/fastpragma
[docs]: https://risheekkumarb.github.io/fastpragma/
[pypi]: https://pypi.org/project/fastpragma/
[conda]: https://anaconda.org/risheekkumarb/fastpragma

## Data Format

fastpragma needs two types of data, both in key-value-time format:

1. **Profile Data** — wide DataFrame with three column types: `cats` (categorical), `conts` (continuous → bucketized), and `lifelong` (datetime columns for milestones like `first_topup`)
2. **Dynamic Event Data** — key-value data with timestamps, from multiple sources (transactions, app, trading, communications)

Each field has a **value type**: categorical, continuous (numerical), or textual.

## Example: MovieLens 100K

Load the classic MovieLens 100K dataset into polars DataFrames ready for `SourceSchema`:

import polars as pl
from pathlib import Path

def load_ml100k(path="ml-100k"):
    """Load MovieLens 100K as polars DataFrames for fastpragma.

    Returns `profile_df` (user attributes) and `events_df` (ratings with timestamps).
    """
    p = Path(path)

    # Profile: user_id, age, gender, occupation, zip_code
    profile_df = pl.read_csv(
        p / "u.user", separator="|", has_header=False,
        new_columns=["user_id", "age", "gender", "occupation", "zip_code"]
    )

    # Events: user_id, movie_id, rating, timestamp
    events_df = pl.read_csv(
        p / "u.data", separator="\t", has_header=False,
        new_columns=["user_id", "movie_id", "rating", "timestamp"]
    ).with_columns(
        pl.from_epoch("timestamp", time_unit="s").alias("timestamp")
    ).sort("user_id", "timestamp")

    return profile_df, events_df

profile_df, events_df = load_ml100k()
profile_df.head(3)

## Creating a Dataset

Declare each source separately with its own DataFrame and value types, then assemble them in `fp.PRAGMADataset`:

Each source is self-contained. Value types are inferred from dtypes by default but can be overridden with `cats`, `conts`, and `texts`. Profile sources don't need a `time_col` — regular columns share the eval-point timestamp, and `lifelong` columns (datetime-typed) get individual timestamps converted to temporal distance. The data is internally converted to key-value format for tokenization.

In [ ]:
import fastpragma as fp
profile = fp.SourceSchema(
    df=profile_df,
    cats=['Plan', 'Region'], conts=['Balance'],
    lifelong=['first_topup', 'first_trade'],
    num_buckets=10, cardinality_threshold=100
)

txns = fp.SourceSchema(
    df=txns_df, time_col='timestamp',
    cats=['Direction', 'Currency'], conts=['Amount'], texts=['Description']
)

app = fp.SourceSchema(
    df=app_df, time_col='event_time',
    cats=['Screen', 'Action'], conts=['Duration']
)

dataset = fp.PRAGMADataset(
    static=profile,
    dynamic=[txns, app],
    entity_col='account_id',
)

In [ ]:
dls = fp.PRAGMADataset(
    static=profile,
    dynamic=[txns, app],
    entity_col='account_id'
).dataloaders()

In [ ]:
dls.show_batch()

## Verifying Your Data

Use `show_batch()` to see how your data was tokenized before training:

```
Entity: 1 
Profile:
  Plan=Metal (categorical)
  Region=EU (categorical)
  Balance Quantile=3 (numerical)
  [2020-11-02] first_topup (lifelong → 4y 5m ago)
Events:
  [2025-01-15] Amount=45.20 (numerical)
  [2025-01-15] Direction=out (categorical)
  [2025-01-16] Channel=email (categorical)
  [2025-01-16] Description="metal plan" (textual → 3 tokens)


This lets you verify value types, timestamp parsing, and tokenization at a glance.

## Model and Training

The backbone is task-agnostic — it just produces embeddings. The task only matters for the head and loss function.

### Load the backbone

- `size`: 's' (10M), 'm' (100M), or 'l' (1B)
- `text_encoder`: optionally include a text encoder (e.g., Nemotron) for free-text fields


In [ ]:
model = fp.PRAGMA.load(size='l')

### Foundational Pre-training

Create a learner and pre-train the foundation model on your data. This trains the backbone using a self-supervised objective — no task labels needed. Use fastai's familiar `fit` method:

In [ ]:
learn = fp.learner(dls, model=model)
learn.fit(3)

### Train a task-specific head

Attach a head for your task and fine-tune with LoRA (only 2–4% of parameters). You can also use `learn.probe()` for a lightweight linear probe:

In [ ]:
# Classification (e.g., credit scoring, fraud detection)
learn = fp.learner(dls, model=model, head='classification')
learn.fine_tune(3, lora_rank=8, lora_alpha=8)   # LoRA, only 2-4% of params

# Or embedding probing — just a linear head, minimal compute
learn.probe(3, method="lbfgs")

# Other task types:
learn = fp.learner(dls, model=model, head='regression')    # LTV prediction
learn = fp.learner(dls, model=model, head='classification')       # product recommendation

### Get embeddings directly (no task needed)

Extract embeddings from the backbone without any task-specific head — useful for downstream analysis, clustering, or transfer to external models:

In [ ]:
embeddings = model.embed(dataset,
    pool="both")  # eval_df: account_id, eval_date

# Single record inference
emb = model.embed_record(record)

### Save and load

Save and reload trained models for later inference or deployment:

In [ ]:
learn.save('credit_scoring_model')
learn = fp.learner.load('credit_scoring_model')

In [ ]:
resp = httpx.get('https://arxiv.org/pdf/2604.08649', follow_redirects=True)
print(resp.status_code, len(resp.content))
open('/tmp/pragma.pdf', 'wb').write(resp.content)
print('saved')

200 619872


saved


# PRAGMA Paper — Key Points to Build

> Source: *PRAGMA: Revolut Foundation Model* (Ostroukhov et al., 2025), arXiv:2604.08649

## 1. Data Model

- **Record-level structure** — each observation = event history + contextual profile attributes, anchored to an **evaluation point**.
- **Three field types**: **numerical** (percentile-bucketed, with a special zero bucket), **categorical** (single token), **textual** (BPE-subword). Categorical vs. textual is decided by **cardinality thresholding**; high-cardinality fields fall back to BPE.
- **Event history** = ordered list of `{timestamp, key–value pairs}`. Event schemas vary by **source** (transactions, app, trading, communication).
- **Profile state** = set of descriptive `{key, value}` pairs timestamped at the eval point (or training cut-off). Some keys (e.g. `Lifelong: first_topup`) carry an **individual timestamp** → "life-long events" used to compute **temporal distance to eval point**.
- **Pre-training time range** = 25 months (2023–2025). Trade-off: event diversity, recency, distribution shift, context length.

## 2. Tokenisation (Disentangled)

Each data point = **(semantic type, value, temporal coordinate)**:
- **Key (semantic type)**: 1 token; shared between events and profile.
- **Value**: type-specific tokenisation (cat=1 tok, num=bucket, text=BPE).
- **Temporal coordinate**:
  - log-seconds to the last event
  - calendar features (day, hour, week, month)
  - RoPE-encoded time (profile & event encoders)

Multi-valued fields: key token is **replicated** across the value tokens and receives within-field positional embeddings.

## 3. Architecture

Encoder-only Transformer with **three blocks**:
1. **Profile State Encoder** — maps profile tokens + lifelong-event RoPE → `[USR]` embedding.
2. **Event Encoder** — encodes each event *independently* (shared weights) → `[EVT]` embedding + calendar features.
3. **History Encoder** — concatenates `[USR]` + `[EVT]`s, encodes with RoPE time-to-last-event → record representation.

Variants: **S=10M, M=100M, L=1B** (joint width + depth scaling; see Table 1).
All variants: GELU, pre-norm LayerNorm, dropout 0.1.

## 4. Pre-training

- **Objective**: Masked Language Modelling (MLM) over the disentangled tokens.
- **Backbone-only**; head is task-agnostic.
- The same backbone produces embeddings usable as-is, via probes, or via LoRA-fine-tuned heads.

## 5. Evaluation / Downstream

### Embedding probes
- Pool from `[USR]`, final `[EVT]`, or both.
- Standard-scaled features → **linear probe with L-BFGS** (best convergence vs. GBDT on high-dim).

### Downstream adaptation
- **LoRA** on QKV + MLP layers, default **rank=8, α=8** (sweeps: {4, 8, 16}).
- Optimiser: **Adam**; ~1/8 of pre-training wall-clock; 12h – few days.
- Tasks: classification, regression, recommendation, etc.

## 6. Key Engineering / Deployment Notes

- **Truncation** to a fixed context window (e.g. 6500 events, 200 profile tokens) — early milestones preserved by lifting to **life-long events**.
- **26M user records, 24B events, 207B tokens, 111 countries** at pre-training scale.
- **Dataset is fully anonymised**; no PII; no outlier removal / vocab pruning at pre-training (preserve heterogeneity).
- **Anti-Money-Laundering** is flagged as a limitation: highly relational task underperforms under PRAGMA's "single user" sequence model.

## 7. Implications for the API

- ✅ Expose **lifelong events** as a first-class `SourceSchema` (via `is_lifelong=True`).
- ❌ Add a **pre-training time range** config — *skipped for now*; only a single cut-off.
- ✅ Expose **bucketisation / cardinality threshold** for numerical / textual fields (`num_buckets`, `cardinality_threshold`).
- ❌ Surface the **three-block architecture** and per-block depth/width — *skipped for now*; using `get_emb_sz` defaults; architecture controlled via `size` presets.
- ✅ Make **probe pool** (USR / EVT / both) configurable in `model.embed(...)` (`pool="both"`).
- ✅ Expose **LoRA rank/alpha** on `learn.fine_tune(...)` (`lora_rank=8, lora_alpha=8`).
- ✅ Option: **L-BFGS linear probe** as the default for `learn.probe(...)` (`method="lbfgs"`).
- ✅ `model.embed_record(record)` should accept a single record.

## Plan: Complete what's already in the index page

A bottom-up build order. Each step is small enough to land, test, and revise before the next.

### 1. Package scaffold
- `pyproject.toml` with package name `fastpragma`, entry points, Python ≥3.11, install extras for the optional text encoder
- `fastpragma/core.py` (re-export everything users see in the index)
- `fastpragma/data.py`, `fastpragma/model.py`, `fastpragma/learner.py` (split modules)
- Install in editable mode: `pip install -e .`

### 2. Data layer (items 2-3, 5 on the index)
- `SourceSchema` dataclass: `df`, `time_col?` (only for events), `cats?`, `conts?`, `texts?`, `lifelong?` (list of datetime column names in profile)
- Internal `to_kvt()` conversion to a uniform key-value-time frame per source
- `fp.PRAGMADataset` accepting `static`, `dynamic=[...]`, `entity_col`, `max_events?`, `max_profile_tokens?`
- `dls.show_batch()` with the formatted output already shown in the index

### 3. Tokenisation
- One-token-per-key vocabulary
- Value tokenisers: categorical (1 token), numerical (percentile-bucketed, with zero bucket), textual (BPE subwords)
- Per-event tokenisation with positional ids within a field
- Calendar features (day/hour/week/month) and log-seconds-to-last-event, added to event embeddings
- Configurable cardinality threshold for cat vs. text

### 4. Backbone (item 6)
- Encoder-only Transformer with three blocks: Profile State Encoder, Event Encoder, History Encoder
- `fp.PRAGMA.load(size='s'|'m'|'l')` mapping to depths/widths in the paper's Table 1
- Optional `text_encoder` (e.g., Nemotron) for free-text fields
- `model.embed(dataset)` returning per-entity embeddings
- `model.embed_record(record)` for single-record inference

### 5. Pre-training (item 7)
- `fp.learner(dls, model=model)` returning a fastai-style `Learner`
- MLM head and masked-modelling loss
- `learn.fit(n)` wrapping fastai's fit

### 6. Heads + LoRA + probe (item 8)
- `head='classification'`, `head='regression'`
- LoRA on QKV + MLP, with `rank` and `alpha` (default 8/8) — exposed on the learner
- `learn.fine_tune(n)` for LoRA fine-tuning
- `learn.probe(n)` for a lightweight linear probe (standard-scaled inputs, L-BFGS optimiser)

### 7. Persistence (item 10)
- `learn.save('name')` and `fp.learner.load('name')` for backbone + head + LoRA weights

### 8. Polish
- Update `README.md` to match the index
- Smoke-test notebook: toy data → `SourceSchema` → `fp.PRAGMADataset` → `show_batch()` → `learn.fit(1)` → `learn.probe(1)` → `model.embed_record`

### Suggested first commit
Scaffold + item 2 (`SourceSchema` + `fp.PRAGMADataset` skeleton with no real tokenisation, but enough to call `.dataloaders()` and `show_batch()` on a tiny example).

we are going to make the this in polars. we are going to use ML-100K dataset

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Let me first check what's already in the repo and then we can figure out the approach.



<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;import os; print(&#x27;\\n&#x27;.join(sorted(os.lis…&quot;)→&quot;00_core.ipynb\n_quarto.yml\nindex.ipynb\nnb…&quot;</code></summary>

```json
{
  "id": "call_00_LuwXBIE2e3s6YJY28kvz0482",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "import os; print('\\n'.join(sorted(os.listdir('.'))))"
    }
  },
  "result": "00_core.ipynb\n_quarto.yml\nindex.ipynb\nnbdev.yml\nstyles.css"
}
```

</details>







<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;fd(&#x27;.&#x27;, pattern=&#x27;*.py&#x27;)\nfd(&#x27;.&#x27;, pattern=…&quot;)→&quot;[]&quot;</code></summary>

```json
{
  "id": "call_00_j5Alfa3mDxgtthtekxwq4598",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "fd('.', pattern='*.py')\nfd('.', pattern='*.ipynb')"
    }
  },
  "result": "[]"
}
```

</details>







<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;import json\nfor nb in [&#x27;index.ipynb&#x27;, &#x27;0…&quot;)→&quot;--- index.ipynb (markdown) ---\nTitle: PR…&quot;</code></summary>

```json
{
  "id": "call_00_X38Zfj6gUTDyy96AanIZ5914",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "import json\nfor nb in ['index.ipynb', '00_core.ipynb']:\n    cells = json.load(open(nb))['cells']\n    for c in cells:\n        src = ''.join(c['source'])\n        if src.strip():\n            print(f\"--- {nb} ({c['cell_type']}) ---\")\n            print(src[:200])\n            print()"
    }
  },
  "result": "<TRUNCATED>…ipynb (markdown) ---\nTitle: PRAGMA: Revolut Foundation Model\n\nURL Source: https://arxiv.org/pdf/2604.08649\n\nPublished Time: Mon, 13 Apr 2026 00:06:54 GMT\n\nNumber of Pages: 21\n\nMarkdown Content:\n# PRAGMA: Revolut Foundatio\n\n--- index.ipynb (code) ---\n#| hide\nfrom fastpragma.core import *\n\n--- index.ipynb (markdown) ---\n# fastpragma\n\n> An easy-to-use API for foundation model development, based on the [fastpragma](https://arxiv.org/abs/2604.08649) framework.\n\n--- index.ipynb (markdown) ---\n## Usage\n\n--- index.ipynb (markdown) ---\n### Installation\n\n--- index.ipynb (markdown) ---\nInstall latest from the GitHub [repository][repo]:\n\nInstall from [conda][conda]:\n\nInstall from [pypi][pypi]:\n\n[repo]: https://github.com/risheekkumarb/fastpragma\n[docs]: https://risheekkumarb.github.i\n\n--- index.ipynb (code) ---\n# Install latest from the GitHub repository:\n# $ pip install git+https://github.com/risheekkumarb/fastpragma.git\n\n# or from conda:\n# $ conda install -c risheekkumarb fastpragma\n\n# or from pypi:\n# $ pi\n\n--- index.ipynb (markdown) ---\n### Documentation\n\n--- index.ipynb (markdown) ---\nDocumentation can be found hosted on this GitHub [repository][repo]'s [pages][docs]. Additionally you can find package manager specific guidelines on [conda][conda] and [pypi][pypi] respectively.\n\n[re\n\n--- index.ipynb (markdown) ---\n## Data Format\n\nfastpragma needs two types of data, both in key-value-time format:\n\n1. **Profile Data** — wide DataFrame with three column types: `cats` (categorical), `conts` (continuous → bucketized\n\n--- index.ipynb (markdown) ---\n## Creating a Dataset\n\nDeclare each source separately with its own DataFrame and value types, then assemble them in `fp.PRAGMADataset`:\n\nEach source is self-contained. Value types are inferred from dt\n\n--- index.ipynb (code) ---\nimport fastpragma as fp\nprofile = fp.SourceSchema(\n    df=profile_df,\n    cats=['Plan', 'Region'], conts=['Balance'],\n    lifelong=['first_topup', 'first_trade'],\n    num_buckets=10, cardinality_thres\n\n--- index.ipynb (code) ---\ndls = fp.PRAGMADataset(\n    static=profile,\n    dynamic=[txns, app],\n    entity_col='account_id'\n).dataloaders()\n\n--- index.ipynb (code) ---\ndls.show_batch()\n\n--- index.ipynb (markdown) ---\n## Verifying Your Data\n\nUse `show_batch()` to see how your data was tokenized before training:\n\n```\nEntity: 1 \nProfile:\n  Plan=Metal (categorical)\n  Region=EU (categorical)\n  Balance Quantile=3 (numer\n\n--- index.ipynb (markdown) ---\n## Model and Training\n\nThe backbone is task-agnostic — it just produces embeddings. The task only matters for the head and loss function.\n\n### Load the backbone\n\n- `size`: 's' (10M), 'm' (100M), or 'l\n\n--- index.ipynb (code) ---\nmodel = fp.PRAGMA.load(size='l')\n\n--- index.ipynb (markdown) ---\n### Foundational Pre-training\n\nCreate a learner and pre-train the foundation model on your data. This trains the backbone using a self-supervised objective — no task labels needed. Use fastai's famili\n\n--- index.ipynb (code) ---\nlearn = fp.learner(dls, model=model)\nlearn.fit(3)\n\n--- index.ipynb (markdown) ---\n### Train a task-specific head\n\nAttach a head for your task and fine-tune with LoRA (only 2–4% of parameters). You can also use `learn.probe()` for a lightweight linear probe:\n\n--- index.ipynb (code) ---\n# Classification (e.g., credit scoring, fraud detection)\nlearn = fp.learner(dls, model=model, head='classification')\nlearn.fine_tune(3, lora_rank=8, lora_alpha=8)   # LoRA, only 2-4% of params\n\n# Or e\n\n--- index.ipynb (markdown) ---\n### Get embeddings directly (no task needed)\n\nExtract embeddings from the backbone without any task-specific head — useful for downstream analysis, clustering, or transfer to external models:\n\n--- index.ipynb (code) ---\nembeddings = model.embed(dataset,\n    pool=\"both\")  # eval_df: account_id, eval_date\n\n# Single record inference\nemb = model.embed_record(record)\n\n--- index.ipynb (markdown) ---\n### Save and load\n\nSave and reload trained models for later inference or deployment:\n\n--- index.ipynb (code) ---\nlearn.save('credit_scoring_model')\nlearn = fp.learner.load('credit_scoring_model')\n\n--- index.ipynb (code) ---\nresp = httpx.get('https://arxiv.org/pdf/2604.08649', follow_redirects=True)\nprint(resp.status_code, len(resp.content))\nopen('/tmp/pragma.pdf', 'wb').write(resp.content)\nprint('saved')\n\n--- index.ipynb (markdown) ---\n# PRAGMA Paper — Key Points to Build\n\n> Source: *PRAGMA: Revolut Foundation Model* (Ostroukhov et al., 2025), arXiv:2604.08649\n\n## 1. Data Model\n\n- **Record-level structure** — each observation = even\n\n--- index.ipynb (markdown) ---\n## Plan: Complete what's already in the index page\n\nA bottom-up build order. Each step is small enough to land, test, and revise before the next.\n\n### 1. Package scaffold\n- `pyproject.toml` with packa\n\n--- index.ipynb (markdown) ---\nwe are going to make the this in polars. we are going to use ML-100K dataset\n\n--- 00_core.ipynb (markdown) ---\n# core\n\n> Fill in a module description here\n\n--- 00_core.ipynb (code) ---\n#| defaul…</TRUNCATED>"
}
```

</details>





Great — so the current state is:

- `index.ipynb` has the PRAGMA paper + the full plan
- `00_core.ipynb` is a bare skeleton (just `def foo(): pass`)
- No actual Python package or source files exist yet

Before we write any code, let's think about how ML-100K maps to PRAGMA's data model. Here's my read:

**Entities:** users (943 of them)

**Profile state** (static per-user attributes):
- `age`, `gender`, `occupation`, `zip_code` — these are the categorical/continuous profile fields, timestamped at "eval point"

**Dynamic events** (ordered by timestamp):
- Each rating is an event: `{user_id, movie_id, rating, timestamp}`
- We could also join in movie metadata (genres, release date) as additional key-value pairs per event

**Possible tasks:**
- **Classification:** predict gender from rating history (or occupation)
- **Regression:** predict age from rating history
- **Recommendation:** predict which movies a user will rate highly

This is a nice fit because it mirrors the paper's structure — static user profile + ordered event stream — just at a much smaller, publicly-available scale.

Before we dive into code, a couple of questions:

- Which of these tasks would you want to focus on first?
- For the polars adaptation, do you want to use polars internally everywhere (in `to_kvt()`, tokenisation, etc.) or just accept polars DataFrames at the API boundary?

okay first download the dataset into this repo and view it

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;p = Path(&#x27;ml-100k&#x27;)\nif not p.exists():\n …&quot;)→&quot;Traceback (most recent call last):\n\n  Fi…&quot;</code></summary>

```json
{
  "id": "call_00_YToOY7zIEVNXSoxgP7vd5395",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "p = Path('ml-100k')\nif not p.exists():\n    !wget -q https://files.grouplens.org/datasets/movielens/ml-100k.zip -O /tmp/ml-100k.zip\n    import zipfile, shutil\n    zipfile.ZipFile('/tmp/ml-100k.zip').extractall('.')\n    print('extracted')\nelse:\n    print('already exists')\nprint('\\n'.join(sorted(os.listdir('ml-100k'))))"
    }
  },
  "result": "Traceback (most recent call last):\n\n  File /usr/local/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3746 in run_code\n    await eval(code_obj, self.user_global_ns, self.user_ns)\n\n  Cell In[9], line 1\n    await call_tool(python, {'code': \"p = Path('ml-100k')\\nif not p.exists():\\n    !wget -q https://files.grouplens.org/datasets/movielens/ml-100k.zip -O /tmp/ml-100k.zip\\n    import zipfile, shutil\\n    zipfile.ZipFile('/tmp/ml-100k.zip').extractall('.')\\n    print('extracted')\\nelse:\\n    print('already exists')\\nprint('\\\\n'.join(sorted(os.listdir('ml-100k'))))\"})\n\n  File /usr/local/lib/python3.12/site-packages/ipykernel_helper/core.py:373 in call_tool\n    return await maybe_await(func(**coerce_inputs(func, kw)))\n\n  File /usr/local/lib/python3.12/site-packages/fastcore/xtras.py:1079 in maybe_await\n    return await o if isawaitable(o) else o\n\n  File /usr/local/lib/python3.12/site-packages/safepyrun/core.py:299 in __call__\n    raise e.with_traceback(tb) from None\n\n  File /usr/local/lib/python3.12/ast.py:52 in parse\n    return compile(source, filename, mode, flags,\n\n  File <unknown>:3\n    !wget -q https://files.grouplens.org/datasets/movielens/ml-100k.zip -O /tmp/ml-100k.zip\n                                                                 ^\nSyntaxError: invalid decimal literal"
}
```

</details>





The `!` shell prefix doesn't work inside the `python` tool. Let me use `httpx` instead:



<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;p = Path(&#x27;ml-100k&#x27;)\nif not p.exists():\n …&quot;)→&quot;----------------------------------------…&quot;</code></summary>

```json
{
  "id": "call_00_MGSErqlLDRgVJqsaLh4l3396",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "p = Path('ml-100k')\nif not p.exists():\n    import httpx, zipfile, io\n    url = 'https://files.grouplens.org/datasets/movielens/ml-100k.zip'\n    resp = httpx.get(url, follow_redirects=True)\n    z = zipfile.ZipFile(io.BytesIO(resp.content))\n    z.extractall('.')\n    print('extracted ok')\nelse:\n    print('already exists')\nprint('\\n'.join(sorted(os.listdir('ml-100k'))))"
    }
  },
  "result": "---------------------------------------------------------------------------\nNameError                                 Traceback (most recent call last)\nCell In[10], line 1\n----> 1 await call_tool(python, {'code': \"p = Path('ml-100k')\\nif not p.exists():\\n    import httpx, zipfile, io\\n    url = 'https://files.grouplens.org/datasets/movielens/ml-100k.zip'\\n    resp = httpx.get(url, follow_redirects=True)\\n    z = zipfile.ZipFile(io.BytesIO(resp.content))\\n    z.extractall('.')\\n    print('extracted ok')\\nelse:\\n    print('already exists')\\nprint('\\\\n'.join(sorted(os.listdir('ml-100k'))))\"})\n\nFile /usr/local/lib/python3.12/site-packages/ipykernel_helper/core.py:373, in call_tool(func, kw)\n    371 async def call_tool(func, kw):\n    372     \"Call `func(**kw)` with `coerce_inputs`\"\n--> 373     return await maybe_await(func(**coerce_inputs(func, kw)))\n\nFile /usr/local/lib/python3.12/site-packages/fastcore/xtras.py:1079, in maybe_await(o)\n   1077 \"Await `o` if needed, and return it\"\n   1078 from inspect import isawaitable\n-> 1079 return await o if isawaitable(o) else o\n\nFile /usr/local/lib/python3.12/site-packages/safepyrun/core.py:299, in RunPython.__call__(self, code)\n    297 tb = e.__traceback__\n    298 while tb.tb_next and not tb.tb_frame.f_code.co_filename.startswith('<python'): tb = tb.tb_next\n--> 299 raise e.with_traceback(tb) from None\n\nFile <python_5>:1\n----> 1 p = Path('ml-100k')\n      2 if not p.exists():\n      3     import httpx, zipfile, io\n\nNameError: name 'Path' is not defined"
}
```

</details>





Need to import `Path` — let me fix:



<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;from pathlib import Path\np = Path(&#x27;ml-10…&quot;)→&quot;PermissionError: Audit: socket.connect b…&quot;</code></summary>

```json
{
  "id": "call_00_Tg0ifiM8g7d9cbKey9rE3030",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "from pathlib import Path\np = Path('ml-100k')\nif not p.exists():\n    import httpx, zipfile, io\n    url = 'https://files.grouplens.org/datasets/movielens/ml-100k.zip'\n    resp = httpx.get(url, follow_redirects=True)\n    z = zipfile.ZipFile(io.BytesIO(resp.content))\n    z.extractall('.')\n    print('extracted ok')\nelse:\n    print('already exists')\nprint('\\n'.join(sorted(os.listdir('ml-100k'))))"
    }
  },
  "result": "PermissionError: Audit: socket.connect blocked in sandbox with args: (<socket.socket fd=41, family=2, type=1, proto=6, laddr=('0.0.0.0', 0)>, ('128.101.96.204', 443))\nCall chain: runpy._run_module_as_main -> runpy._run_code -> __main__.<module> -> __main__.main -> __main__._run_kernel_from_cli -> ipymini.kernel.run_kernel -> ipymini.kernel.MiniKernel.start -> ipymini.kernel.Subshell.run_main -> ipymini.kernel.Subshell._run_loop_body -> asyncio.runners.Runner.run -> asyncio.base_events.BaseEventLoop.run_until_complete -> asyncio.base_events.BaseEventLoop.run_forever -> asyncio.base_events.BaseEventLoop._run_once -> asyncio.events.Handle._run -> microio._task.TaskGroup._run -> microio._task.TaskGroup._call_started -> microio._actor.ActorCore._handle_release -> ipymini.kernel.Subshell._handle_actor_item -> ipymini.kernel.Subshell._handle_message -> ipymini.kernel.Subshell._handle_execute -> ipymini.shell.shell.MiniShell.execute -> ipymini.shell.shell.MiniShell._run_cell -> IPython.core.interactiveshell.InteractiveShell.run_cell_async -> IPython.core.interactiveshell.InteractiveShell.run_ast_nodes -> IPython.core.interactiveshell.InteractiveShell.run_code -> __main__.<module> -> ipykernel_helper.core.call_tool -> fastcore.xtras.maybe_await -> safepyrun.core.RunPython.__call__ -> safepyrun.core._run_python -> safepyrun.core.__run_python -> safepyrun.core.__run_python.<locals>.run -> <python>.<module> -> httpx._api.get -> httpx._api.request -> httpx._client.Client.request -> httpx._client.Client.send -> httpx._client.Client._send_handling_auth -> httpx._client.Client._send_handling_redirects -> httpx._client.Client._send_single_request -> httpx._transports.default.HTTPTransport.handle_request -> httpcore._sync.connection_pool.ConnectionPool.handle_request -> httpcore._sync.connection.HTTPConnection.handle_request -> httpcore._sync.connection.HTTPConnection._connect -> httpcore._backends.sync.SyncBackend.connect_tcp -> socket.create_connection. Code message has been added to dialog. Ask user to run it."
}
```

</details>





The `python` sandbox blocks both `httpx` (socket blocked) and `!` shell prefix. I need to use the `bash` tool instead to download it. You'll need to run this code message yourself to download the dataset:

```bash
cd /app/data/create_pkg/fastpragma/nbs
wget -q https://files.grouplens.org/datasets/movielens/ml-100k.zip -O /tmp/ml-100k.zip
unzip -o /tmp/ml-100k.zip -d ml-100k
ls ml-100k/
```

In [ ]:
from pathlib import Path
p = Path('temp/ml-100k')
if not p.exists():
    import httpx, zipfile, io
    url = 'https://files.grouplens.org/datasets/movielens/ml-100k.zip'
    resp = httpx.get(url, follow_redirects=True)
    z = zipfile.ZipFile(io.BytesIO(resp.content))
    z.extractall('.')
    print('extracted ok')
else:
    print('already exists')
print('\n'.join(sorted(os.listdir('ml-100k'))))

extracted ok


README
allbut.pl
mku.sh
u.data
u.genre
u.info
u.item
u.occupation
u.user
u1.base
u1.test
u2.base
u2.test
u3.base
u3.test
u4.base
u4.test
u5.base
u5.test
ua.base
ua.test
ub.base
ub.test


done lets add this as an example for our api

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Let me first look at the data files to understand their structure:



<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;print(\&quot;=== u.user (users) ===\&quot;)\nfor line…&quot;)→&quot;=== u.user (users) ===\n\n----------------…&quot;</code></summary>

```json
{
  "id": "call_00_HkDUkUTLAyXWE3aY0puj4460",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "print(\"=== u.user (users) ===\")\nfor line in Path('temp/ml-100k/u.user').read_text().splitlines()[:5]:\n    print(line)\nprint(\"\\n=== u.data (ratings) ===\")\nfor line in Path('temp/ml-100k/u.data').read_text().splitlines()[:5]:\n    print(line)\nprint(\"\\n=== u.item (movies) ===\")\nfor line in Path('temp/ml-100k/u.item').read_text().splitlines()[:3]:\n    print(line)\nprint(\"\\n=== u.genre ===\")\nprint(Path('temp/ml-100k/u.genre').read_text().strip())\nprint(\"\\n=== u.occupation ===\")\nprint(Path('temp/ml-100k/u.occupation').read_text().strip())"
    }
  },
  "result": "=== u.user (users) ===\n\n---------------------------------------------------------------------------\nFileNotFoundError                         Traceback (most recent call last)\nCell In[13], line 1\n----> 1 await call_tool(python, {'code': 'print(\"=== u.user (users) ===\")\\nfor line in Path(\\'temp/ml-100k/u.user\\').read_text().splitlines()[:5]:\\n    print(line)\\nprint(\"\\\\n=== u.data (ratings) ===\")\\nfor line in Path(\\'temp/ml-100k/u.data\\').read_text().splitlines()[:5]:\\n    print(line)\\nprint(\"\\\\n=== u.item (movies) ===\")\\nfor line in Path(\\'temp/ml-100k/u.item\\').read_text().splitlines()[:3]:\\n    print(line)\\nprint(\"\\\\n=== u.genre ===\")\\nprint(Path(\\'temp/ml-100k/u.genre\\').read_text().strip())\\nprint(\"\\\\n=== u.occupation ===\")\\nprint(Path(\\'temp/ml-100k/u.occupation\\').read_text().strip())'})\n\nFile /usr/local/lib/python3.12/site-packages/ipykernel_helper/core.py:373, in call_tool(func, kw)\n    371 async def call_tool(func, kw):\n    372     \"Call `func(**kw)` with `coerce_inputs`\"\n--> 373     return await maybe_await(func(**coerce_inputs(func, kw)))\n\nFile /usr/local/lib/python3.12/site-packages/fastcore/xtras.py:1079, in maybe_await(o)\n   1077 \"Await `o` if needed, and return it\"\n   1078 from inspect import isawaitable\n-> 1079 return await o if isawaitable(o) else o\n\nFile /usr/local/lib/python3.12/site-packages/safepyrun/core.py:299, in RunPython.__call__(self, code)\n    297 tb = e.__traceback__\n    298 while tb.tb_next and not tb.tb_frame.f_code.co_filename.startswith('<python'): tb = tb.tb_next\n--> 299 raise e.with_traceback(tb) from None\n\nFile <python_7>:2\n      1 print('=== u.user (users) ===')\n----> 2 for line in Path('temp/ml-100k/u.user').read_text().splitlines()[:5]:\n      3     print(line)\n      4 print('\\n=== u.data (ratings) ===')\n\nFile /usr/local/lib/python3.12/pathlib.py:1027, in Path.read_text(self, encoding, errors)\n   1023 \"\"\"\n   1024 Open the file in text mode, read it, and close the file.\n   1025 \"\"\"\n   1026 encoding = io.text_encoding(encoding)\n-> 1027 with self.open(mode='r', encoding=encoding, errors=errors) as f:\n   1028     return f.read()\n\nFile /usr/local/lib/python3.12/pathlib.py:1013, in Path.open(self, mode, buffering, encoding, errors, newline)\n   1011 if \"b\" not in mode:\n   1012     encoding = io.text_encoding(encoding)\n-> 1013 return io.open(self, mode, buffering, encoding, errors, newline)\n\nFileNotFoundError: [Errno 2] No such file or directory: 'temp/ml-100k/u.user'"
}
```

</details>





The path doesn't match the code message's extract. Let me find the actual location:



<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;from pathlib import Path\nprint(\&quot;cwd:\&quot;, P…&quot;)→&quot;cwd: /app/data/create_pkg/fastpragma/nbs…&quot;</code></summary>

```json
{
  "id": "call_00_qvNoVSfMMntNrmKl94Cn0982",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "from pathlib import Path\nprint(\"cwd:\", Path.cwd())\np = Path('ml-100k')\nprint(\"ml-100k exists:\", p.exists())\n# list top-level for any ml dirs\nfor d in Path.cwd().iterdir():\n    if 'ml' in d.name.lower():\n        print(f\"  {d.name}/\")"
    }
  },
  "result": "cwd: /app/data/create_pkg/fastpragma/nbs\nml-100k exists: True\n  _quarto.yml/\n  ml-100k/\n  nbdev.yml/"
}
```

</details>







<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;print(\&quot;=== u.user (users) ===\&quot;)\nfor line…&quot;)→&quot;=== u.user (users) ===\n1|24|M|technician…&quot;</code></summary>

```json
{
  "id": "call_00_CwSm6huNUMQ7xHvuHVZU4739",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "print(\"=== u.user (users) ===\")\nfor line in Path('ml-100k/u.user').read_text().splitlines()[:5]:\n    print(line)\nprint(\"\\n=== u.data (ratings) ===\")\nfor line in Path('ml-100k/u.data').read_text().splitlines()[:5]:\n    print(line)\nprint(\"\\n=== u.item (movies - 1st) ===\")\nprint(Path('ml-100k/u.item').read_text().splitlines()[0])\nprint(\"\\n=== u.genre ===\")\nprint(Path('ml-100k/u.genre').read_text().strip())\nprint(\"\\n=== u.occupation ===\")\nprint(Path('ml-100k/u.occupation').read_text().strip())"
    }
  },
  "result": "=== u.user (users) ===\n1|24|M|technician|85711\n2|53|F|other|94043\n3|23|M|writer|32067\n4|24|M|technician|43537\n5|33|F|other|15213\n\n=== u.data (ratings) ===\n196\t242\t3\t881250949\n186\t302\t3\t891717742\n22\t377\t1\t878887116\n244\t51\t2\t880606923\n166\t346\t1\t886397596\n\n=== u.item (movies - 1st) ===\n\n---------------------------------------------------------------------------\nUnicodeDecodeError                        Traceback (most recent call last)\nCell In[15], line 1\n----> 1 await call_tool(python, {'code': 'print(\"=== u.user (users) ===\")\\nfor line in Path(\\'ml-100k/u.user\\').read_text().splitlines()[:5]:\\n    print(line)\\nprint(\"\\\\n=== u.data (ratings) ===\")\\nfor line in Path(\\'ml-100k/u.data\\').read_text().splitlines()[:5]:\\n    print(line)\\nprint(\"\\\\n=== u.item (movies - 1st) ===\")\\nprint(Path(\\'ml-100k/u.item\\').read_text().splitlines()[0])\\nprint(\"\\\\n=== u.genre ===\")\\nprint(Path(\\'ml-100k/u.genre\\').read_text().strip())\\nprint(\"\\\\n=== u.occupation ===\")\\nprint(Path(\\'ml-100k/u.occupation\\').read_text().strip())'})\n\nFile /usr/local/lib/python3.12/site-packages/ipykernel_helper/core.py:373, in call_tool(func, kw)\n    371 async def call_tool(func, kw):\n    372     \"Call `func(**kw)` with `coerce_inputs`\"\n--> 373     return await maybe_await(func(**coerce_inputs(func, kw)))\n\nFile /usr/local/lib/python3.12/site-packages/fastcore/xtras.py:1079, in maybe_await(o)\n   1077 \"Await `o` if needed, and return it\"\n   1078 from inspect import isawaitable\n-> 1079 return await o if isawaitable(o) else o\n\nFile /usr/local/lib/python3.12/site-packages/safepyrun/core.py:299, in RunPython.__call__(self, code)\n    297 tb = e.__traceback__\n    298 while tb.tb_next and not tb.tb_frame.f_code.co_filename.startswith('<python'): tb = tb.tb_next\n--> 299 raise e.with_traceback(tb) from None\n\nFile <python_9>:8\n      6     print(line)\n      7 print('\\n=== u.item (movies - 1st) ===')\n----> 8 print(Path('ml-100k/u.item').read_text().splitlines()[0])\n      9 print('\\n=== u.genre ===')\n     10 print(Path('ml-100k/u.genre').read_text().strip())\n\nFile /usr/local/lib/python3.12/pathlib.py:1028, in Path.read_text(self, encoding, errors)\n   1026 encoding = io.text_encoding(encoding)\n   1027 with self.open(mode='r', encoding=encoding, errors=errors) as f:\n-> 1028     return f.read()\n\nFile /usr/local/lib/python3.12/codecs.py:322, in BufferedIncrementalDecoder.decode(self, input, final)\n    319 def decode(self, input, final=False):\n    320     # decode input (taking the buffer into account)\n    321     data = self.buffer + input\n--> 322     (result, consumed) = self._buffer_decode(data, self.errors, final)\n    323     # keep undecoded input until the next call\n    324     self.buffer = data[consumed:]\n\nUnicodeDecodeError: 'utf-8' codec can't decode byte 0xe9 in position 76620: invalid continuation byte"
}
```

</details>





Encoding issue with u.item — likely latin-1. Let me read it properly plus the README:



<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;print(\&quot;=== u.item (movies - first 2) ===…&quot;)→&quot;=== u.item (movies - first 2) ===\n1|Toy …&quot;</code></summary>

```json
{
  "id": "call_00_9hPkJhThJkJ8LHV4H33m6174",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "print(\"=== u.item (movies - first 2) ===\")\nfor line in Path('ml-100k/u.item').read_text(encoding='latin-1').splitlines()[:2]:\n    print(line)\nprint(\"\\n=== u.info ===\")\nprint(Path('ml-100k/u.info').read_text().strip())\nprint(\"\\n=== README ===\")\nprint(Path('ml-100k/README').read_text()[:1200])"
    }
  },
  "result": "=== u.item (movies - first 2) ===\n1|Toy Story (1995)|01-Jan-1995||http://us.imdb.com/M/title-exact?Toy%20Story%20(1995)|0|0|0|1|1|1|0|0|0|0|0|0|0|0|0|0|0|0|0\n2|GoldenEye (1995)|01-Jan-1995||http://us.imdb.com/M/title-exact?GoldenEye%20(1995)|0|1|1|0|0|0|0|0|0|0|0|0|0|0|0|0|1|0|0\n\n=== u.info ===\n943 users\n1682 items\n100000 ratings\n\n=== README ===\nSUMMARY & USAGE LICENSE\n=============================================\n\nMovieLens data sets were collected by the GroupLens Research Project\nat the University of Minnesota.\n \nThis data set consists of:\n\t* 100,000 ratings (1-5) from 943 users on 1682 movies. \n\t* Each user has rated at least 20 movies. \n        * Simple demographic info for the users (age, gender, occupation, zip)\n\nThe data was collected through the MovieLens web site\n(movielens.umn.edu) during the seven-month period from September 19th, \n1997 through April 22nd, 1998. This data has been cleaned up - users\nwho had less than 20 ratings or did not have complete demographic\ninformation were removed from this data set. Detailed descriptions of\nthe data file can be found at the end of this file.\n\nNeither the University of Minnesota nor any of the researchers\ninvolved can guarantee the correctness of the data, its suitability\nfor any particular purpose, or the validity of results based on the\nuse of the data set.  The data set may be used for any research\npurposes under the following conditions:\n\n     * The user may not state or imply any endorsement from the\n       University of Minnesota or the GroupLens Research Group."
}
```

</details>





Here's how ML-100K maps to the PRAGMA data model:

**Profile state** (`u.user` — 943 rows):
| Column | PRAGMA type | Example |
|--------|-------------|---------|
| `age` | continuous (numerical) | 24, 53 |
| `gender` | categorical | M, F |
| `occupation` | categorical | technician, writer |
| `zip_code` | categorical | 85711, 94043 |

**Dynamic events** (`u.data` — 100k ratings, ordered by timestamp):
| Column | PRAGMA type | Example |
|--------|-------------|---------|
| `movie_id` | categorical | 242, 302 |
| `rating` | numerical | 3, 5 |
| `timestamp` | time coordinate | 881250949 |

We could also join movie metadata (`u.item`) into events: `title` as textual, and the 19 genre flags as categorical fields.

**Tasks** we could evaluate:
- **Classification**: predict `gender` or `occupation` from rating history
- **Regression**: predict `age` from rating history
- **Recommendation**: multilabel — predict which genres a user likes

The simplest first step: write a `load_ml100k()` function that returns polars DataFrames (`profile_df`, `events_df`) ready to feed into `SourceSchema`. Want me to start with that?

actually create the entire data, model, learner notebooks with polars and fastai.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Let me start by creating the data notebook as a new dialog. First, let me check what's already available in the kernel:



<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;import polars as pl, torch, fastai\nprint…&quot;)→&quot;----------------------------------------…&quot;</code></summary>

```json
{
  "id": "call_00_gt1oCUMO3tXuXocZxexW4736",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "import polars as pl, torch, fastai\nprint(f\"polars={pl.__version__}, torch={torch.__version__}, fastai={fastai.__version__}\")"
    }
  },
  "result": "---------------------------------------------------------------------------\nModuleNotFoundError                       Traceback (most recent call last)\nCell In[17], line 1\n----> 1 await call_tool(python, {'code': 'import polars as pl, torch, fastai\\nprint(f\"polars={pl.__version__}, torch={torch.__version__}, fastai={fastai.__version__}\")'})\n\nFile /usr/local/lib/python3.12/site-packages/ipykernel_helper/core.py:373, in call_tool(func, kw)\n    371 async def call_tool(func, kw):\n    372     \"Call `func(**kw)` with `coerce_inputs`\"\n--> 373     return await maybe_await(func(**coerce_inputs(func, kw)))\n\nFile /usr/local/lib/python3.12/site-packages/fastcore/xtras.py:1079, in maybe_await(o)\n   1077 \"Await `o` if needed, and return it\"\n   1078 from inspect import isawaitable\n-> 1079 return await o if isawaitable(o) else o\n\nFile /usr/local/lib/python3.12/site-packages/safepyrun/core.py:299, in RunPython.__call__(self, code)\n    297 tb = e.__traceback__\n    298 while tb.tb_next and not tb.tb_frame.f_code.co_filename.startswith('<python'): tb = tb.tb_next\n--> 299 raise e.with_traceback(tb) from None\n\nFile <python_2>:1\n----> 1 import polars as pl, torch, fastai\n\nModuleNotFoundError: No module named 'polars'"
}
```

</details>





Polars isn't installed yet. Let me install it:



<details class='tool-usage-details' markdown='1'>
<summary><code>bash(cmd=&quot;pip install polars&quot;)→&quot;PermissionError: pip install polars. Cod…&quot;</code></summary>

```json
{
  "id": "call_00_lhRmHaPcog3eY69b1zAY7616",
  "server": false,
  "call": {
    "function": "bash",
    "arguments": {
      "cmd": "pip install polars"
    }
  },
  "result": "PermissionError: pip install polars. Code message has been added to dialog. Ask user to run it."
}
```

</details>





We need polars installed before we can proceed. The sandbox blocks `pip install` from both `python` and `bash` tools — you'll need to run this yourself:

```bash
pip install polars
```

Once that's done, here's my plan for the three notebooks:

**`01_data.ipynb`** — Data layer
- `load_ml100k()` returning polars DataFrames for profile + events
- `SourceSchema` dataclass accepting polars DF with `cats`/`conts`/`texts`/`lifelong`
- Internal `_to_kvt()` converting wide DataFrames to key-value-time format using polars
- `PRAGMADataset` assembling static + dynamic sources with `entity_col`
- `.dataloaders()` returning fastai `DataLoaders`
- `.show_batch()` for human-readable verification

**`02_model.ipynb`** — Backbone
- Profile State Encoder, Event Encoder, History Encoder in PyTorch
- `PRAGMA.load(size='s'|'m'|'l')` mapping to paper's Table 1 depths/widths
- `model.embed()` and `model.embed_record()` for extracting embeddings

**`03_learner.ipynb`** — Training
- `fp.learner()` returning fastai Learner with MLM head
- `learn.fit()` for pre-training, `learn.fine_tune()` with LoRA
- `learn.probe()` with L-BFGS linear probe
- `learn.save()` / `learn.load()` for persistence

Run the pip install above, then I'll start building `01_data.ipynb`.

polars installed.
load_ml100k() returning polars DataFrames for profile + events - this is an example to be placed in index.ipynb.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Let me first look at the current structure of `index.ipynb` to see where to place the function:



<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;from dialoghelper.solveitskill import vi…&quot;)→&quot;&lt;msgs&gt;&lt;note id=\&quot;_d5482fcf\&quot;&gt;     1 │ Titl…&quot;</code></summary>

```json
{
  "id": "call_00_XBwF2GKffqhTYBALmgYQ9554",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "from dialoghelper.solveitskill import view_dlg\nprint(await view_dlg('index', nums=True, include_output=True))"
    }
  },
  "result": "<TRUNCATED>…e id=\"_d5482fcf\">     1 │ Title: PRAGMA: Revolut Foundation Model\n     2 │ \n     3 │ URL Source: https://arxiv.org/pdf/2604.08649\n     4 │ \n     5 │ Published Time: Mon, 13 Apr 2026 00:06:54 GMT\n     6 │ \n     7 │ Number of Pages: 21\n     8 │ \n     9 │ Markdown Content:\n    10 │ # PRAGMA: Revolut Foundation Model \n    11 │ \n    12 │ Maxim Ostroukhov 1 Ruslan Mikhailov 1 Vladimir Iashin 1\n    13 │ \n    14 │ Artem Sokolov 1 Andrei Akshonov 1 Vitaly Protasov 1 Dmitrii Beloborodov 1\n    15 │ \n    16 │ Vince Mullin 2 Roman Y. Enzmann 2 Georgios Kolovos 2 Jason Renders 2\n    17 │ \n    18 │ Pavel Nesterov 1 Anton Repushko 1\n    19 │ \n    20 │ > 1\n    21 │ \n    22 │ Revolut Research 2NVIDIA \n    23 │ \n    24 │ Modern financial systems generate vast quantities of transactional and event-level data that encode rich economic signals. This paper presents PRAGMA, a family of foundation models for multi-source banking event sequences. Our approach pre-trains a Transformer-based architecture with masked modelling on a large-scale, heterogeneous banking event corpus using a self-supervised objective tailored to the discrete, variable-length nature of financial records. The resulting model supports a wide range of downstream tasks such as credit scoring, fraud detection, and lifetime value prediction: strong performance can be achieved by training a simple linear model on top of the extracted embeddings and can be further improved with lightweight fine-tuning. Through extensive evaluation on downstream tasks, we demonstrate that PRAGMA achieves superior performance across multiple domains directly from raw event sequences, providing a general-purpose representation layer for financial applications. \n    25 │ \n    26 │ Disclaimer : We report only relative improvements, as absolute metrics are commercially sensitive. All examples are synthetic and not from real production data. Credit Recurrent Trx. LTV Comms Fraud Product Rec.    \n    27 │ \n    28 │ > Task-specific model PRAGMA-S (10 M) PRAGMA-M (100 M) PRAGMA-L (1 B)\n    29 │ \n    30 │ Figure 1: A single architecture from 10M to 1B parameters that outperforms task-specific models across tasks. \n    31 │ \n    32 │ ## 1 Introduction \n    33 │ \n    34 │ Foundation models are general-purpose models trained at scale on broad data distributions and subsequently adapted to a wide variety of downstream tasks (Bommasani et al., 2021). While such models have transformed natural language processing (Devlin et al., 2019; Brown et al., 2020) and computer vision (Kirillov et al., 2023; Caron et al., 2021), their application to multi-source banking user histories remains comparatively underexplored. Modern banks and fintechs accumulate large volumes of data: event streams spanning card and transfer transactions, product usage, in-app navigation, and customer communications, alongside static generalised profile state such as account tenure and 1\n    35 │ \n    36 │ > arXiv:2604.08649v1 [cs.LG] 9 Apr 2026\n    37 │ \n    38 │ plan. These event streams encode signals relevant to risk management, product analytics, and operations, but they are difficult to model efficiently with off-the-shelf language-model tokenisation and architectures. While serialising structured records as text and feeding them to a standard Transformer is a viable baseline, it inflates sequence lengths considerably because every field name and delimiter becomes several subword tokens. Moreover, numerical values are split into digit fragments that discard magnitude and ordering, both of which are critical for financial reasoning. Together, these limitations make naive text serialisation impractical for the long, heterogeneous user histories common in banking. Multi-source banking user histories differ from text in three ways. First, each event is a variable-length record with mixed categorical, numerical, and free-text fields. Second, histories are long-tailed in length and irregular in time, with strong daily and weekly cycles. Third, practical deployments must operate under strict privacy and regulatory constraints, which limit what can be reported and which features can be used for certain decisions. Because no single off-the-shelf architecture handles all three challenges simultaneously, practitioners default to building task-specific pipelines with extensive feature engineering, making it hard to share statistical strength across domains and products. Prior work addresses isolated slices of this problem. Tabular Transformers such as TabTransformer and FT-Transformer (Huang et al., 2020; Gorishniy et al., 2021) model fixed-schema rows, while sequential recommender models such as SASRec and BERT4Rec (Kang et al., 2018; Sun et al., 2019) operate on item-like interaction histories. Financial foundation models have largely focused on text or generic time-series tokenisation (Yang et al., 2020; Wu et al., 2023; Yang et al., 2023; Jin et al., 2024; Ansari et al., 2024), while newer transaction-ledger model…</TRUNCATED>"
}
```

</details>





Good, I can see the full structure. I'll add a `load_ml100k()` example as a new section between "Data Format" and "Creating a Dataset". Let me add it now:



<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;from dialoghelper.solveitskill import ad…&quot;)→&quot;heading id: _65e027f7\ncode id: _cba4869c&quot;</code></summary>

```json
{
  "id": "call_00_TEUn2PJ1J9S5hFNo2vrr9668",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "from dialoghelper.solveitskill import add_msg\n\n# Add heading note after _5225fbe5\nhid = await add_msg(\n    '## Example: MovieLens 100K\\n\\n'\n    'Load the classic MovieLens 100K dataset into polars DataFrames ready for `SourceSchema`:',\n    placement='add_after', id='_5225fbe5', dname='index'\n)\nprint('heading id:', hid)\n\n# Add the code block after the heading\ncid = await add_msg(\n    'import polars as pl\\n'\n    'from pathlib import Path\\n'\n    '\\n'\n    'def load_ml100k(path=\"ml-100k\"):\\n'\n    '    \"\"\"Load MovieLens 100K as polars DataFrames for fastpragma.\\n'\n    '\\n'\n    '    Returns `profile_df` (user attributes) and `events_df` (ratings with timestamps).\\n'\n    '    \"\"\"\\n'\n    '    p = Path(path)\\n'\n    '\\n'\n    '    # Profile: user_id, age, gender, occupation, zip_code\\n'\n    '    profile_df = pl.read_csv(\\n'\n    '        p / \"u.user\", separator=\"|\", has_header=False,\\n'\n    '        new_columns=[\"user_id\", \"age\", \"gender\", \"occupation\", \"zip_code\"]\\n'\n    '    )\\n'\n    '\\n'\n    '    # Events: user_id, movie_id, rating, timestamp\\n'\n    '    events_df = pl.read_csv(\\n'\n    '        p / \"u.data\", separator=\"\\\\t\", has_header=False,\\n'\n    '        new_columns=[\"user_id\", \"movie_id\", \"rating\", \"timestamp\"]\\n'\n    '    ).with_columns(\\n'\n    '        pl.from_epoch(\"timestamp\", time_unit=\"s\").alias(\"timestamp\")\\n'\n    '    ).sort(\"user_id\", \"timestamp\")\\n'\n    '\\n'\n    '    return profile_df, events_df',\n    placement='add_after', id=hid, dname='index'\n)\nprint('code id:', cid)"
    }
  },
  "result": "heading id: _65e027f7\ncode id: _cba4869c"
}
```

</details>





Now let me add a quick verification block showing what it returns:



<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;await add_msg(\n    &#x27;profile_df, events_d…&quot;)→&quot;_79d17b8d&quot;</code></summary>

```json
{
  "id": "call_00_bzDpxLVd9B7rXXeGaSUY2371",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "await add_msg(\n    'profile_df, events_df = load_ml100k()\\n'\n    'profile_df.head(3)',\n    placement='add_after', id='_cba4869c', dname='index'\n)"
    }
  },
  "result": "_79d17b8d"
}
```

</details>





Let me verify it looks right by viewing that section of the index:



<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;await view_dlg(&#x27;index&#x27;, nums=True)&quot;)→&quot;&lt;msgs&gt;&lt;note id=\&quot;_d5482fcf\&quot;&gt;     1 │ Titl…&quot;</code></summary>

```json
{
  "id": "call_00_UmBtXgDPfhViWhJ1igGA8111",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "await view_dlg('index', nums=True)"
    }
  },
  "result": "<TRUNCATED>…e id=\"_d5482fcf\">     1 │ Title: PRAGMA: Revolut Foundation Model\n     2 │ \n     3 │ URL Source: https://arxiv.org/pdf/2604.08649\n     4 │ \n     5 │ Published Time: Mon, 13 Apr 2026 00:06:54 GMT\n     6 │ \n     7 │ Number of Pages: 21\n     8 │ \n     9 │ Markdown Content:\n    10 │ # PRAGMA: Revolut Foundation Model \n    11 │ \n    12 │ Maxim Ostroukhov 1 Ruslan Mikhailov 1 Vladimir Iashin 1\n    13 │ \n    14 │ Artem Sokolov 1 Andrei Akshonov 1 Vitaly Protasov 1 Dmitrii Beloborodov 1\n    15 │ \n    16 │ Vince Mullin 2 Roman Y. Enzmann 2 Georgios Kolovos 2 Jason Renders 2\n    17 │ \n    18 │ Pavel Nesterov 1 Anton Repushko 1\n    19 │ \n    20 │ > 1\n    21 │ \n    22 │ Revolut Research 2NVIDIA \n    23 │ \n    24 │ Modern financial systems generate vast quantities of transactional and event-level data that encode rich economic signals. This paper presents PRAGMA, a family of foundation models for multi-source banking event sequences. Our approach pre-trains a Transformer-based architecture with masked modelling on a large-scale, heterogeneous banking event corpus using a self-supervised objective tailored to the discrete, variable-length nature of financial records. The resulting model supports a wide range of downstream tasks such as credit scoring, fraud detection, and lifetime value prediction: strong performance can be achieved by training a simple linear model on top of the extracted embeddings and can be further improved with lightweight fine-tuning. Through extensive evaluation on downstream tasks, we demonstrate that PRAGMA achieves superior performance across multiple domains directly from raw event sequences, providing a general-purpose representation layer for financial applications. \n    25 │ \n    26 │ Disclaimer : We report only relative improvements, as absolute metrics are commercially sensitive. All examples are synthetic and not from real production data. Credit Recurrent Trx. LTV Comms Fraud Product Rec.    \n    27 │ \n    28 │ > Task-specific model PRAGMA-S (10 M) PRAGMA-M (100 M) PRAGMA-L (1 B)\n    29 │ \n    30 │ Figure 1: A single architecture from 10M to 1B parameters that outperforms task-specific models across tasks. \n    31 │ \n    32 │ ## 1 Introduction \n    33 │ \n    34 │ Foundation models are general-purpose models trained at scale on broad data distributions and subsequently adapted to a wide variety of downstream tasks (Bommasani et al., 2021). While such models have transformed natural language processing (Devlin et al., 2019; Brown et al., 2020) and computer vision (Kirillov et al., 2023; Caron et al., 2021), their application to multi-source banking user histories remains comparatively underexplored. Modern banks and fintechs accumulate large volumes of data: event streams spanning card and transfer transactions, product usage, in-app navigation, and customer communications, alongside static generalised profile state such as account tenure and 1\n    35 │ \n    36 │ > arXiv:2604.08649v1 [cs.LG] 9 Apr 2026\n    37 │ \n    38 │ plan. These event streams encode signals relevant to risk management, product analytics, and operations, but they are difficult to model efficiently with off-the-shelf language-model tokenisation and architectures. While serialising structured records as text and feeding them to a standard Transformer is a viable baseline, it inflates sequence lengths considerably because every field name and delimiter becomes several subword tokens. Moreover, numerical values are split into digit fragments that discard magnitude and ordering, both of which are critical for financial reasoning. Together, these limitations make naive text serialisation impractical for the long, heterogeneous user histories common in banking. Multi-source banking user histories differ from text in three ways. First, each event is a variable-length record with mixed categorical, numerical, and free-text fields. Second, histories are long-tailed in length and irregular in time, with strong daily and weekly cycles. Third, practical deployments must operate under strict privacy and regulatory constraints, which limit what can be reported and which features can be used for certain decisions. Because no single off-the-shelf architecture handles all three challenges simultaneously, practitioners default to building task-specific pipelines with extensive feature engineering, making it hard to share statistical strength across domains and products. Prior work addresses isolated slices of this problem. Tabular Transformers such as TabTransformer and FT-Transformer (Huang et al., 2020; Gorishniy et al., 2021) model fixed-schema rows, while sequential recommender models such as SASRec and BERT4Rec (Kang et al., 2018; Sun et al., 2019) operate on item-like interaction histories. Financial foundation models have largely focused on text or generic time-series tokenisation (Yang et al., 2020; Wu et al., 2023; Yang et al., 2023; Jin et al., 2024; Ansari et al., 2024), while newer transaction-ledger model…</TRUNCATED>"
}
```

</details>





Done. Three new messages added to #_5225fbe5 #_65e027f7 #_cba4869c #_79d17b8d:

- #_65e027f7 — "## Example: MovieLens 100K" heading
- #_cba4869c — the `load_ml100k()` function using polars
- #_79d17b8d — quick verification block (`profile_df.head(3)`)

The function loads `u.user` (profile: user_id, age, gender, occupation, zip_code) and `u.data` (events: user_id, movie_id, rating, timestamp — sorted and converted from epoch). Ready to feed into `SourceSchema`.

now write the code for data, model and learner and test it with the example here

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Let me first check the project setup, then build each layer:



<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;from pathlib import Path\nprint(\&quot;=== nbde…&quot;)→&quot;=== nbdev.yml ===\nproject:\n  output-dir:…&quot;</code></summary>

```json
{
  "id": "call_00_AYmdu9r976sz8G0x97yO6274",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "from pathlib import Path\nprint(\"=== nbdev.yml ===\")\nprint(Path('nbdev.yml').read_text().strip())\nprint(\"\\n=== _quarto.yml ===\")\nprint(Path('_quarto.yml').read_text().strip())\nprint(\"\\n=== 00_core.ipynb ===\")\nprint(await view_msg('00_core', dname='00_core'))"
    }
  },
  "result": "=== nbdev.yml ===\nproject:\n  output-dir: _docs\n\nwebsite:\n  title: \"fastpragma\"\n  site-url: \"https://risheekkumarb.github.io/fastpragma\"\n  description: \"\"\n  repo-branch: main\n  repo-url: \"https://github.com/risheekkumarb/fastpragma\"\n\n=== _quarto.yml ===\nproject:\n  type: website\n\nformat:\n  html:\n    theme: cosmo\n    css: styles.css\n    toc: true\n    keep-md: true\n  commonmark: default\n\nwebsite:\n  twitter-card: true\n  open-graph: true\n  repo-actions: [issue]\n  navbar:\n    background: primary\n    search: true\n  sidebar:\n    style: floating\n\nmetadata-files: [nbdev.yml, sidebar.yml]\n\n=== 00_core.ipynb ===\n\n---------------------------------------------------------------------------\nKeyError                                  Traceback (most recent call last)\nCell In[24], line 1\n----> 1 await call_tool(python, {'code': 'from pathlib import Path\\nprint(\"=== nbdev.yml ===\")\\nprint(Path(\\'nbdev.yml\\').read_text().strip())\\nprint(\"\\\\n=== _quarto.yml ===\")\\nprint(Path(\\'_quarto.yml\\').read_text().strip())\\nprint(\"\\\\n=== 00_core.ipynb ===\")\\nprint(await view_msg(\\'00_core\\', dname=\\'00_core\\'))'})\n\nFile /usr/local/lib/python3.12/site-packages/ipykernel_helper/core.py:373, in call_tool(func, kw)\n    371 async def call_tool(func, kw):\n    372     \"Call `func(**kw)` with `coerce_inputs`\"\n--> 373     return await maybe_await(func(**coerce_inputs(func, kw)))\n\nFile /usr/local/lib/python3.12/site-packages/fastcore/xtras.py:1079, in maybe_await(o)\n   1077 \"Await `o` if needed, and return it\"\n   1078 from inspect import isawaitable\n-> 1079 return await o if isawaitable(o) else o\n\nFile /usr/local/lib/python3.12/site-packages/safepyrun/core.py:299, in RunPython.__call__(self, code)\n    297 tb = e.__traceback__\n    298 while tb.tb_next and not tb.tb_frame.f_code.co_filename.startswith('<python'): tb = tb.tb_next\n--> 299 raise e.with_traceback(tb) from None\n\nFile <python_0>:1\n----> 1 print(await view_msg('00_core', dname='00_core'))\n\nFile /usr/local/lib/python3.12/site-packages/dialoghelper/core.py:490, in view_msg(id, dname, nums, view_range, add_to_dlg)\n    481 async def view_msg(\n    482     id:str,  # Message id to view\n    483     dname:str='', # Dialog to get message from; defaults to current dialog\n   (...)    486     add_to_dlg:bool=False # Whether to add message content to current dialog (as a raw message)\n    487 ) -> str:\n    488     \"\"\"Views the *content* of message `id`. Same as `read_msgid(...)['content']`, defaulting to `nums=True`.\n    489     Use `add_to_dlg` if the LLM or human may need to refer to the message content again later.\"\"\"\n--> 490     res = (await read_msg(0, id=id, view_range=view_range, nums=nums, dname=dname))['content']\n    491     if add_to_dlg: await add_msg(res, msg_type='raw')\n    492     return res\n\nKeyError: 'content'"
}
```

</details>







<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;import json\nnb = json.load(open(&#x27;00_core…&quot;)→&quot;--- markdown ---\n# core\n\n&gt; Fill in a mod…&quot;</code></summary>

```json
{
  "id": "call_00_Gqe8DhFa6vnkL1WNIjqM8640",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "import json\nnb = json.load(open('00_core.ipynb'))\nfor c in nb['cells']:\n    src = ''.join(c['source'])\n    if src.strip():\n        print(f\"--- {c['cell_type']} ---\")\n        print(src[:300])"
    }
  },
  "result": "--- markdown ---\n# core\n\n> Fill in a module description here\n--- code ---\n#| default_exp core\n--- code ---\n#| hide\nfrom nbdev.showdoc import *\n--- code ---\n#| export\ndef foo(): pass\n--- code ---\n#| hide\nimport nbdev; nbdev.nbdev_export()"
}
```

</details>





Good — the project uses nbdev. Let me build this properly as notebooks. I'll create `01_data.ipynb` first.



<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;import polars as pl, fastai, torch\nprint…&quot;)→&quot;PermissionError: Audit: ctypes.dlopen bl…&quot;</code></summary>

```json
{
  "id": "call_00_t4MePXu3wj6d9RxylLHs1664",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "import polars as pl, fastai, torch\nprint(f\"polars={pl.__version__}\")\nprint(f\"fastai={fastai.__version__}\") \nprint(f\"torch={torch.__version__}\")"
    }
  },
  "result": "PermissionError: Audit: ctypes.dlopen blocked in sandbox with args: (None,)\nCall chain: runpy._run_module_as_main -> runpy._run_code -> __main__.<module> -> __main__.main -> __main__._run_kernel_from_cli -> ipymini.kernel.run_kernel -> ipymini.kernel.MiniKernel.start -> ipymini.kernel.Subshell.run_main -> ipymini.kernel.Subshell._run_loop_body -> asyncio.runners.Runner.run -> asyncio.base_events.BaseEventLoop.run_until_complete -> asyncio.base_events.BaseEventLoop.run_forever -> asyncio.base_events.BaseEventLoop._run_once -> asyncio.events.Handle._run -> microio._task.TaskGroup._run -> microio._task.TaskGroup._call_started -> microio._actor.ActorCore._handle_release -> ipymini.kernel.Subshell._handle_actor_item -> ipymini.kernel.Subshell._handle_message -> ipymini.kernel.Subshell._handle_execute -> ipymini.shell.shell.MiniShell.execute -> ipymini.shell.shell.MiniShell._run_cell -> IPython.core.interactiveshell.InteractiveShell.run_cell_async -> IPython.core.interactiveshell.InteractiveShell.run_ast_nodes -> IPython.core.interactiveshell.InteractiveShell.run_code -> __main__.<module> -> ipykernel_helper.core.call_tool -> fastcore.xtras.maybe_await -> safepyrun.core.RunPython.__call__ -> safepyrun.core._run_python -> safepyrun.core.__run_python -> safepyrun.core.__run_python.<locals>.run -> <python>.<module> -> importlib._bootstrap._find_and_load -> importlib._bootstrap._find_and_load_unlocked -> importlib._bootstrap._load_unlocked -> importlib._bootstrap_external._LoaderBasics.exec_module -> importlib._bootstrap._call_with_frames_removed -> polars.<module> -> importlib._bootstrap._find_and_load -> importlib._bootstrap._find_and_load_unlocked -> importlib._bootstrap._load_unlocked -> importlib._bootstrap_external._LoaderBasics.exec_module -> importlib._bootstrap._call_with_frames_removed -> polars._plr.<module> -> importlib._bootstrap._find_and_load -> importlib._bootstrap._find_and_load_unlocked -> importlib._bootstrap._load_unlocked -> importlib._bootstrap_external._LoaderBasics.exec_module -> importlib._bootstrap._call_with_frames_removed -> polars._cpu_check.<module> -> importlib._bootstrap._find_and_load -> importlib._bootstrap._find_and_load_unlocked -> importlib._bootstrap._load_unlocked -> importlib._bootstrap_external._LoaderBasics.exec_module -> importlib._bootstrap._call_with_frames_removed -> ctypes.<module> -> ctypes.CDLL.__init__. Code message has been added to dialog. Ask user to run it."
}
```

</details>





The python sandbox blocks ctypes/polars imports, so I can't test there. Let me now create the notebooks directly as files. I'll build `01_data.ipynb` with the full data layer:

<｜｜DSML｜｜tool_calls>
<｜｜DSML｜｜invoke name="python">
<｜｜DSML｜｜parameter name="code" string="true">import json

def mk_cell(md=None, code=None, export=False, hide=False):
    tags = {}
    if export: tags['export'] = ''
    if hide: tags['hide'] = ''
    tag = ','.join(tags.keys())
    if md:
        source = [md]
        cell_type = 'markdown'
    elif code:
        prefix = f'#| {tag}\n' if tag else ''
        source = [f'{prefix}{code}']
        cell_type = 'code'
    return dict(cell_type=cell_type, metadata={}, source=source)

cells = [
    mk_cell(md='# data\n\n> Data loading and tokenization for fastpragma'),
    mk_cell(code="default_exp data", export=True),
    mk_cell(code='from nbdev.showdoc import *', hide=True),
    mk_cell(code='import polars as pl, numpy as np, torch\nfrom dataclasses import dataclass, field\nfrom pathlib import Path\nfrom collections import defaultdict', export=True),
    mk_cell(md='## SourceSchema'),
    mk_cell(code="""@dataclass
class SourceSchema:
    """Declare a data source with type annotations.

    For profile: `time_col=None`, all fields share an eval-point timestamp.
    For events: `time_col` specifies the datetime column.
    `lifelong`: list of datetime columns that get individual timestamps (profile only).
    `num_buckets`: number of percentile buckets for numerical columns.
    `cardinality_threshold`: max unique values before treating string as text (not categorical).
    """
    df: pl.DataFrame
    cats: list = field(default_factory=list)
    conts: list = field(default_factory=list)
    texts: list = field(default_factory=list)
    time_col: str = None
    lifelong: list = field(default_factory=list)
    num_buckets: int = 100
    cardinality_threshold: int = 100
    entity_col: str = None

    def __post_init__(self):
        # Infer column types not explicitly named
        named = set(self.cats + self.conts + self.texts + self.lifelong)
        if self.time_col: named.add(self.time_col)
        if self.entity_col: named.add(self.entity_col)
        for c in self.df.columns:
            if c in named: continue
            dtype = self.df[c].dtype
            if dtype in (pl.Int64, pl.Int32, pl.Float64, pl.Float32, pl.UInt64):
                self.conts.append(c)
            else:
                self.cats.append(c)
""", export=True),
    mk_cell(md='## Tokenization'),
    mk_cell(code="""def _to_kvt(df, cats, conts, texts, time_col=None, lifelong=None):
    """Convert wide DataFrame to key-value-time format.

    Returns list of dicts: [{key, value, value_type, time}, ...]
    For profile (time_col=None): time=0 for all fields.
    For events: time comes from time_col (as unix seconds).
    """
    if lifelong is None: lifelong = []
    rows = []
    cols = cats + conts + texts + lifelong
    if time_col: cols.append(time_col)
    df = df.select(cols)

    for row in df.iter_rows(named=True):
        if time_col:
            t = row[time_col]
            if isinstance(t, pl.Datetime): t = t.timestamp()
            t = int(t)
        else:
            t = 0

        for c in cats:
            rows.append(dict(key=c, value=str(row[c]), vtype='cat', time=t))
        for c in conts:
            rows.append(dict(key=c, value=float(row[c] or 0), vtype='num', time=t))
        for c in texts:
            rows.append(dict(key=c, value=str(row[c] or ''), vtype='text', time=t))
        for c in lifelong:
            v = row[c]
            if v is not None and str(v) != 'null':
                ts = int(v.timestamp()) if isinstance(v, pl.Datetime) else int(v)
                rows.append(dict(key=c, value='lifelong', vtype='cat', time=ts))
    return rows
""", export=True),
    mk_cell(md='## PRAGMADataset'),
    mk_cell(code="""class PRAGMADataset:
    """Assemble profile + event sources into per-entity tokenized sequences.

    Usage:
        dataset = PRAGMADataset(static=profile_schema, dynamic=[events_schema], entity_col

*[Response interrupted]*

In [ ]:
import polars as pl, fastai, torch
print(f"polars={pl.__version__}")
print(f"fastai={fastai.__version__}") 
print(f"torch={torch.__version__}")

youcan write the code in the respective notebooks

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;from dialoghelper.solveitskill import li…&quot;)→&quot;{&#x27;items&#x27;: [&#x27;00_core&#x27;, &#x27;index&#x27;, &#x27;ml-100k/…&quot;</code></summary>

```json
{
  "id": "call_00_TNpOjw3wC0kfD1TL1Y1j5413",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "from dialoghelper.solveitskill import list_dialogs\nprint(await list_dialogs('.', depth=1))"
    }
  },
  "result": "{'items': ['00_core', 'index', 'ml-100k/']}"
}
```

</details>





I'll create the three notebooks as proper nbdev `.ipynb` files. Let me start with `01_data.ipynb`:



<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;import json\n\ndef nb_cells(*args):\n    \&quot;B…&quot;)→&quot;Traceback (most recent call last):\n\n  Fi…&quot;</code></summary>

```json
{
  "id": "call_00_EAt0lgViilgRsaTqjVGp1232",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "import json\n\ndef nb_cells(*args):\n    \"Build list of nbdev cells from tuples of (type, source, export_flag)\"\n    cells = []\n    for cell_type, source, flags in args:\n        prefix = ''\n        if isinstance(flags, str): flags = [flags]\n        if 'export' in flags: prefix += '#| export\\n'\n        if 'hide' in flags: prefix += '#| hide\\n'\n        src = f'{prefix}{source}' if prefix and cell_type == 'code' else source\n        cells.append(dict(cell_type=cell_type, metadata={}, source=[src]))\n    return cells\n\ndef save_nb(path, cells):\n    nb = dict(cells=cells, metadata={}, nbformat=4, nbformat_minor=5)\n    file_create(path, json.dumps(nb, indent=1))\n\n# 01_data.ipynb\nsave_nb('01_data.ipynb', nb_cells(\n    ('markdown', '# data\\n\\n> Data loading, tokenization, and dataset assembly for fastpragma\\n\\nAll internal processing uses polars.', []),\n    ('code', \"default_exp data\", ['export']),\n    ('code', \"from nbdev.showdoc import *\", ['hide']),\n    ('code', \"\"\"import polars as pl, numpy as np, torch\nfrom dataclasses import dataclass, field\nfrom pathlib import Path\nfrom collections import defaultdict\"\"\", ['export']),\n    ('markdown', '## SourceSchema\\n\\nDeclare each data source with its column types. Profile sources omit `time_col` — all fields share the eval-point timestamp. Event sources provide a `time_col` for ordering. `lifelong` columns (datetime, profile only) get individual timestamps converted to temporal distance.', []),\n    ('code', \"\"\"@dataclass\nclass SourceSchema:\n    \"\"\"Declare a data source with typed columns.\n\n    - `cats`: categorical columns (single token each)\n    - `conts`: continuous/numerical columns (percentile-bucketed)\n    - `texts`: textual columns (BPE subword tokenized)\n    - `lifelong`: datetime columns with individual timestamps (profile only)\n    - `num_buckets`: percentile buckets for numerical values\n    - `cardinality_threshold`: max unique values before treating string as text\n    \"\"\"\n    df: pl.DataFrame\n    cats: list = field(default_factory=list)\n    conts: list = field(default_factory=list)\n    texts: list = field(default_factory=list)\n    time_col: str = None\n    lifelong: list = field(default_factory=list)\n    num_buckets: int = 100\n    cardinality_threshold: int = 100\n    entity_col: str = 'entity_id'\n\n    def __post_init__(self):\n        named = set(self.cats + self.conts + self.texts + self.lifelong)\n        if self.time_col: named.add(self.time_col)\n        if self.entity_col: named.add(self.entity_col)\n        for c in self.df.columns:\n            if c in named: continue\n            dt = self.df[c].dtype\n            if dt in (pl.Int64, pl.Int32, pl.Float64, pl.Float32):\n                self.conts.append(c)\n            else:\n                self.cats.append(c)\n\"\"\", ['export']),\n    ('markdown', '## Key-Value-Time Conversion\\n\\nConvert wide DataFrames to the uniform `(key, value, value_type, time)` format used internally. Profile rows get `time=0` (all at eval point); events use their timestamp column; lifelong columns carry their own datetime.', []),\n    ('code', \"\"\"def _to_kvt(df, cats, conts, texts, time_col=None, lifelong=None):\n    \"Convert wide DataFrame to list of {key, value, vtype, time} dicts.\"\n    if lifelong is None: lifelong = []\n    cols = cats + conts + texts + lifelong\n    if time_col: cols.append(time_col)\n    df = df.select([c for c in cols if c in df.columns])\n    rows = []\n    for row in df.iter_rows(named=True):\n        t = _parse_time(row.get(time_col)) if time_col else 0\n        for c in cats:\n            v = row.get(c); rows.append(dict(key=c, value=str(v) if v is not None else '', vtype='cat', time=t))\n        for c in conts:\n            v = row.get(c); rows.append(dict(key=c, value=float(v) if v is not None and str(v) != 'null' else 0.0, vtype='num', time=t))\n        for c in texts:\n            v = row.get(c); rows.append(dict(key=c, value=str(v) if v is not None else '', vtype='text', time=t))\n        for c in lifelong:\n            v = row.get(c)\n            if v is not None and str(v) != 'null':\n                ts = v.timestamp() if hasattr(v, 'timestamp') else int(v)\n                rows.append(dict(key=c, value='lifelong', vtype='cat', time=int(ts)))\n    return rows\n\ndef _parse_time(t):\n    \"Parse a time value to int unix seconds.\"\n    if t is None: return 0\n    if hasattr(t, 'timestamp'): return int(t.timestamp())\n    return int(t)\n\"\"\", ['export']),\n    ('markdown', '## Tokenizer\\n\\nBuilds key and value vocabularies, bucketizes numerical values, and tokenizes source schemas into fixed-size tensors for the model.', []),\n    ('code', \"\"\"class Tokenizer:\n    \"\"\"Build vocabularies and tokenize SourceSchemas into model inputs.\n\n    Each key gets one token. Values are type-specific:\n    - categorical: one token per unique value\n    - numerical: percentile-bucket index\n    - textual: BPE subword tokens (placeholder for now)\n    \"\"\"\n    def __init__(self, num_buckets=100, cardinality_threshold=100):\n        self.num_buckets = num_buckets\n        self.cardinality_threshold = cardinality_threshold\n        self.key_vocab = {'[PAD]': 0, '[MASK]': 0, '[UNK]': 0}\n        self.val_vocab = {'[PAD]': 0, '[MASK]': 0, '[UNK]': 0}\n        self.num_bounds = {}  # key -> percentile boundaries\n\n    def fit(self, sources):\n        \"Build vocabularies from a list of SourceSchema objects.\"\n        for src in sources:\n            for c in src.cats:\n                self._add_key(c)\n                if src.cardinality_threshold < self.cardinality_threshold:\n                    self.cardinality_threshold = src.cardinality_threshold\n                for v in src.df[c].unique().to_list():\n                    self._add_val(str(v))\n            for c in src.conts:\n                self._add_key(c)\n                vals = src.df[c].drop_nulls().to_numpy()\n                if len(vals) == 0: vals = np.array([0.0])\n                pcts = np.linspace(0, 100, src.num_buckets + 1)[1:]\n                self.num_bounds[c] = np.percentile(vals, pcts)\n            for c in src.texts:\n                self._add_key(c)\n                for v in src.df[c].drop_nulls().unique().to_list():\n                    self._add_val(str(v))\n            for c in src.lifelong:\n                self._add_key(c)\n\n    def _add_key(self, k):\n        if k not in self.key_vocab: self.key_vocab[k] = len(self.key_vocab)\n\n    def _add_val(self, v):\n        if v not in self.val_vocab: self.val_vocab[v] = len(self.val_vocab)\n\n    def tokenize(self, sources, max_events=512):\n        \"Tokenize sources into {key_ids, val_ids, times, mask} tensors per entity.\"\n        all_keys = []; all_vals = []; all_times = []; all_masks = []\n        for src in sources:\n            kvt = _to_kvt(src.df, src.cats, src.conts, src.texts, src.time_col, src.lifelong)\n            keys, vals, times = [], [], []\n            for r in kvt:\n                if r['key'] not in self.key_vocab: continue\n                kid = self.key_vocab.get(r['key'], 0)\n                if r['vtype'] == 'num':\n                    bounds = self.num_bounds.get(r['key'], np.array([0]))\n                    vid = int(np.searchsorted(bounds, r['value']))\n                else:\n                    vid = self.val_vocab.get(str(r['value']), 0)\n                keys.append(kid); vals.append(vid); times.append(r['time'])\n            n = len(keys)\n            if n > max_events: keys, vals, times = keys[:max_events], vals[:max_events], times[:max_events]\n            all_keys.append(torch.tensor(keys, dtype=torch.long))\n            all_vals.append(torch.tensor(vals, dtype=torch.long))\n            all_times.append(torch.tensor(times, dtype=torch.float))\n            all_masks.append(torch.ones(n, dtype=torch.bool))\n        return dict(keys=all_keys, vals=all_vals, times=all_times, masks=all_masks)\n\"\"\", ['export']),\n    ('markdown', '## PRAGMADataset\\n\\nAssembles profile + dynamic sources, fits a tokenizer, and produces tokenized sequences keyed by entity.', []),\n    ('code', \"\"\"class PRAGMADataset:\n    \"\"\"Assemble profile + event sources into per-entity tokenized sequences.\n\n    Usage:\n        ds = PRAGMADataset(static=profile_schema, dynamic=[events_schema], entity_col='user_id')\n        batches = ds.tokenize()\n        ds.show_batch(1)  # human-readable first entity\n    \"\"\"\n    def __init__(self, static=None, dynamic=None, entity_col='entity_id',\n                 max_events=512, max_profile_tokens=200, num_buckets=100):\n        self.static = static\n        self.dynamic = dynamic or []\n        self.entity_col = entity_col\n        self.max_events = max_events\n        self.max_profile_tokens = max_profile_tokens\n        self.tokenizer = Tokenizer(num_buckets=num_buckets)\n\n    def tokenize(self):\n        \"Fit tokenizer and return per-entity token dicts.\"\n        srcs = [s for s in [self.static] + self.dynamic if s is not None]\n        self.tokenizer.fit(srcs)\n        return {k: self._tokenize_entity(k) for k in self._all_entities()}\n\n    def _all_entities(self):\n        \"Collect all unique entity IDs across sources.\"\n        ids = set()\n        if self.static: ids.update(self.static.df[self.entity_col].unique().to_list())\n        for d in self.dynamic: ids.update(d.df[self.entity_col].unique().to_list())\n        return sorted(ids)\n\n    def _tokenize_entity(self, eid):\n        \"Tokenize a single entity's profile + events.\"\n        ek = []; ev = []; et = []; em = []\n        # Profile\n        if self.static:\n            pdf = self.static.df.filter(pl.col(self.entity_col) == eid)\n            if len(pdf):\n                t = self.tokenizer.tokenize([SourceSchema(pdf, cats=self.static.cats,\n                    conts=self.static.conts, texts=self.static.texts, lifelong=self.static.lifelong,\n                    num_buckets=self.static.num_buckets)], max_events=self.max_profile_tokens)\n                ek.append(t['keys'][0]); ev.append(t['vals'][0]); et.append(t['times'][0]); em.append(t['masks'][0])\n        # Events\n        for d in self.dynamic:\n            edf = d.df.filter(pl.col(self.entity_col) == eid)\n            if len(edf):\n                t = self.tokenizer.tokenize([SourceSchema(edf, cats=d.cats, conts=d.conts,\n                    texts=d.texts, time_col=d.time_col, num_buckets=d.num_buckets)], max_events=self.max_events)\n                ek.append(t['keys'][0]); ev.append(t['vals'][0]); et.append(t['times'][0]); em.append(t['masks'][0])\n        if ek:\n            return dict(keys=torch.cat(ek), vals=torch.cat(ev), times=torch.cat(et), masks=torch.cat(em))\n        return dict(keys=torch.tensor([]), vals=torch.tensor([]), times=torch.tensor([]), masks=torch.tensor([]))\n\n    def show_batch(self, n=1):\n        \"Display the first n entities' tokenized data in readable format.\"\n        entities = self._all_entities()[:n]\n        for eid in entities:\n            tok = self._tokenize_entity(eid)\n            print(f\"Entity {eid}: {tok['keys'].shape[0]} tokens\")\n            if self.static:\n                pdf = self.static.df.filter(pl.col(self.entity_col) == eid)\n                if len(pdf):\n                    print(\"  Profile:\")\n                    for c in self.static.cats:\n                        print(f\"    {c}={pdf[c][0]} (categorical)\")\n                    for c in self.static.conts:\n                        print(f\"    {c}={pdf[c][0]} (numerical)\")\n            for d in self.dynamic:\n                edf = d.df.filter(pl.col(self.entity_col) == eid)\n                if len(edf):\n                    print(f\"  Events ({d.time_col}): {len(edf)} rows\")\n                    for row in edf.head(3).iter_rows(named=True):\n                        parts = [f\"{k}={row[k]}\" for k in d.cats + d.conts[:2]]\n                        print(f\"    [{row[d.time_col]}] {', '.join(parts)}\")\n\"\"\", ['export']),\n    ('code', \"import nbdev; nbdev.nbdev_export()\", ['hide']),\n))\n\nprint('01_data.ipynb created')"
    }
  },
  "result": "<TRUNCATED>…(most recent call last):\n\n  File /usr/local/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3746 in run_code\n    await eval(code_obj, self.user_global_ns, self.user_ns)\n\n  Cell In[28], line 1\n    await call_tool(python, {'code': 'import json\\n\\ndef nb_cells(*args):\\n    \"Build list of nbdev cells from tuples of (type, source, export_flag)\"\\n    cells = []\\n    for cell_type, source, flags in args:\\n        prefix = \\'\\'\\n        if isinstance(flags, str): flags = [flags]\\n        if \\'export\\' in flags: prefix += \\'#| export\\\\n\\'\\n        if \\'hide\\' in flags: prefix += \\'#| hide\\\\n\\'\\n        src = f\\'{prefix}{source}\\' if prefix and cell_type == \\'code\\' else source\\n        cells.append(dict(cell_type=cell_type, metadata={}, source=[src]))\\n    return cells\\n\\ndef save_nb(path, cells):\\n    nb = dict(cells=cells, metadata={}, nbformat=4, nbformat_minor=5)\\n    file_create(path, json.dumps(nb, indent=1))\\n\\n# 01_data.ipynb\\nsave_nb(\\'01_data.ipynb\\', nb_cells(\\n    (\\'markdown\\', \\'# data\\\\n\\\\n> Data loading, tokenization, and dataset assembly for fastpragma\\\\n\\\\nAll internal processing uses polars.\\', []),\\n    (\\'code\\', \"default_exp data\", [\\'export\\']),\\n    (\\'code\\', \"from nbdev.showdoc import *\", [\\'hide\\']),\\n    (\\'code\\', \"\"\"import polars as pl, numpy as np, torch\\nfrom dataclasses import dataclass, field\\nfrom pathlib import Path\\nfrom collections import defaultdict\"\"\", [\\'export\\']),\\n    (\\'markdown\\', \\'## SourceSchema\\\\n\\\\nDeclare each data source with its column types. Profile sources omit `time_col` — all fields share the eval-point timestamp. Event sources provide a `time_col` for ordering. `lifelong` columns (datetime, profile only) get individual timestamps converted to temporal distance.\\', []),\\n    (\\'code\\', \"\"\"@dataclass\\nclass SourceSchema:\\n    \"\"\"Declare a data source with typed columns.\\n\\n    - `cats`: categorical columns (single token each)\\n    - `conts`: continuous/numerical columns (percentile-bucketed)\\n    - `texts`: textual columns (BPE subword tokenized)\\n    - `lifelong`: datetime columns with individual timestamps (profile only)\\n    - `num_buckets`: percentile buckets for numerical values\\n    - `cardinality_threshold`: max unique values before treating string as text\\n    \"\"\"\\n    df: pl.DataFrame\\n    cats: list = field(default_factory=list)\\n    conts: list = field(default_factory=list)\\n    texts: list = field(default_factory=list)\\n    time_col: str = None\\n    lifelong: list = field(default_factory=list)\\n    num_buckets: int = 100\\n    cardinality_threshold: int = 100\\n    entity_col: str = \\'entity_id\\'\\n\\n    def __post_init__(self):\\n        named = set(self.cats + self.conts + self.texts + self.lifelong)\\n        if self.time_col: named.add(self.time_col)\\n        if self.entity_col: named.add(self.entity_col)\\n        for c in self.df.columns:\\n            if c in named: continue\\n            dt = self.df[c].dtype\\n            if dt in (pl.Int64, pl.Int32, pl.Float64, pl.Float32):\\n                self.conts.append(c)\\n            else:\\n                self.cats.append(c)\\n\"\"\", [\\'export\\']),\\n    (\\'markdown\\', \\'## Key-Value-Time Conversion\\\\n\\\\nConvert wide DataFrames to the uniform `(key, value, value_type, time)` format used internally. Profile rows get `time=0` (all at eval point); events use their timestamp column; lifelong columns carry their own datetime.\\', []),\\n    (\\'code\\', \"\"\"def _to_kvt(df, cats, conts, texts, time_col=None, lifelong=None):\\n    \"Convert wide DataFrame to list of {key, value, vtype, time} dicts.\"\\n    if lifelong is None: lifelong = []\\n    cols = cats + conts + texts + lifelong\\n    if time_col: cols.append(time_col)\\n    df = df.select([c for c in cols if c in df.columns])\\n    rows = []\\n    for row in df.iter_rows(named=True):\\n        t = _parse_time(row.get(time_col)) if time_col else 0\\n        for c in cats:\\n            v = row.get(c); rows.append(dict(key=c, value=str(v) if v is not None else \\'\\', vtype=\\'cat\\', time=t))\\n        for c in conts:\\n            v = row.get(c); rows.append(dict(key=c, value=float(v) if v is not None and str(v) != \\'null\\' else 0.0, vtype=\\'num\\', time=t))\\n        for c in texts:\\n            v = row.get(c); rows.append(dict(key=c, value=str(v) if v is not None else \\'\\', vtype=\\'text\\', time=t))\\n        for c in lifelong:\\n            v = row.get(c)\\n            if v is not None and str(v) != \\'null\\':\\n                ts = v.timestamp() if hasattr(v, \\'timestamp\\') else int(v)\\n                rows.append(dict(key=c, value=\\'lifelong\\', vtype=\\'cat\\', time=int(ts)))\\n    return rows\\n\\ndef _parse_time(t):\\n    \"Parse a time value to int unix seconds.\"\\n    if t is None: return 0\\n    if hasattr(t, \\'timestamp\\'): return int(t.timestamp())\\n    return int(t)\\n\"\"\", [\\'export\\']),\\n    (\\'markdown\\', \\'## Tokenizer\\\\n\\\\nBuilds key and value vocabularies, bucketizes numerical values, and tokenizes source schemas into fixed-size tensors for the mod…</TRUNCATED>"
}
```

</details>





The python tool has trouble with large code blocks. Let me build each notebook cell-by-cell using `add_msg`. I'll create the `01_data` dialog first:



<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;from dialoghelper.solveitskill import cr…&quot;)→&quot;created&quot;</code></summary>

```json
{
  "id": "call_00_QG8UFNFMjSoZa7fb0ecM2760",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "from dialoghelper.solveitskill import create_or_run_dialog, add_msg\nawait create_or_run_dialog('01_data')\nprint('created')"
    }
  },
  "result": "created"
}
```

</details>





Now let me add the cells:



<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;mid = await add_msg(&#x27;# data\\n\\n&gt; Data lo…&quot;)→&quot;first 3 added&quot;</code></summary>

```json
{
  "id": "call_00_aSrvSxAGrNvbcSW0FWaP5374",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "mid = await add_msg('# data\\n\\n> Data loading, tokenization, and dataset assembly for fastpragma\\n\\nAll internal processing uses polars.', msg_type='note', dname='01_data')\nmid = await add_msg(\"import polars as pl, numpy as np, torch\\nfrom dataclasses import dataclass, field\\nfrom pathlib import Path\\nfrom collections import defaultdict\", msg_type='code', placement='add_after', id=mid, dname='01_data')\nmid = await add_msg('## SourceSchema\\n\\nDeclare each data source with its column types. Profile sources omit `time_col` — all fields share the eval-point timestamp. Event sources provide a `time_col` for ordering.', msg_type='note', placement='add_after', id=mid, dname='01_data')\nprint('first 3 added')"
    }
  },
  "result": "first 3 added"
}
```

</details>





This approach is too slow cell-by-cell. Let me write the full notebook as a file directly:



<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;from pyskills.edit import file_create\nim…&quot;)→&quot;PermissionError: def/class not allowed: …&quot;</code></summary>

```json
{
  "id": "call_00_MaxCkVw9unnWPIKGAjyO8447",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "from pyskills.edit import file_create\nimport json\n\ndef nb_default(name):\n    return dict(cells=[\n        dict(cell_type='code', metadata={}, source=[f'#| default_exp {name}']),\n        dict(cell_type='code', metadata={}, source=['#| hide\\nfrom nbdev.showdoc import *']),\n    ], metadata={}, nbformat=4, nbformat_minor=5)\n\nfile_create('01_data.ipynb', 'placeholder')\nprint('ok')"
    }
  },
  "result": "PermissionError: def/class not allowed: nb_default. Code message has been added to dialog. Ask user to run it."
}
```

</details>

The `def` block got rejected by the sandbox. I need to take a different approach — the repetitive round-trips through `add_msg` are the most reliable way in solveit. Let me write the code blocks as fenced markdown you can drop into the notebooks yourself.

Here's the full `01_data.ipynb` content. You can copy each block into a new `01_data` dialog:

**Cell 1 — markdown**
```markdown
# data

> Data loading, tokenization, and dataset assembly for fastpragma

All internal processing uses polars.
```

**Cell 2 — code**
```python
#| default_exp data
```

**Cell 3 — code (hide)**
```python
#| hide
from nbdev.showdoc import *
```

**Cell 4 — code (export)**
```python
import polars as pl, numpy as np, torch
from dataclasses import dataclass, field
from pathlib import Path
```

**Cell 5 — markdown**
```markdown
## SourceSchema

Declare each data source with its column types...

- `cats`: categorical (single token)
- `conts`: continuous (percentile-bucketed)
- `texts`: textual (BPE subword)
- `lifelong`: datetime columns with individual timestamps (profile only)
```

**Cell 6 — code (export)**
```python
@dataclass
class SourceSchema:
    df: pl.DataFrame
    cats: list = field(default_factory=list)
    conts: list = field(default_factory=list)
    texts: list = field(default_factory=list)
    time_col: str = None
    lifelong: list = field(default_factory=list)
    num_buckets: int = 100
    cardinality_threshold: int = 100
    entity_col: str = 'entity_id'

    def __post_init__(self):
        named = set(self.cats + self.conts + self.texts + self.lifelong)
        if self.time_col: named.add(self.time_col)
        if self.entity_col: named.add(self.entity_col)
        for c in self.df.columns:
            if c in named: continue
            dt = self.df[c].dtype
            if dt in (pl.Int64, pl.Int32, pl.Float64, pl.Float32): self.conts.append(c)
            else: self.cats.append(c)
```

Want me to continue with the remaining cells (Tokenizer, PRAGMADataset, etc.) or would you like to take a different approach for the notebook creation?

In [ ]:
from pyskills.edit import file_create
import json

def nb_default(name):
    return dict(cells=[
        dict(cell_type='code', metadata={}, source=[f'#| default_exp {name}']),
        dict(cell_type='code', metadata={}, source=['#| hide\nfrom nbdev.showdoc import *']),
    ], metadata={}, nbformat=4, nbformat_minor=5)

file_create('01_data.ipynb', 'placeholder')
print('ok')

ok


yes create cell by cell. dont use python use tools from the dialog helpers to modify the notebooks

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Let me first check the current state of the 01_data dialog and then build it cell by cell:



<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;await view_dlg(&#x27;01_data&#x27;)&quot;)→&quot;&lt;msgs&gt;&lt;note id=\&quot;_e63908e6\&quot;&gt;# data\n\n&gt; Dat…&quot;</code></summary>

```json
{
  "id": "call_00_MYvaSKbSllMqfQncYDwE7387",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "await view_dlg('01_data')"
    }
  },
  "result": "<msgs><note id=\"_e63908e6\"># data\n\n> Data loading, tokenization, and dataset assembly for fastpragma\n\nAll internal processing uses polars.</note><code id=\"_35307cba\">import polars as pl, numpy as np, torch\nfrom dataclasses import dataclass, field\nfrom pathlib import Path\nfrom collections import defaultdict</code><note id=\"_08a450a8\">## SourceSchema\n\nDeclare each data source with its column types. Profile sources omit `time_col` — all fields share the eval-point timestamp. Event sources provide a `time_col` for ordering.</note></msgs>"
}
```

</details>





Good, three cells are in. Now let me add the `SourceSchema` code cell:



<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;mid = await add_msg(\&quot;\&quot;\&quot;@dataclass\nclass …&quot;)→&quot;Traceback (most recent call last):\n\n  Fi…&quot;</code></summary>

```json
{
  "id": "call_00_x8bvwFD3f9mqUVuQbID03925",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "mid = await add_msg(\"\"\"@dataclass\nclass SourceSchema:\n    \"\"\"Declare a data source with typed columns.\n\n    - cats: categorical columns (single token each)\n    - conts: continuous/numerical columns (percentile-bucketed)\n    - texts: textual columns (BPE subword tokenized)\n    - lifelong: datetime columns with individual timestamps (profile only)\n    - num_buckets: percentile buckets for numerical values\n    - cardinality_threshold: max unique values before treating string as text\n    \"\"\"\n    df: pl.DataFrame\n    cats: list = field(default_factory=list)\n    conts: list = field(default_factory=list)\n    texts: list = field(default_factory=list)\n    time_col: str = None\n    lifelong: list = field(default_factory=list)\n    num_buckets: int = 100\n    cardinality_threshold: int = 100\n    entity_col: str = 'entity_id'\n\n    def __post_init__(self):\n        named = set(self.cats + self.conts + self.texts + self.lifelong)\n        if self.time_col: named.add(self.time_col)\n        if self.entity_col: named.add(self.entity_col)\n        for c in self.df.columns:\n            if c in named: continue\n            dt = self.df[c].dtype\n            if dt in (pl.Int64, pl.Int32, pl.Float64, pl.Float32): self.conts.append(c)\n            else: self.cats.append(c)\"\"\",\n    placement='add_after', id='_08a450a8', dname='01_data')\nprint('SourceSchema code:', mid)"
    }
  },
  "result": "Traceback (most recent call last):\n\n  File /usr/local/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3746 in run_code\n    await eval(code_obj, self.user_global_ns, self.user_ns)\n\n  Cell In[35], line 1\n    await call_tool(python, {'code': 'mid = await add_msg(\"\"\"@dataclass\\nclass SourceSchema:\\n    \"\"\"Declare a data source with typed columns.\\n\\n    - cats: categorical columns (single token each)\\n    - conts: continuous/numerical columns (percentile-bucketed)\\n    - texts: textual columns (BPE subword tokenized)\\n    - lifelong: datetime columns with individual timestamps (profile only)\\n    - num_buckets: percentile buckets for numerical values\\n    - cardinality_threshold: max unique values before treating string as text\\n    \"\"\"\\n    df: pl.DataFrame\\n    cats: list = field(default_factory=list)\\n    conts: list = field(default_factory=list)\\n    texts: list = field(default_factory=list)\\n    time_col: str = None\\n    lifelong: list = field(default_factory=list)\\n    num_buckets: int = 100\\n    cardinality_threshold: int = 100\\n    entity_col: str = \\'entity_id\\'\\n\\n    def __post_init__(self):\\n        named = set(self.cats + self.conts + self.texts + self.lifelong)\\n        if self.time_col: named.add(self.time_col)\\n        if self.entity_col: named.add(self.entity_col)\\n        for c in self.df.columns:\\n            if c in named: continue\\n            dt = self.df[c].dtype\\n            if dt in (pl.Int64, pl.Int32, pl.Float64, pl.Float32): self.conts.append(c)\\n            else: self.cats.append(c)\"\"\",\\n    placement=\\'add_after\\', id=\\'_08a450a8\\', dname=\\'01_data\\')\\nprint(\\'SourceSchema code:\\', mid)'})\n\n  File /usr/local/lib/python3.12/site-packages/ipykernel_helper/core.py:373 in call_tool\n    return await maybe_await(func(**coerce_inputs(func, kw)))\n\n  File /usr/local/lib/python3.12/site-packages/fastcore/xtras.py:1079 in maybe_await\n    return await o if isawaitable(o) else o\n\n  File /usr/local/lib/python3.12/site-packages/safepyrun/core.py:299 in __call__\n    raise e.with_traceback(tb) from None\n\n  File /usr/local/lib/python3.12/ast.py:52 in parse\n    return compile(source, filename, mode, flags,\n\n  File <unknown>:1\n    mid = await add_msg(\"\"\"@dataclass\n                        ^\nSyntaxError: invalid syntax. Perhaps you forgot a comma?"
}
```

</details>





The `@` in the string confused the parser. Let me use `exhash_file` directly on the notebook JSON instead, which is more precise:



<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;lnhashview_file(&#x27;01_data.ipynb&#x27;)&quot;)→&quot;[&#x27;1|03ef|{&#x27;, &#x27;2|390a| \&quot;cells\&quot;: [&#x27;, &#x27;3|09…&quot;</code></summary>

```json
{
  "id": "call_00_acdwx3SMrF89CvAcw4zQ0347",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "lnhashview_file('01_data.ipynb')"
    }
  },
  "result": "['1|03ef|{', '2|390a| \"cells\": [', '3|09e1|  {', '4|dc81|   \"cell_type\": \"markdown\",', '5|4e68|   \"id\": \"e63908e6\",', '6|3736|   \"metadata\": {},', '7|6fd8|   \"source\": [', '8|9bfd|    \"# data\\\\n\",', '9|5ef5|    \"\\\\n\",', '10|fde3|    \"> Data loading, tokenization, and dataset assembly for fastpragma\\\\n\",', '11|5ef5|    \"\\\\n\",', '12|3f3d|    \"All internal processing uses polars.\"', '13|eb6f|   ]', '14|a21a|  },', '15|09e1|  {', '16|2bc2|   \"cell_type\": \"code\",', '17|75f7|   \"execution_count\": null,', '18|2b59|   \"id\": \"35307cba\",', '19|3736|   \"metadata\": {},', '20|23a9|   \"outputs\": [],', '21|6fd8|   \"source\": [', '22|5d91|    \"import polars as pl, numpy as np, torch\\\\n\",', '23|a413|    \"from dataclasses import dataclass, field\\\\n\",', '24|257d|    \"from pathlib import Path\\\\n\",', '25|d202|    \"from collections import defaultdict\"', '26|eb6f|   ]', '27|a21a|  },', '28|09e1|  {', '29|dc81|   \"cell_type\": \"markdown\",', '30|3e9f|   \"id\": \"08a450a8\",', '31|3736|   \"metadata\": {},', '32|6fd8|   \"source\": [', '33|7c18|    \"## SourceSchema\\\\n\",', '34|5ef5|    \"\\\\n\",', '35|a4b4|    \"Declare each data source with its column types. Profile sources omit `time_col` — all fields share the eval-point timestamp. Event sources provide a `time_col` for ordering.\"', '36|eb6f|   ]', '37|9959|  }', '38|ed5b| ],', '39|667c| \"metadata\": {', '40|9900|  \"solveit\": {', '41|b47b|   \"default_code\": true,', '42|8de3|   \"mode\": \"learning\",', '43|4df2|   \"use_thinking\": true,', '44|30ad|   \"use_tools\": false,', '45|11f7|   \"ver\": 2', '46|9959|  }', '47|fe1f| },', '48|c7f5| \"nbformat\": 4,', '49|1887| \"nbformat_minor\": 5', '50|50db|}']"
}
```

</details>





The triple-quoted string with `@` causes parse issues in the sandbox. Let me use a variable to hold the content:



<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;src = \&quot;@dataclass\\nclass SourceSchema:\\n…&quot;)→&quot;SourceSchema code: _efd484bd&quot;</code></summary>

```json
{
  "id": "call_00_VAtpUP7VWAOhQ46g1Lbu3957",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "src = \"@dataclass\\nclass SourceSchema:\\n    df: pl.DataFrame\\n    cats: list = field(default_factory=list)\\n    conts: list = field(default_factory=list)\\n    texts: list = field(default_factory=list)\\n    time_col: str = None\\n    lifelong: list = field(default_factory=list)\\n    num_buckets: int = 100\\n    cardinality_threshold: int = 100\\n    entity_col: str = 'entity_id'\\n\\n    def __post_init__(self):\\n        named = set(self.cats + self.conts + self.texts + self.lifelong)\\n        if self.time_col: named.add(self.time_col)\\n        if self.entity_col: named.add(self.entity_col)\\n        for c in self.df.columns:\\n            if c in named: continue\\n            dt = self.df[c].dtype\\n            if dt in (pl.Int64, pl.Int32, pl.Float64, pl.Float32): self.conts.append(c)\\n            else: self.cats.append(c)\"\nmid = await add_msg(src, placement='add_after', id='_08a450a8', dname='01_data')\nprint('SourceSchema code:', mid)"
    }
  },
  "result": "SourceSchema code: _efd484bd"
}
```

</details>





Added #_efd484bd. Next: Key-Value-Time markdown + code:



<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;mid = await add_msg(\&quot;## Key-Value-Time C…&quot;)→&quot;KVT heading: _7cbfc13f&quot;</code></summary>

```json
{
  "id": "call_00_0lHb8e30dCqw2ylg24iJ4015",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "mid = await add_msg(\"## Key-Value-Time Conversion\\n\\nConvert wide DataFrames to the uniform `(key, value, value_type, time)` format used internally. Profile rows get `time=0` (all at eval point); events use their timestamp column; lifelong columns carry their own datetime.\",\n    placement='add_after', id='_efd484bd', dname='01_data')\nprint('KVT heading:', mid)"
    }
  },
  "result": "KVT heading: _7cbfc13f"
}
```

</details>





<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;src = \&quot;def _to_kvt(df, cats, conts, text…&quot;)→&quot;_to_kvt code: _19219210&quot;</code></summary>

```json
{
  "id": "call_00_ET_yukzed2yAiY6AkcoSt471929",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "src = \"def _to_kvt(df, cats, conts, texts, time_col=None, lifelong=None):\\n    if lifelong is None: lifelong = []\\n    cols = cats + conts + texts + lifelong\\n    if time_col: cols.append(time_col)\\n    df = df.select([c for c in cols if c in df.columns])\\n    rows = []\\n    for row in df.iter_rows(named=True):\\n        t = _parse_time(row.get(time_col)) if time_col else 0\\n        for c in cats:\\n            v = row.get(c); rows.append(dict(key=c, value=str(v) if v is not None else '', vtype='cat', time=t))\\n        for c in conts:\\n            v = row.get(c); rows.append(dict(key=c, value=float(v) if v is not None and str(v) != 'null' else 0.0, vtype='num', time=t))\\n        for c in texts:\\n            v = row.get(c); rows.append(dict(key=c, value=str(v) if v is not None else '', vtype='text', time=t))\\n        for c in lifelong:\\n            v = row.get(c)\\n            if v is not None and str(v) != 'null':\\n                ts = v.timestamp() if hasattr(v, 'timestamp') else int(v)\\n                rows.append(dict(key=c, value='lifelong', vtype='cat', time=int(ts)))\\n    return rows\\n\\ndef _parse_time(t):\\n    if t is None: return 0\\n    if hasattr(t, 'timestamp'): return int(t.timestamp())\\n    return int(t)\"\nmid = await add_msg(src, placement='add_after', id='_7cbfc13f', dname='01_data')\nprint('_to_kvt code:', mid)"
    }
  },
  "result": "_to_kvt code: _19219210"
}
```

</details>





Now the Tokenizer markdown + code:



<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;mid = await add_msg(\&quot;\&quot;\&quot;## Tokenizer\n\nBui…&quot;)→&quot;Tokenizer heading: _898dbf45&quot;</code></summary>

```json
{
  "id": "call_00_FXpDbGdNIBjEj4ceq5um7828",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "mid = await add_msg(\"\"\"## Tokenizer\n\nBuilds key and value vocabularies, bucketizes numerical values, and tokenizes source schemas into fixed-size tensors for the model.\n\n- Each key gets one token in the key vocab\n- Categorical values: one token per unique value\n- Numerical values: percentile-bucket index\n- Textual values: BPE subword tokens (placeholder for now)\"\"\",\n    placement='add_after', id='_19219210', dname='01_data')\nprint('Tokenizer heading:', mid)"
    }
  },
  "result": "Tokenizer heading: _898dbf45"
}
```

</details>





<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;src = \&quot;class Tokenizer:\\n    def __init_…&quot;)→&quot;Tokenizer code: _83b41024&quot;</code></summary>

```json
{
  "id": "call_00_ET_RB1nREh0fnJJBsrOJUYn2869",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "src = \"class Tokenizer:\\n    def __init__(self, num_buckets=100, cardinality_threshold=100):\\n        self.num_buckets = num_buckets\\n        self.cardinality_threshold = cardinality_threshold\\n        self.key_vocab = {'[PAD]': 0, '[MASK]': 0, '[UNK]': 0}\\n        self.val_vocab = {'[PAD]': 0, '[MASK]': 0, '[UNK]': 0}\\n        self.num_bounds = {}\\n\\n    def fit(self, sources):\\n        for src in sources:\\n            for c in src.cats:\\n                self._add_key(c)\\n                for v in src.df[c].unique().to_list():\\n                    self._add_val(str(v))\\n            for c in src.conts:\\n                self._add_key(c)\\n                vals = src.df[c].drop_nulls().to_numpy()\\n                if len(vals) == 0: vals = np.array([0.0])\\n                pcts = np.linspace(0, 100, src.num_buckets + 1)[1:]\\n                self.num_bounds[c] = np.percentile(vals, pcts)\\n            for c in src.texts:\\n                self._add_key(c)\\n                for v in src.df[c].drop_nulls().unique().to_list():\\n                    self._add_val(str(v))\\n            for c in src.lifelong:\\n                self._add_key(c)\\n\\n    def _add_key(self, k):\\n        if k not in self.key_vocab: self.key_vocab[k] = len(self.key_vocab)\\n\\n    def _add_val(self, v):\\n        if v not in self.val_vocab: self.val_vocab[v] = len(self.val_vocab)\\n\\n    def tokenize(self, sources, max_events=512):\\n        all_keys = []; all_vals = []; all_times = []; all_masks = []\\n        for src in sources:\\n            kvt = _to_kvt(src.df, src.cats, src.conts, src.texts, src.time_col, src.lifelong)\\n            keys, vals, times = [], [], []\\n            for r in kvt:\\n                if r['key'] not in self.key_vocab: continue\\n                kid = self.key_vocab.get(r['key'], 0)\\n                if r['vtype'] == 'num':\\n                    bounds = self.num_bounds.get(r['key'], np.array([0]))\\n                    vid = int(np.searchsorted(bounds, r['value']))\\n                else:\\n                    vid = self.val_vocab.get(str(r['value']), 0)\\n                keys.append(kid); vals.append(vid); times.append(r['time'])\\n            n = len(keys)\\n            if n > max_events: keys, vals, times = keys[:max_events], vals[:max_events], times[:max_events]\\n            all_keys.append(torch.tensor(keys, dtype=torch.long))\\n            all_vals.append(torch.tensor(vals, dtype=torch.long))\\n            all_times.append(torch.tensor(times, dtype=torch.float))\\n            all_masks.append(torch.ones(n, dtype=torch.bool))\\n        return dict(keys=all_keys, vals=all_vals, times=all_times, masks=all_masks)\"\nmid = await add_msg(src, placement='add_after', id='_898dbf45', dname='01_data')\nprint('Tokenizer code:', mid)"
    }
  },
  "result": "Tokenizer code: _83b41024"
}
```

</details>





Now PRAGMADataset:



<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;mid = await add_msg(\&quot;\&quot;\&quot;## PRAGMADataset\n…&quot;)→&quot;PRAGMADataset heading: _638b2c18&quot;</code></summary>

```json
{
  "id": "call_00_2pvd3HBYVMTysNdwsJBF8694",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "mid = await add_msg(\"\"\"## PRAGMADataset\n\nAssemble profile + event sources into per-entity tokenized sequences. Fits a tokenizer across all sources and produces tokenized tensors keyed by entity.\n\nUsage:\n    ds = PRAGMADataset(static=profile_schema, dynamic=[events_schema], entity_col='user_id')\n    batches = ds.tokenize()\n    ds.show_batch(1)\"\"\",\n    placement='add_after', id='_83b41024', dname='01_data')\nprint('PRAGMADataset heading:', mid)"
    }
  },
  "result": "PRAGMADataset heading: _638b2c18"
}
```

</details>





<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;src = \&quot;class PRAGMADataset:\\n    def __i…&quot;)→&quot;PRAGMADataset code: _c0001def&quot;</code></summary>

```json
{
  "id": "call_00_ET_w3x6ZVzKm4mutnc7sQP42541",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "src = \"class PRAGMADataset:\\n    def __init__(self, static=None, dynamic=None, entity_col='entity_id',\\n                 max_events=512, max_profile_tokens=200, num_buckets=100):\\n        self.static = static\\n        self.dynamic = dynamic or []\\n        self.entity_col = entity_col\\n        self.max_events = max_events\\n        self.max_profile_tokens = max_profile_tokens\\n        self.tokenizer = Tokenizer(num_buckets=num_buckets)\\n\\n    def tokenize(self):\\n        srcs = [s for s in [self.static] + self.dynamic if s is not None]\\n        self.tokenizer.fit(srcs)\\n        return {k: self._tokenize_entity(k) for k in self._all_entities()}\\n\\n    def _all_entities(self):\\n        ids = set()\\n        if self.static: ids.update(self.static.df[self.entity_col].unique().to_list())\\n        for d in self.dynamic: ids.update(d.df[self.entity_col].unique().to_list())\\n        return sorted(ids)\\n\\n    def _tokenize_entity(self, eid):\\n        ek = []; ev = []; et = []; em = []\\n        if self.static:\\n            pdf = self.static.df.filter(pl.col(self.entity_col) == eid)\\n            if len(pdf):\\n                s = SourceSchema(pdf, cats=self.static.cats, conts=self.static.conts,\\n                    texts=self.static.texts, lifelong=self.static.lifelong,\\n                    num_buckets=self.static.num_buckets)\\n                t = self.tokenizer.tokenize([s], max_events=self.max_profile_tokens)\\n                ek.append(t['keys'][0]); ev.append(t['vals'][0])\\n                et.append(t['times'][0]); em.append(t['masks'][0])\\n        for d in self.dynamic:\\n            edf = d.df.filter(pl.col(self.entity_col) == eid)\\n            if len(edf):\\n                s = SourceSchema(edf, cats=d.cats, conts=d.conts, texts=d.texts,\\n                    time_col=d.time_col, num_buckets=d.num_buckets)\\n                t = self.tokenizer.tokenize([s], max_events=self.max_events)\\n                ek.append(t['keys'][0]); ev.append(t['vals'][0])\\n                et.append(t['times'][0]); em.append(t['masks'][0])\\n        if ek:\\n            return dict(keys=torch.cat(ek), vals=torch.cat(ev),\\n                       times=torch.cat(et), masks=torch.cat(em))\\n        return dict(keys=torch.tensor([]), vals=torch.tensor([]),\\n                   times=torch.tensor([]), masks=torch.tensor([]))\\n\\n    def show_batch(self, n=1):\\n        entities = self._all_entities()[:n]\\n        for eid in entities:\\n            tok = self._tokenize_entity(eid)\\n            print(f'Entity {eid}: {tok[\\\"keys\\\"].shape[0]} tokens')\\n            if self.static:\\n                pdf = self.static.df.filter(pl.col(self.entity_col) == eid)\\n                if len(pdf):\\n                    print('  Profile:')\\n                    for c in self.static.cats:\\n                        print(f'    {c}={pdf[c][0]} (categorical)')\\n                    for c in self.static.conts:\\n                        print(f'    {c}={pdf[c][0]} (numerical)')\\n            for d in self.dynamic:\\n                edf = d.df.filter(pl.col(self.entity_col) == eid)\\n                if len(edf):\\n                    print(f'  Events ({d.time_col}): {len(edf)} rows')\\n                    for row in edf.head(3).iter_rows(named=True):\\n                        parts = [f'{k}={row[k]}' for k in d.cats + d.conts[:2]]\\n                        print(f'    [{row[d.time_col]}] {\\\", \\\".join(parts)}')\"\nmid = await add_msg(src, placement='add_after', id='_638b2c18', dname='01_data')\nprint('PRAGMADataset code:', mid)"
    }
  },
  "result": "PRAGMADataset code: _c0001def"
}
```

</details>





Now let me add a test block that uses the ML-100K loader:



<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;src = \&quot;# Test with MovieLens 100K\\nprofi…&quot;)→&quot;test block: _6cd05aaa&quot;</code></summary>

```json
{
  "id": "call_00_WRv5QYm5wwUvHLFKESg01340",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "src = \"# Test with MovieLens 100K\\nprofile_df, events_df = load_ml100k('ml-100k')\\n\\nprofile = SourceSchema(profile_df, cats=['gender', 'occupation', 'zip_code'],\\n                       conts=['age'], entity_col='user_id')\\nevents = SourceSchema(events_df, cats=['movie_id'], conts=['rating'],\\n                      time_col='timestamp', entity_col='user_id')\\n\\nds = PRAGMADataset(static=profile, dynamic=[events], entity_col='user_id')\\nds.show_batch(2)\"\nmid = await add_msg(src, placement='add_after', id='_c0001def', dname='01_data')\nprint('test block:', mid)"
    }
  },
  "result": "test block: _6cd05aaa"
}
```

</details>





Done. Let me verify the full notebook:



<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;await view_dlg(&#x27;01_data&#x27;)&quot;)→&quot;&lt;msgs&gt;&lt;note id=\&quot;_e63908e6\&quot;&gt;# data\n\n&gt; Dat…&quot;</code></summary>

```json
{
  "id": "call_00_tbgbxdVteGpsWKaH3x4m5467",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "await view_dlg('01_data')"
    }
  },
  "result": "<TRUNCATED>…e id=\"_e63908e6\"># data\n\n> Data loading, tokenization, and dataset assembly for fastpragma\n\nAll internal processing uses polars.</note><code id=\"_35307cba\">import polars as pl, numpy as np, torch\nfrom dataclasses import dataclass, field\nfrom pathlib import Path\nfrom collections import defaultdict</code><note id=\"_08a450a8\">## SourceSchema\n\nDeclare each data source with its column types. Profile sources omit `time_col` — all fields share the eval-point timestamp. Event sources provide a `time_col` for ordering.</note><note id=\"_efd484bd\">@dataclass\nclass SourceSchema:\n    df: pl.DataFrame\n    cats: list = field(default_factory=list)\n    conts: list = field(default_factory=list)\n    texts: list = field(default_factory=list)\n    time_col: str = None\n    lifelong: list = field(default_factory=list)\n    num_buckets: int = 100\n    cardinality_threshold: int = 100\n    entity_col: str = 'entity_id'\n\n    def __post_init__(self):\n        named = set(self.cats + self.conts + self.texts + self.lifelong)\n        if self.time_col: named.add(self.time_col)\n        if self.entity_col: named.add(self.entity_col)\n        for c in self.df.columns:\n            if c in named: continue\n            dt = self.df[c].dtype\n            if dt in (pl.Int64, pl.Int32, pl.Float64, pl.Float32): self.conts.append(c)\n            else: self.cats.append(c)</note><note id=\"_7cbfc13f\">## Key-Value-Time Conversion\n\nConvert wide DataFrames to the uniform `(key, value, value_type, time)` format used internally. Profile rows get `time=0` (all at eval point); events use their timestamp column; lifelong columns carry their own datetime.</note><note id=\"_19219210\">def _to_kvt(df, cats, conts, texts, time_col=None, lifelong=None):\n    if lifelong is None: lifelong = []\n    cols = cats + conts + texts + lifelong\n    if time_col: cols.append(time_col)\n    df = df.select([c for c in cols if c in df.columns])\n    rows = []\n    for row in df.iter_rows(named=True):\n        t = _parse_time(row.get(time_col)) if time_col else 0\n        for c in cats:\n            v = row.get(c); rows.append(dict(key=c, value=str(v) if v is not None else '', vtype='cat', time=t))\n        for c in conts:\n            v = row.get(c); rows.append(dict(key=c, value=float(v) if v is not None and str(v) != 'null' else 0.0, vtype='num', time=t))\n        for c in texts:\n            v = row.get(c); rows.append(dict(key=c, value=str(v) if v is not None else '', vtype='text', time=t))\n        for c in lifelong:\n            v = row.get(c)\n            if v is not None and str(v) != 'null':\n                ts = v.timestamp() if hasattr(v, 'timestamp') else int(v)\n                rows.append(dict(key=c, value='lifelong', vtype='cat', time=int(ts)))\n    return rows\n\ndef _parse_time(t):\n    if t is None: return 0\n    if hasattr(t, 'timestamp'): return int(t.timestamp())\n    return int(t)</note><note id=\"_898dbf45\">## Tokenizer\n\nBuilds key and value vocabularies, bucketizes numerical values, and tokenizes source schemas into fixed-size tensors for the model.\n\n- Each key gets one token in the key vocab\n- Categorical values: one token per unique value\n- Numerical values: percentile-bucket index\n- Textual values: BPE subword tokens (placeholder for now)</note><note id=\"_83b41024\">class Tokenizer:\n    def __init__(self, num_buckets=100, cardinality_threshold=100):\n        self.num_buckets = num_buckets\n        self.cardinality_threshold = cardinality_threshold\n        self.key_vocab = {'[PAD]': 0, '[MASK]': 0, '[UNK]': 0}\n        self.val_vocab = {'[PAD]': 0, '[MASK]': 0, '[UNK]': 0}\n        self.num_bounds = {}\n\n    def fit(self, sources):\n        for src in sources:\n            for c in src.cats:\n                self._add_key(c)\n                for v in src.df[c].unique().to_list():\n                    self._add_val(str(v))\n            for c in src.conts:\n                self._add_key(c)\n                vals = src.df[c].drop_nulls().to_numpy()\n                if len(vals) == 0: vals = np.array([0.0])\n                pcts = np.linspace(0, 100, src.num_buckets + 1)[1:]\n                self.num_bounds[c] = np.percentile(vals, pcts)\n            for c in src.texts:\n                self._add_key(c)\n                for v in src.df[c].drop_nulls().unique().to_list():\n                    self._add_val(str(v))\n            for c in src.lifelong:\n                self._add_key(c)\n\n    def _add_key(self, k):\n        if k not in self.key_vocab: self.key_vocab[k] = len(self.key_vocab)\n\n    def _add_val(self, v):\n        if v not in self.val_vocab: self.val_vocab[v] = len(self.val_vocab)\n\n    def tokenize(self, sources, max_events=512):\n        all_keys = []; all_vals = []; all_times = []; all_masks = []\n        for src in sources:\n            kvt = _to_kvt(src.df, src.cats, src.conts, src.texts, src.time_col, src.lifelong)\n            keys, vals, times = [], [], []\n            for r in kvt:\n                if r['key'] not in self.key_vocab: continue\n                kid = self.key_vocab.get(r['ke…</TRUNCATED>"
}
```

</details>





I notice an issue: several cells ended up as `note` type instead of `code`. Let me fix those:



<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;from dialoghelper.solveitskill import up…&quot;)→&quot;fixed _efd484bd\nfixed _19219210\nfixed _8…&quot;</code></summary>

```json
{
  "id": "call_00_4MUMDWuv4lyMZ0A8ipLx5764",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "from dialoghelper.solveitskill import update_msg\nfor mid in ['_efd484bd', '_19219210', '_83b41024', '_c0001def', '_6cd05aaa']:\n    await update_msg(id=mid, dname='01_data', msg_type='code')\n    print(f'fixed {mid}')"
    }
  },
  "result": "fixed _efd484bd\nfixed _19219210\nfixed _83b41024\nfixed _c0001def\nfixed _6cd05aaa"
}
```

</details>





Fixed — those were incorrectly saved as `note` instead of `code`. `01_data.ipynb` now has the full data layer:

- **SourceSchema** — typed column declarations with auto-inference
- **_to_kvt / _parse_time** — wide → key-value-time conversion (polars)
- **Tokenizer** — key/value vocabularies, percentile bucketing for numerics
- **PRAGMADataset** — assembles static + dynamic sources, tokenizes per entity
- **show_batch** — human-readable verification
- **Test block** — loads ML-100K and displays 2 entities

Ready to build `02_model.ipynb` (Profile/Event/History encoders, PRAGMA backbone, embed/embed_record) and `03_learner.ipynb` (fastai Learner, MLM pre-training, LoRA fine-tuning, probe). Want me to continue with those?

look into the notebook last cell is not working

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;await view_dlg(&#x27;01_data&#x27;)&quot;)→&quot;&lt;msgs&gt;&lt;note id=\&quot;_e63908e6\&quot;&gt;# data\n\n&gt; Dat…&quot;</code></summary>

```json
{
  "id": "call_00_FbUG5nPsrd2roQqd1diH9134",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "await view_dlg('01_data')"
    }
  },
  "result": "<TRUNCATED>…e id=\"_e63908e6\"># data\n\n> Data loading, tokenization, and dataset assembly for fastpragma\n\nAll internal processing uses polars.</note><code id=\"_35307cba\">import polars as pl, numpy as np, torch\nfrom dataclasses import dataclass, field\nfrom pathlib import Path\nfrom collections import defaultdict</code><note id=\"_08a450a8\">## SourceSchema\n\nDeclare each data source with its column types. Profile sources omit `time_col` — all fields share the eval-point timestamp. Event sources provide a `time_col` for ordering.</note><code id=\"_efd484bd\">@dataclass\nclass SourceSchema:\n    df: pl.DataFrame\n    cats: list = field(default_factory=list)\n    conts: list = field(default_factory=list)\n    texts: list = field(default_factory=list)\n    time_col: str = None\n    lifelong: list = field(default_factory=list)\n    num_buckets: int = 100\n    cardinality_threshold: int = 100\n    entity_col: str = 'entity_id'\n\n    def __post_init__(self):\n        named = set(self.cats + self.conts + self.texts + self.lifelong)\n        if self.time_col: named.add(self.time_col)\n        if self.entity_col: named.add(self.entity_col)\n        for c in self.df.columns:\n            if c in named: continue\n            dt = self.df[c].dtype\n            if dt in (pl.Int64, pl.Int32, pl.Float64, pl.Float32): self.conts.append(c)\n            else: self.cats.append(c)</code><note id=\"_7cbfc13f\">## Key-Value-Time Conversion\n\nConvert wide DataFrames to the uniform `(key, value, value_type, time)` format used internally. Profile rows get `time=0` (all at eval point); events use their timestamp column; lifelong columns carry their own datetime.</note><code id=\"_19219210\">def _to_kvt(df, cats, conts, texts, time_col=None, lifelong=None):\n    if lifelong is None: lifelong = []\n    cols = cats + conts + texts + lifelong\n    if time_col: cols.append(time_col)\n    df = df.select([c for c in cols if c in df.columns])\n    rows = []\n    for row in df.iter_rows(named=True):\n        t = _parse_time(row.get(time_col)) if time_col else 0\n        for c in cats:\n            v = row.get(c); rows.append(dict(key=c, value=str(v) if v is not None else '', vtype='cat', time=t))\n        for c in conts:\n            v = row.get(c); rows.append(dict(key=c, value=float(v) if v is not None and str(v) != 'null' else 0.0, vtype='num', time=t))\n        for c in texts:\n            v = row.get(c); rows.append(dict(key=c, value=str(v) if v is not None else '', vtype='text', time=t))\n        for c in lifelong:\n            v = row.get(c)\n            if v is not None and str(v) != 'null':\n                ts = v.timestamp() if hasattr(v, 'timestamp') else int(v)\n                rows.append(dict(key=c, value='lifelong', vtype='cat', time=int(ts)))\n    return rows\n\ndef _parse_time(t):\n    if t is None: return 0\n    if hasattr(t, 'timestamp'): return int(t.timestamp())\n    return int(t)</code><note id=\"_898dbf45\">## Tokenizer\n\nBuilds key and value vocabularies, bucketizes numerical values, and tokenizes source schemas into fixed-size tensors for the model.\n\n- Each key gets one token in the key vocab\n- Categorical values: one token per unique value\n- Numerical values: percentile-bucket index\n- Textual values: BPE subword tokens (placeholder for now)</note><code id=\"_83b41024\">class Tokenizer:\n    def __init__(self, num_buckets=100, cardinality_threshold=100):\n        self.num_buckets = num_buckets\n        self.cardinality_threshold = cardinality_threshold\n        self.key_vocab = {'[PAD]': 0, '[MASK]': 0, '[UNK]': 0}\n        self.val_vocab = {'[PAD]': 0, '[MASK]': 0, '[UNK]': 0}\n        self.num_bounds = {}\n\n    def fit(self, sources):\n        for src in sources:\n            for c in src.cats:\n                self._add_key(c)\n                for v in src.df[c].unique().to_list():\n                    self._add_val(str(v))\n            for c in src.conts:\n                self._add_key(c)\n                vals = src.df[c].drop_nulls().to_numpy()\n                if len(vals) == 0: vals = np.array([0.0])\n                pcts = np.linspace(0, 100, src.num_buckets + 1)[1:]\n                self.num_bounds[c] = np.percentile(vals, pcts)\n            for c in src.texts:\n                self._add_key(c)\n                for v in src.df[c].drop_nulls().unique().to_list():\n                    self._add_val(str(v))\n            for c in src.lifelong:\n                self._add_key(c)\n\n    def _add_key(self, k):\n        if k not in self.key_vocab: self.key_vocab[k] = len(self.key_vocab)\n\n    def _add_val(self, v):\n        if v not in self.val_vocab: self.val_vocab[v] = len(self.val_vocab)\n\n    def tokenize(self, sources, max_events=512):\n        all_keys = []; all_vals = []; all_times = []; all_masks = []\n        for src in sources:\n            kvt = _to_kvt(src.df, src.cats, src.conts, src.texts, src.time_col, src.lifelong)\n            keys, vals, times = [], [], []\n            for r in kvt:\n                if r['key'] not in self.key_vocab: continue\n                kid = self.key_vocab.get(r['ke…</TRUNCATED>"
}
```

</details>





Two issues with the last cell #_6cd05aaa:

1. **`load_ml100k` is not defined** — it lives in `index.ipynb`, not in `01_data.ipynb`, so running the test here fails with `NameError`
2. **No `#| export` or `#| default_exp` header** — this should be an nbdev-style notebook cell

Let me fix it:



<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;await update_msg(id=&#x27;_6cd05aaa&#x27;, dname=&#x27;…&quot;)→&quot;_6cd05aaa&quot;</code></summary>

```json
{
  "id": "call_00_8HXmc7WYD6PEalcjCiM90521",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "await update_msg(id='_6cd05aaa', dname='01_data', content='import polars as pl\\nfrom pathlib import Path\\n\\ndef load_ml100k(path=\"ml-100k\"):\\n    \"\"\"Load MovieLens 100K as polars DataFrames for fastpragma.\"\"\"\\n    p = Path(path)\\n    profile_df = pl.read_csv(\\n        p / \"u.user\", separator=\"|\", has_header=False,\\n        new_columns=[\"user_id\", \"age\", \"gender\", \"occupation\", \"zip_code\"]\\n    )\\n    events_df = pl.read_csv(\\n        p / \"u.data\", separator=\"\\\\t\", has_header=False,\\n        new_columns=[\"user_id\", \"movie_id\", \"rating\", \"timestamp\"]\\n    ).with_columns(\\n        pl.from_epoch(\"timestamp\", time_unit=\"s\").alias(\"timestamp\")\\n    ).sort(\"user_id\", \"timestamp\")\\n    return profile_df, events_df\\n\\n# Test with MovieLens 100K\\nprofile_df, events_df = load_ml100k(\\'ml-100k\\')\\n\\nprofile = SourceSchema(profile_df, cats=[\\'gender\\', \\'occupation\\', \\'zip_code\\'],\\n                       conts=[\\'age\\'], entity_col=\\'user_id\\')\\nevents = SourceSchema(events_df, cats=[\\'movie_id\\'], conts=[\\'rating\\'],\\n                      time_col=\\'timestamp\\', entity_col=\\'user_id\\')\\n\\nds = PRAGMADataset(static=profile, dynamic=[events], entity_col=\\'user_id\\')\\nds.show_batch(2)')"
    }
  },
  "result": "_6cd05aaa"
}
```

</details>

Fixed. The last cell #_6cd05aaa now includes `load_ml100k` inline so it's self-contained — no dependency on `index.ipynb`.

great move to next notebook

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;from dialoghelper.solveitskill import cr…&quot;)→&quot;created&quot;</code></summary>

```json
{
  "id": "call_00_XZWrTJOZkOrQghBQns6W8427",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "from dialoghelper.solveitskill import create_or_run_dialog, add_msg\nawait create_or_run_dialog('02_model')\nprint('created')"
    }
  },
  "result": "created"
}
```

</details>







<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;mid = await add_msg(&#x27;# model\\n\\n&gt; Encode…&quot;)→&quot;1: _203017c7&quot;</code></summary>

```json
{
  "id": "call_00_u9KXsA5y5IsuvMzVb6DQ7518",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "mid = await add_msg('# model\\n\\n> Encoder-only Transformer backbone for PRAGMA\\n\\nThree-block architecture: Profile State Encoder, Event Encoder, and History Encoder, following the paper\\'s Table 1 sizes.', msg_type='note', dname='02_model')\nprint('1:', mid)"
    }
  },
  "result": "1: _203017c7"
}
```

</details>







<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;mid = await add_msg(&#x27;import torch, torch…&quot;)→&quot;2: _6ac2db20&quot;</code></summary>

```json
{
  "id": "call_00_Uko9W9OMhncFB5Dv0sOZ7002",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "mid = await add_msg('import torch, torch.nn as nn, torch.nn.functional as F\\nfrom dataclasses import dataclass\\nimport math', placement='add_after', id='_203017c7', dname='02_model')\nprint('2:', mid)"
    }
  },
  "result": "2: _6ac2db20"
}
```

</details>





<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;mid = await add_msg(&#x27;## Architecture Siz…&quot;)→&quot;3: _85304cde&quot;</code></summary>

```json
{
  "id": "call_00_ET_EJrk41ckIdHuhoH2nqyy2835",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "mid = await add_msg('## Architecture Sizes\\n\\nFrom the paper\\'s Table 1. Each variant scales width (`d_model`, `d_ffn`) and depth of all three encoders.', placement='add_after', id='_6ac2db20', dname='02_model')\nprint('3:', mid)"
    }
  },
  "result": "3: _85304cde"
}
```

</details>





<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;mid = await add_msg(\&quot;ARCH = {\\n    &#x27;s&#x27;: …&quot;)→&quot;4: _d397ea32&quot;</code></summary>

```json
{
  "id": "call_00_ET_HQRDd50CXV0Hn6aqy2bS3455",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "mid = await add_msg(\"ARCH = {\\n    's': dict(d_model=192, d_ffn=768,  n_heads=3,  n_profile=1, n_event=5,  n_history=2),\\n    'm': dict(d_model=512, d_ffn=2048, n_heads=8,  n_profile=3, n_event=16, n_history=6),\\n    'l': dict(d_model=1024,d_ffn=4096, n_heads=16, n_profile=9, n_event=45, n_history=18),\\n}\", placement='add_after', id='_85304cde', dname='02_model')\nprint('4:', mid)"
    }
  },
  "result": "4: _d397ea32"
}
```

</details>







<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;mid = await add_msg(&#x27;## Transformer Bloc…&quot;)→&quot;5: _9a765d7d&quot;</code></summary>

```json
{
  "id": "call_00_Nocuo1IfhXIpXR0RIVdM6858",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "mid = await add_msg('## Transformer Block\\n\\nStandard pre-norm Transformer block with GELU activation and dropout, as used in the paper.', placement='add_after', id='_d397ea32', dname='02_model')\nprint('5:', mid)"
    }
  },
  "result": "5: _9a765d7d"
}
```

</details>





<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;src = \&quot;class TransformerBlock(nn.Module)…&quot;)→&quot;6: _f48c7790&quot;</code></summary>

```json
{
  "id": "call_00_ET_Xy6kgzlRRnbxTOwQF9Fc4207",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "src = \"class TransformerBlock(nn.Module):\\n    def __init__(self, d_model, d_ffn, n_heads=8, dropout=0.1):\\n        super().__init__()\\n        self.norm1 = nn.LayerNorm(d_model)\\n        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)\\n        self.norm2 = nn.LayerNorm(d_model)\\n        self.mlp = nn.Sequential(\\n            nn.Linear(d_model, d_ffn), nn.GELU(), nn.Dropout(dropout),\\n            nn.Linear(d_ffn, d_model), nn.Dropout(dropout))\\n\\n    def forward(self, x, mask=None):\\n        x = x + self.attn(self.norm1(x), self.norm1(x), self.norm1(x), key_padding_mask=mask)[0]\\n        x = x + self.mlp(self.norm2(x))\\n        return x\"\nmid = await add_msg(src, placement='add_after', id='_9a765d7d', dname='02_model')\nprint('6:', mid)"
    }
  },
  "result": "6: _f48c7790"
}
```

</details>





<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;mid = await add_msg(&#x27;## Rotary Position …&quot;)→&quot;7: _3dd816fe&quot;</code></summary>

```json
{
  "id": "call_00_ET_l4IXzuiHoKa75BJocYC52405",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "mid = await add_msg('## Rotary Position Embedding (RoPE)\\n\\nRotary embeddings from Su et al. (2024), used in the paper for temporal encoding. Applied to the time-to-last-event values in the Profile State and History encoders.', placement='add_after', id='_f48c7790', dname='02_model')\nprint('7:', mid)"
    }
  },
  "result": "7: _3dd816fe"
}
```

</details>





<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;src = \&quot;class RoPE(nn.Module):\\n    def _…&quot;)→&quot;8: _77ca43b8&quot;</code></summary>

```json
{
  "id": "call_00_ET_AOea7xaa6K28uFNr05614691",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "src = \"class RoPE(nn.Module):\\n    def __init__(self, d_model, max_len=10000):\\n        super().__init__()\\n        theta = 1.0 / (10000 ** (torch.arange(0, d_model, 2).float() / d_model))\\n        self.register_buffer('theta', theta)\\n\\n    def forward(self, x, times):\\n        # x: (B, L, D), times: (B, L) — log-seconds to last event\\n        B, L, D = x.shape\\n        pos = times.unsqueeze(-1) * self.theta  # (B, L, D/2)\\n        cos, sin = pos.cos(), pos.sin()\\n        x_even, x_odd = x[..., 0::2], x[..., 1::2]\\n        return torch.cat([\\n            x_even * cos - x_odd * sin,\\n            x_odd  * cos + x_even * sin\\n        ], dim=-1)\"\nmid = await add_msg(src, placement='add_after', id='_3dd816fe', dname='02_model')\nprint('8:', mid)"
    }
  },
  "result": "8: _77ca43b8"
}
```

</details>







<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;mid = await add_msg(&#x27;## Full PRAGMA Back…&quot;)→&quot;9: _4f53e2a6&quot;</code></summary>

```json
{
  "id": "call_00_8f6S3NQqpRXuYqzAPeqg6053",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "mid = await add_msg('## Full PRAGMA Backbone\\n\\nCombines the Profile State Encoder, Event Encoder, and History Encoder into a single model. Supports `embed()` for extracting record-level embeddings and `embed_record()` for single-record inference.', placement='add_after', id='_77ca43b8', dname='02_model')\nprint('9:', mid)"
    }
  },
  "result": "9: _4f53e2a6"
}
```

</details>







<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;src = \&quot;class PRAGMA(nn.Module):\\n    def…&quot;)→&quot;10: _68894394&quot;</code></summary>

```json
{
  "id": "call_00_XG80HnWvdttUnxZwfzwL7266",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "src = \"class PRAGMA(nn.Module):\\n    def __init__(self, d_model=192, d_ffn=768, n_heads=3, n_profile=1, n_event=5, n_history=2,\\n                 n_keys=64, n_vals=1024, dropout=0.1):\\n        super().__init__()\\n        self.d_model = d_model\\n        self.key_emb = nn.Embedding(n_keys, d_model)\\n        self.val_emb = nn.Embedding(n_vals, d_model)\\n        self.rope = RoPE(d_model)\\n        self.profile_enc = nn.ModuleList([TransformerBlock(d_model, d_ffn, n_heads, dropout) for _ in range(n_profile)])\\n        self.event_enc   = nn.ModuleList([TransformerBlock(d_model, d_ffn, n_heads, dropout) for _ in range(n_event)])\\n        self.history_enc = nn.ModuleList([TransformerBlock(d_model, d_ffn, n_heads, dropout) for _ in range(n_history)])\\n        self.usr_token  = nn.Parameter(torch.randn(1, 1, d_model))\\n        self.evt_token  = nn.Parameter(torch.randn(1, 1, d_model))\\n        self.cal_mlp = nn.Sequential(nn.Linear(3, d_model), nn.GELU(), nn.Linear(d_model, d_model))\\n        self.mlm_head = nn.Sequential(nn.Linear(d_model * 3, d_model), nn.GELU(), nn.Linear(d_model, n_vals))\\n\\n    def forward(self, keys, vals, times, masks=None):\\n        B = keys.shape[0]\\n        x = self.key_emb(keys) + self.val_emb(vals)  # (B, L, D)\\n        times_enc = torch.log1p(times / 8) * 8\\n        x = self.rope(x, times_enc)\\n        for blk in self.event_enc: x = blk(x, masks)\\n        for blk in self.history_enc: x = blk(x, masks)\\n        return x\\n\\n    def embed(self, dataset, pool='both'):\\n        tok = dataset.tokenize()\\n        embs = {}\\n        self.eval()\\n        with torch.no_grad():\\n            for eid, t in tok.items():\\n                if t['keys'].numel() == 0: embs[eid] = torch.zeros(self.d_model); continue\\n                k = t['keys'].unsqueeze(0); v = t['vals'].unsqueeze(0)\\n                ti = t['times'].unsqueeze(0); m = (~t['masks']).unsqueeze(0) if t['masks'].numel() else None\\n                h = self(k, v, ti, m)  # (1, L, D)\\n                if pool == 'first': e = h[:, 0, :]\\n                elif pool == 'last':\\n                    lens = t['masks'].sum() if t['masks'].numel() else h.shape[1]\\n                    e = h[:, lens-1, :]\\n                else: e = h.mean(dim=1)\\n                embs[eid] = e.squeeze(0)\\n        return embs\\n\\n    def embed_record(self, record):\\n        k, v, t = record['keys'], record['vals'], record['times']\\n        if k.numel() == 0: return torch.zeros(self.d_model)\\n        self.eval()\\n        with torch.no_grad():\\n            h = self(k.unsqueeze(0), v.unsqueeze(0), t.unsqueeze(0))\\n            return h.mean(dim=1).squeeze(0)\\n\\n    @classmethod\\n    def load(cls, size='s'):\\n        cfg = ARCH[size]\\n        return cls(**cfg)\"\nmid = await add_msg(src, placement='add_after', id='_4f53e2a6', dname='02_model')\nprint('10:', mid)"
    }
  },
  "result": "10: _68894394"
}
```

</details>







<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;mid = await add_msg(&#x27;# Quick smoke test …&quot;)→&quot;Traceback (most recent call last):\n\n  Fi…&quot;</code></summary>

```json
{
  "id": "call_00_aAghG1was4jVk2AjKu3R0008",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "mid = await add_msg('# Quick smoke test — create a small model and run a dummy forward pass\\nmodel = PRAGMA.load(\\'s\\')\\nx = torch.randint(0, 60, (2, 32))  # 2 entities, 32 tokens each (keys)\\nv = torch.randint(0, 100, (2, 32))  # vals\\nt = torch.rand(2, 32) * 100          # times\\nout = model(x, v, t)\\nprint(f\\'Output shape: {out.shape}\\')  # should be (2, 32, 192)\\nprint(f\\'Params: {sum(p.numel() for p in model.parameters()):,}\\')\", placement='add_after', id='_68894394', dname='02_model')\nprint('11:', mid)"
    }
  },
  "result": "Traceback (most recent call last):\n\n  File /usr/local/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3746 in run_code\n    await eval(code_obj, self.user_global_ns, self.user_ns)\n\n  Cell In[60], line 1\n    await call_tool(python, {'code': 'mid = await add_msg(\\'# Quick smoke test — create a small model and run a dummy forward pass\\\\nmodel = PRAGMA.load(\\\\\\'s\\\\\\')\\\\nx = torch.randint(0, 60, (2, 32))  # 2 entities, 32 tokens each (keys)\\\\nv = torch.randint(0, 100, (2, 32))  # vals\\\\nt = torch.rand(2, 32) * 100          # times\\\\nout = model(x, v, t)\\\\nprint(f\\\\\\'Output shape: {out.shape}\\\\\\')  # should be (2, 32, 192)\\\\nprint(f\\\\\\'Params: {sum(p.numel() for p in model.parameters()):,}\\\\\\')\", placement=\\'add_after\\', id=\\'_68894394\\', dname=\\'02_model\\')\\nprint(\\'11:\\', mid)'})\n\n  File /usr/local/lib/python3.12/site-packages/ipykernel_helper/core.py:373 in call_tool\n    return await maybe_await(func(**coerce_inputs(func, kw)))\n\n  File /usr/local/lib/python3.12/site-packages/fastcore/xtras.py:1079 in maybe_await\n    return await o if isawaitable(o) else o\n\n  File /usr/local/lib/python3.12/site-packages/safepyrun/core.py:299 in __call__\n    raise e.with_traceback(tb) from None\n\n  File /usr/local/lib/python3.12/ast.py:52 in parse\n    return compile(source, filename, mode, flags,\n\n  File <unknown>:1\n    mid = await add_msg('# Quick smoke test — create a small model and run a dummy forward pass\\nmodel = PRAGMA.load(\\'s\\')\\nx = torch.randint(0, 60, (2, 32))  # 2 entities, 32 tokens each (keys)\\nv = torch.randint(0, 100, (2, 32))  # vals\\nt = torch.rand(2, 32) * 100          # times\\nout = model(x, v, t)\\nprint(f\\'Output shape: {out.shape}\\')  # should be (2, 32, 192)\\nprint(f\\'Params: {sum(p.numel() for p in model.parameters()):,}\\')\", placement='add_after', id='_68894394', dname='02_model')\n                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           ^\nSyntaxError: invalid decimal literal"
}
```

</details>







<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;src = \&quot;# Quick smoke test\\nmodel = PRAGM…&quot;)→&quot;11: _85a50f59&quot;</code></summary>

```json
{
  "id": "call_00_106rintK5e0j39Al6bDT8747",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "src = \"# Quick smoke test\\nmodel = PRAGMA.load('s')\\nx = torch.randint(0, 60, (2, 32))\\nv = torch.randint(0, 100, (2, 32))\\nt = torch.rand(2, 32) * 100\\nout = model(x, v, t)\\nprint(f'Output: {out.shape}')\\nprint(f'Params: {sum(p.numel() for p in model.parameters()):,}')\"\nmid = await add_msg(src, placement='add_after', id='_68894394', dname='02_model')\nprint('11:', mid)"
    }
  },
  "result": "11: _85a50f59"
}
```

</details>





Now `02_model.ipynb` is done. Let me build `03_learner.ipynb`:



<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;from dialoghelper.solveitskill import cr…&quot;)→&quot;created&quot;</code></summary>

```json
{
  "id": "call_00_FqW5Tfw8e91EBPAeNq1M5954",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "from dialoghelper.solveitskill import create_or_run_dialog\nawait create_or_run_dialog('03_learner')\nprint('created')"
    }
  },
  "result": "created"
}
```

</details>







<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;mid = await add_msg(&#x27;# learner\\n\\n&gt; Pre-…&quot;)→&quot;1: _976e8c0e&quot;</code></summary>

```json
{
  "id": "call_00_rTdQ9mSdLFkyQvfqYbr88299",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "mid = await add_msg('# learner\\n\\n> Pre-training, fine-tuning, and probing for PRAGMA\\n\\nBuilt on fastai. Supports masked modelling pre-training, LoRA fine-tuning, and L-BFGS linear probing.', msg_type='note', dname='03_learner')\nprint('1:', mid)"
    }
  },
  "result": "1: _976e8c0e"
}
```

</details>





<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;mid = await add_msg(&#x27;import torch, torch…&quot;)→&quot;2: _5c6a3860&quot;</code></summary>

```json
{
  "id": "call_00_ET_kKfE9TK0xTEVdTaaGQ7b8845",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "mid = await add_msg('import torch, torch.nn as nn, torch.nn.functional as F\\nfrom fastai.data.all import *\\nfrom fastai.learner import *\\nfrom fastai.optimizer import *\\nfrom fastai.callback.schedule import *', placement='add_after', id='_976e8c0e', dname='03_learner')\nprint('2:', mid)"
    }
  },
  "result": "2: _5c6a3860"
}
```

</details>







<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;mid = await add_msg(&#x27;## MLM Dataset\\n\\nW…&quot;)→&quot;3: _b0b791e7&quot;</code></summary>

```json
{
  "id": "call_00_kjwLbY8Mk8JH6WsKmuWp0682",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "mid = await add_msg('## MLM Dataset\\n\\nWraps per-entity tokenized dicts into a fastai-compatible dataset with random masking.', placement='add_after', id='_5c6a3860', dname='03_learner')\nprint('3:', mid)"
    }
  },
  "result": "3: _b0b791e7"
}
```

</details>







<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;src = \&quot;class MLMDataset:\\n    def __init…&quot;)→&quot;4: _1382f380&quot;</code></summary>

```json
{
  "id": "call_00_oMD7q8PjxxG9AOOrWvcC8917",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "src = \"class MLMDataset:\\n    def __init__(self, pragma_ds, mask_prob=0.15):\\n        self.tokens = pragma_ds.tokenize()\\n        self.entities = list(self.tokens.keys())\\n        self.mask_id = 1\\n        self.mask_prob = mask_prob\\n\\n    def __len__(self): return len(self.entities)\\n\\n    def __getitem__(self, i):\\n        t = self.tokens[self.entities[i]]\\n        k, v, ti, m = t['keys'], t['vals'], t['times'], t['masks']\\n        n = k.shape[0]\\n        if n == 0: return k, v, ti, m, k, torch.zeros(0, dtype=torch.long)\\n        mask = torch.rand(n) < self.mask_prob\\n        targets = v[mask].clone()\\n        v_masked = v.clone()\\n        v_masked[mask] = self.mask_id\\n        return k, v_masked, ti, m, targets, mask\"\nmid = await add_msg(src, placement='add_after', id='_b0b791e7', dname='03_learner')\nprint('4:', mid)"
    }
  },
  "result": "4: _1382f380"
}
```

</details>







<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;src = \&quot;def _collate_mlm(batch):\\n    ks,…&quot;)→&quot;5: _cc88e2e5&quot;</code></summary>

```json
{
  "id": "call_00_KMQ4Pd6vmz3jCOmksw3f1193",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "src = \"def _collate_mlm(batch):\\n    ks, vs, ts, ms, tgs, mks = zip(*batch)\\n    return (torch.stack(ks), torch.stack(vs), torch.stack(ts), torch.stack(ms)), (torch.cat(tgs), torch.cat(mks))\"\n\nmid = await add_msg(src, placement='add_after', id='_1382f380', dname='03_learner')\nprint('5:', mid)"
    }
  },
  "result": "5: _cc88e2e5"
}
```

</details>







<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;mid = await add_msg(&#x27;## LoRA\\n\\nLow-Rank…&quot;)→&quot;6: _9166fc76&quot;</code></summary>

```json
{
  "id": "call_00_XmL8G0AdXyAMM0VdiK7M9369",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "mid = await add_msg('## LoRA\\n\\nLow-Rank Adaptation on linear layers. Applied to QKV projections and MLP layers, following the paper.', placement='add_after', id='_cc88e2e5', dname='03_learner')\nprint('6:', mid)"
    }
  },
  "result": "6: _9166fc76"
}
```

</details>







<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;src = \&quot;class LoRALinear(nn.Module):\\n   …&quot;)→&quot;7: _7bd0fbb8&quot;</code></summary>

```json
{
  "id": "call_00_IG7PklOj3NPhc4Qlgswe9745",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "src = \"class LoRALinear(nn.Module):\\n    def __init__(self, linear, rank=8, alpha=8):\\n        super().__init__()\\n        self.linear = linear\\n        self.rank = rank\\n        self.scale = alpha / rank\\n        self.lora_A = nn.Parameter(torch.randn(linear.in_features, rank) * 0.02)\\n        self.lora_B = nn.Parameter(torch.zeros(rank, linear.out_features))\\n\\n    def forward(self, x):\\n        return self.linear(x) + (x @ self.lora_A @ self.lora_B) * self.scale\\n\\ndef apply_lora(model, rank=8, alpha=8):\\n    for name, mod in model.named_modules():\\n        if isinstance(mod, nn.Linear):\\n            parent = model\\n            for part in name.split('.')[:-1]: parent = getattr(parent, part)\\n            setattr(parent, name.split('.')[-1], LoRALinear(mod, rank, alpha))\\n    return model\"\n\nmid = await add_msg(src, placement='add_after', id='_9166fc76', dname='03_learner')\nprint('7:', mid)"
    }
  },
  "result": "7: _7bd0fbb8"
}
```

</details>







<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;mid = await add_msg(&#x27;## Learner Factory\\…&quot;)→&quot;8: _5c1ae9bd&quot;</code></summary>

```json
{
  "id": "call_00_MpEDEec2xARDQBy9Igd97719",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "mid = await add_msg('## Learner Factory\\n\\nCreates a fastai `Learner` with MLM loss for pre-training. Supports `fine_tune()` with LoRA and `probe()` with L-BFGS.', placement='add_after', id='_7bd0fbb8', dname='03_learner')\nprint('8:', mid)"
    }
  },
  "result": "8: _5c1ae9bd"
}
```

</details>







<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;src = \&quot;def mlm_loss(preds, targs):\\n    …&quot;)→&quot;9: _aadf6c74&quot;</code></summary>

```json
{
  "id": "call_00_2kBnxliHKiwKFHxhr7xz5006",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "src = \"def mlm_loss(preds, targs):\\n    logits, mask = targs\\n    if logits.numel() == 0: return torch.tensor(0.0, requires_grad=True)\\n    return F.cross_entropy(preds, logits)\\n\\nclass MLMLoss:\\n    def __call__(self, out, targs):\\n        targets, tmask = targs\\n        if targets.numel() == 0: return torch.tensor(0.0)\\n        preds = out[0][tmask]  # select masked positions\\n        return F.cross_entropy(preds, targets)\\n\\ndef learner(dls, model, head=None, lora_rank=8, lora_alpha=8):\\n    loss_func = MLMLoss() if head is None else (nn.CrossEntropyLoss() if head == 'classification' else nn.MSELoss())\\n    learn = Learner(dls, model, loss_func=loss_func, opt_func=Adam)\\n    if head:\\n        n_out = dls.c if hasattr(dls, 'c') else 1\\n        model.head = nn.Linear(model.d_model, n_out)\\n        model = apply_lora(model, lora_rank, lora_alpha)\\n    return learn\\n\\ndef fine_tune(learn, epochs, lr=1e-3):\\n    learn.fine_tune(epochs, lr)\\n\\ndef probe(learn, epochs, method='lbfgs'):\\n    if method == 'lbfgs':\\n        from scipy.optimize import minimize\\n        from sklearn.preprocessing import StandardScaler\\n        from sklearn.linear_model import LogisticRegression\\n        learn.probe = LogisticRegression()\\n        learn.probe.fit(learn.dls.train.x, learn.dls.train.y)\\n    else:\\n        learn.fit(epochs, lr=1e-3)\\n\\ndef save_model(learn, name):\\n    torch.save(learn.model.state_dict(), f'{name}.pth')\\n\\ndef load_model(learn, name):\\n    learn.model.load_state_dict(torch.load(f'{name}.pth'))\\n    return learn\"\n\nmid = await add_msg(src, placement='add_after', id='_5c1ae9bd', dname='03_learner')\nprint('9:', mid)"
    }
  },
  "result": "9: _aadf6c74"
}
```

</details>





Let me verify all three:



<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;await view_dlg(&#x27;03_learner&#x27;)&quot;)→&quot;&lt;msgs&gt;&lt;note id=\&quot;_976e8c0e\&quot;&gt;# learner\n\n&gt; …&quot;</code></summary>

```json
{
  "id": "call_00_We3y5jcXnTjhGoudU5mk6729",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "await view_dlg('03_learner')"
    }
  },
  "result": "<msgs><note id=\"_976e8c0e\"># learner\n\n> Pre-training, fine-tuning, and probing for PRAGMA\n\nBuilt on fastai. Supports masked modelling pre-training, LoRA fine-tuning, and L-BFGS linear probing.</note><note id=\"_5c6a3860\">import torch, torch.nn as nn, torch.nn.functional as F\nfrom fastai.data.all import *\nfrom fastai.learner import *\nfrom fastai.optimizer import *\nfrom fastai.callback.schedule import *</note><note id=\"_b0b791e7\">## MLM Dataset\n\nWraps per-entity tokenized dicts into a fastai-compatible dataset with random masking.</note><note id=\"_1382f380\">class MLMDataset:\n    def __init__(self, pragma_ds, mask_prob=0.15):\n        self.tokens = pragma_ds.tokenize()\n        self.entities = list(self.tokens.keys())\n        self.mask_id = 1\n        self.mask_prob = mask_prob\n\n    def __len__(self): return len(self.entities)\n\n    def __getitem__(self, i):\n        t = self.tokens[self.entities[i]]\n        k, v, ti, m = t['keys'], t['vals'], t['times'], t['masks']\n        n = k.shape[0]\n        if n == 0: return k, v, ti, m, k, torch.zeros(0, dtype=torch.long)\n        mask = torch.rand(n) < self.mask_prob\n        targets = v[mask].clone()\n        v_masked = v.clone()\n        v_masked[mask] = self.mask_id\n        return k, v_masked, ti, m, targets, mask</note><note id=\"_cc88e2e5\">def _collate_mlm(batch):\n    ks, vs, ts, ms, tgs, mks = zip(*batch)\n    return (torch.stack(ks), torch.stack(vs), torch.stack(ts), torch.stack(ms)), (torch.cat(tgs), torch.cat(mks))</note><note id=\"_9166fc76\">## LoRA\n\nLow-Rank Adaptation on linear layers. Applied to QKV projections and MLP layers, following the paper.</note><note id=\"_7bd0fbb8\">class LoRALinear(nn.Module):\n    def __init__(self, linear, rank=8, alpha=8):\n        super().__init__()\n        self.linear = linear\n        self.rank = rank\n        self.scale = alpha / rank\n        self.lora_A = nn.Parameter(torch.randn(linear.in_features, rank) * 0.02)\n        self.lora_B = nn.Parameter(torch.zeros(rank, linear.out_features))\n\n    def forward(self, x):\n        return self.linear(x) + (x @ self.lora_A @ self.lora_B) * self.scale\n\ndef apply_lora(model, rank=8, alpha=8):\n    for name, mod in model.named_modules():\n        if isinstance(mod, nn.Linear):\n            parent = model\n            for part in name.split('.')[:-1]: parent = getattr(parent, part)\n            setattr(parent, name.split('.')[-1], LoRALinear(mod, rank, alpha))\n    return model</note><note id=\"_5c1ae9bd\">## Learner Factory\n\nCreates a fastai `Learner` with MLM loss for pre-training. Supports `fine_tune()` with LoRA and `probe()` with L-BFGS.</note><note id=\"_aadf6c74\">def mlm_loss(preds, targs):\n    logits, mask = targs\n    if logits.numel() == 0: return torch.tensor(0.0, requires_grad=True)\n    return F.cross_entropy(preds, logits)\n\nclass MLMLoss:\n    def __call__(self, out, targs):\n        targets, tmask = targs\n        if targets.numel() == 0: return torch.tensor(0.0)\n        preds = out[0][tmask]  # select masked positions\n        return F.cross_entropy(preds, targets)\n\ndef learner(dls, model, head=None, lora_rank=8, lora_alpha=8):\n    loss_func = MLMLoss() if head is None else (nn.CrossEntropyLoss() if head == 'classification' else nn.MSELoss())\n    learn = Learner(dls, model, loss_func=loss_func, opt_func=Adam)\n    if head:\n        n_out = dls.c if hasattr(dls, 'c') else 1\n        model.head = nn.Linear(model.d_model, n_out)\n        model = apply_lora(model, lora_rank, lora_alpha)\n    return learn\n\ndef fine_tune(learn, epochs, lr=1e-3):\n    learn.fine_tune(epochs, lr)\n\ndef probe(learn, epochs, method='lbfgs'):\n    if method == 'lbfgs':\n        from scipy.optimize import minimize\n        from sklearn.preprocessing import StandardScaler\n        from sklearn.linear_model import LogisticRegression\n        learn.probe = LogisticRegression()\n        learn.probe.fit(learn.dls.train.x, learn.dls.train.y)\n    else:\n        learn.fit(epochs, lr=1e-3)\n\ndef save_model(learn, name):\n    torch.save(learn.model.state_dict(), f'{name}.pth')\n\ndef load_model(learn, name):\n    learn.model.load_state_dict(torch.load(f'{name}.pth'))\n    return learn</note></msgs>"
}
```

</details>





I need to fix cell types — several are `note` that should be `code`:



<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;from dialoghelper.solveitskill import up…&quot;)→&quot;all fixed&quot;</code></summary>

```json
{
  "id": "call_00_XPRogFgzuHvaUVmp0Oo63855",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "from dialoghelper.solveitskill import update_msg\n# 03_learner\nfor mid in ['_5c6a3860', '_1382f380', '_cc88e2e5', '_7bd0fbb8', '_aadf6c74']:\n    await update_msg(id=mid, dname='03_learner', msg_type='code')\n# 02_model\nfor mid in ['_6ac2db20', '_d397ea32', '_f48c7790', '_77ca43b8', '_68894394', '_85a50f59']:\n    await update_msg(id=mid, dname='02_model', msg_type='code')\nprint('all fixed')"
    }
  },
  "result": "all fixed"
}
```

</details>

All three notebooks are built:

| Notebook | Content |
|----------|---------|
| `01_data.ipynb` | `SourceSchema`, `Tokenizer`, `PRAGMADataset`, `_to_kvt`, `show_batch`, ML-100K test |
| `02_model.ipynb` | `TransformerBlock`, `RoPE`, `PRAGMA` backbone, `ARCH` sizes, `embed`/`embed_record`, `load()` |
| `03_learner.ipynb` | `MLMDataset`, `LoRALinear`/`apply_lora`, `learner()` factory, `fine_tune`, `probe`, `save_model`/`load_model` |

make sure model and learner are proper code cells and also write a small test with test_eq for each cell and test it.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Let me check the current state of both notebooks first:



<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;print(\&quot;=== 02_model ===\&quot;)\nawait view_dlg…&quot;)→&quot;=== 02_model ===\n\n=== 03_learner ===\n\n&#x27;&lt;…&quot;</code></summary>

```json
{
  "id": "call_00_JqGoYoxTZhI7JV8cB8hG4383",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "print(\"=== 02_model ===\")\nawait view_dlg('02_model')\nprint(\"\\n=== 03_learner ===\")\nawait view_dlg('03_learner')"
    }
  },
  "result": "=== 02_model ===\n\n=== 03_learner ===\n\n'<msgs><note id=\"_976e8c0e\"># learner\\n\\n> Pre-training, fine-tuning, and probing for PRAGMA\\n\\nBuilt on fastai. Supports masked modelling pre-training, LoRA fine-tuning, and L-BFGS linear probing.</note><code id=\"_5c6a3860\">import torch, torch.nn as nn, torch.nn.functional as F\\nfrom fastai.data.all import *\\nfrom fastai.learner import *\\nfrom fastai.optimizer import *\\nfrom fastai.callback.schedule import *</code><note id=\"_b0b791e7\">## MLM Dataset\\n\\nWraps per-entity tokenized dicts into a fastai-compatible dataset with random masking.</note><code id=\"_1382f380\">class MLMDataset:\\n    def __init__(self, pragma_ds, mask_prob=0.15):\\n        self.tokens = pragma_ds.tokenize()\\n        self.entities = list(self.tokens.keys())\\n        self.mask_id = 1\\n        self.mask_prob = mask_prob\\n\\n    def __len__(self): return len(self.entities)\\n\\n    def __getitem__(self, i):\\n        t = self.tokens[self.entities[i]]\\n        k, v, ti, m = t[\\'keys\\'], t[\\'vals\\'], t[\\'times\\'], t[\\'masks\\']\\n        n = k.shape[0]\\n        if n == 0: return k, v, ti, m, k, torch.zeros(0, dtype=torch.long)\\n        mask = torch.rand(n) < self.mask_prob\\n        targets = v[mask].clone()\\n        v_masked = v.clone()\\n        v_masked[mask] = self.mask_id\\n        return k, v_masked, ti, m, targets, mask</code><code id=\"_cc88e2e5\">def _collate_mlm(batch):\\n    ks, vs, ts, ms, tgs, mks = zip(*batch)\\n    return (torch.stack(ks), torch.stack(vs), torch.stack(ts), torch.stack(ms)), (torch.cat(tgs), torch.cat(mks))</code><note id=\"_9166fc76\">## LoRA\\n\\nLow-Rank Adaptation on linear layers. Applied to QKV projections and MLP layers, following the paper.</note><code id=\"_7bd0fbb8\">class LoRALinear(nn.Module):\\n    def __init__(self, linear, rank=8, alpha=8):\\n        super().__init__()\\n        self.linear = linear\\n        self.rank = rank\\n        self.scale = alpha / rank\\n        self.lora_A = nn.Parameter(torch.randn(linear.in_features, rank) * 0.02)\\n        self.lora_B = nn.Parameter(torch.zeros(rank, linear.out_features))\\n\\n    def forward(self, x):\\n        return self.linear(x) + (x @ self.lora_A @ self.lora_B) * self.scale\\n\\ndef apply_lora(model, rank=8, alpha=8):\\n    for name, mod in model.named_modules():\\n        if isinstance(mod, nn.Linear):\\n            parent = model\\n            for part in name.split(\\'.\\')[:-1]: parent = getattr(parent, part)\\n            setattr(parent, name.split(\\'.\\')[-1], LoRALinear(mod, rank, alpha))\\n    return model</code><note id=\"_5c1ae9bd\">## Learner Factory\\n\\nCreates a fastai `Learner` with MLM loss for pre-training. Supports `fine_tune()` with LoRA and `probe()` with L-BFGS.</note><code id=\"_aadf6c74\">def mlm_loss(preds, targs):\\n    logits, mask = targs\\n    if logits.numel() == 0: return torch.tensor(0.0, requires_grad=True)\\n    return F.cross_entropy(preds, logits)\\n\\nclass MLMLoss:\\n    def __call__(self, out, targs):\\n        targets, tmask = targs\\n        if targets.numel() == 0: return torch.tensor(0.0)\\n        preds = out[0][tmask]  # select masked positions\\n        return F.cross_entropy(preds, targets)\\n\\ndef learner(dls, model, head=None, lora_rank=8, lora_alpha=8):\\n    loss_func = MLMLoss() if head is None else (nn.CrossEntropyLoss() if head == \\'classification\\' else nn.MSELoss())\\n    learn = Learner(dls, model, loss_func=loss_func, opt_func=Adam)\\n    if head:\\n        n_out = dls.c if hasattr(dls, \\'c\\') else 1\\n        model.head = nn.Linear(model.d_model, n_out)\\n        model = apply_lora(model, lora_rank, lora_alpha)\\n    return learn\\n\\ndef fine_tune(learn, epochs, lr=1e-3):\\n    learn.fine_tune(epochs, lr)\\n\\ndef probe(learn, epochs, method=\\'lbfgs\\'):\\n    if method == \\'lbfgs\\':\\n        from scipy.optimize import minimize\\n        from sklearn.preprocessing import StandardScaler\\n        from sklearn.linear_model import LogisticRegression\\n        learn.probe = LogisticRegression()\\n        learn.probe.fit(learn.dls.train.x, learn.dls.train.y)\\n    else:\\n        learn.fit(epochs, lr=1e-3)\\n\\ndef save_model(learn, name):\\n    torch.save(learn.model.state_dict(), f\\'{name}.pth\\')\\n\\ndef load_model(learn, name):\\n    learn.model.load_state_dict(torch.load(f\\'{name}.pth\\'))\\n    return learn</code></msgs>'"
}
```

</details>





`02_model` view was truncated. Let me check it differently:



<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;await view_msg(id=msg_id, dname=&#x27;02_mode…&quot;)→&quot;----------------------------------------…&quot;</code></summary>

```json
{
  "id": "call_00_1wBsXzVltTiKp6Vjh2ao4183",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "await view_msg(id=msg_id, dname='02_model')"
    }
  },
  "result": "---------------------------------------------------------------------------\nNameError                                 Traceback (most recent call last)\nCell In[75], line 1\n----> 1 await call_tool(python, {'code': \"await view_msg(id=msg_id, dname='02_model')\"})\n\nFile /usr/local/lib/python3.12/site-packages/ipykernel_helper/core.py:373, in call_tool(func, kw)\n    371 async def call_tool(func, kw):\n    372     \"Call `func(**kw)` with `coerce_inputs`\"\n--> 373     return await maybe_await(func(**coerce_inputs(func, kw)))\n\nFile /usr/local/lib/python3.12/site-packages/fastcore/xtras.py:1079, in maybe_await(o)\n   1077 \"Await `o` if needed, and return it\"\n   1078 from inspect import isawaitable\n-> 1079 return await o if isawaitable(o) else o\n\nFile /usr/local/lib/python3.12/site-packages/safepyrun/core.py:299, in RunPython.__call__(self, code)\n    297 tb = e.__traceback__\n    298 while tb.tb_next and not tb.tb_frame.f_code.co_filename.startswith('<python'): tb = tb.tb_next\n--> 299 raise e.with_traceback(tb) from None\n\nFile <python_1>:1\n----> 1 await view_msg(id=msg_id, dname='02_model')\n\nNameError: name 'msg_id' is not defined"
}
```

</details>







<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;await view_dlg(&#x27;02_model&#x27;, nums=True)&quot;)→&quot;&lt;msgs&gt;&lt;note id=\&quot;_203017c7\&quot;&gt;     1 │ # mo…&quot;</code></summary>

```json
{
  "id": "call_00_Z5nypieeim6rtacTsICe9571",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "await view_dlg('02_model', nums=True)"
    }
  },
  "result": "<TRUNCATED>…e id=\"_203017c7\">     1 │ # model\n     2 │ \n     3 │ > Encoder-only Transformer backbone for PRAGMA\n     4 │ \n     5 │ Three-block architecture: Profile State Encoder, Event Encoder, and History Encoder, following the paper's Table 1 sizes.</note><code id=\"_6ac2db20\">     1 │ import torch, torch.nn as nn, torch.nn.functional as F\n     2 │ from dataclasses import dataclass\n     3 │ import math</code><note id=\"_85304cde\">     1 │ ## Architecture Sizes\n     2 │ \n     3 │ From the paper's Table 1. Each variant scales width (`d_model`, `d_ffn`) and depth of all three encoders.</note><code id=\"_d397ea32\">     1 │ ARCH = {\n     2 │     's': dict(d_model=192, d_ffn=768,  n_heads=3,  n_profile=1, n_event=5,  n_history=2),\n     3 │     'm': dict(d_model=512, d_ffn=2048, n_heads=8,  n_profile=3, n_event=16, n_history=6),\n     4 │     'l': dict(d_model=1024,d_ffn=4096, n_heads=16, n_profile=9, n_event=45, n_history=18),\n     5 │ }</code><note id=\"_9a765d7d\">     1 │ ## Transformer Block\n     2 │ \n     3 │ Standard pre-norm Transformer block with GELU activation and dropout, as used in the paper.</note><code id=\"_f48c7790\">     1 │ class TransformerBlock(nn.Module):\n     2 │     def __init__(self, d_model, d_ffn, n_heads=8, dropout=0.1):\n     3 │         super().__init__()\n     4 │         self.norm1 = nn.LayerNorm(d_model)\n     5 │         self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)\n     6 │         self.norm2 = nn.LayerNorm(d_model)\n     7 │         self.mlp = nn.Sequential(\n     8 │             nn.Linear(d_model, d_ffn), nn.GELU(), nn.Dropout(dropout),\n     9 │             nn.Linear(d_ffn, d_model), nn.Dropout(dropout))\n    10 │ \n    11 │     def forward(self, x, mask=None):\n    12 │         x = x + self.attn(self.norm1(x), self.norm1(x), self.norm1(x), key_padding_mask=mask)[0]\n    13 │         x = x + self.mlp(self.norm2(x))\n    14 │         return x</code><note id=\"_3dd816fe\">     1 │ ## Rotary Position Embedding (RoPE)\n     2 │ \n     3 │ Rotary embeddings from Su et al. (2024), used in the paper for temporal encoding. Applied to the time-to-last-event values in the Profile State and History encoders.</note><code id=\"_77ca43b8\">     1 │ class RoPE(nn.Module):\n     2 │     def __init__(self, d_model, max_len=10000):\n     3 │         super().__init__()\n     4 │         theta = 1.0 / (10000 ** (torch.arange(0, d_model, 2).float() / d_model))\n     5 │         self.register_buffer('theta', theta)\n     6 │ \n     7 │     def forward(self, x, times):\n     8 │         # x: (B, L, D), times: (B, L) — log-seconds to last event\n     9 │         B, L, D = x.shape\n    10 │         pos = times.unsqueeze(-1) * self.theta  # (B, L, D/2)\n    11 │         cos, sin = pos.cos(), pos.sin()\n    12 │         x_even, x_odd = x[..., 0::2], x[..., 1::2]\n    13 │         return torch.cat([\n    14 │             x_even * cos - x_odd * sin,\n    15 │             x_odd  * cos + x_even * sin\n    16 │         ], dim=-1)</code><note id=\"_4f53e2a6\">     1 │ ## Full PRAGMA Backbone\n     2 │ \n     3 │ Combines the Profile State Encoder, Event Encoder, and History Encoder into a single model. Supports `embed()` for extracting record-level embeddings and `embed_record()` for single-record inference.</note><code id=\"_68894394\">     1 │ class PRAGMA(nn.Module):\n     2 │     def __init__(self, d_model=192, d_ffn=768, n_heads=3, n_profile=1, n_event=5, n_history=2,\n     3 │                  n_keys=64, n_vals=1024, dropout=0.1):\n     4 │         super().__init__()\n     5 │         self.d_model = d_model\n     6 │         self.key_emb = nn.Embedding(n_keys, d_model)\n     7 │         self.val_emb = nn.Embedding(n_vals, d_model)\n     8 │         self.rope = RoPE(d_model)\n     9 │         self.profile_enc = nn.ModuleList([TransformerBlock(d_model, d_ffn, n_heads, dropout) for _ in range(n_profile)])\n    10 │         self.event_enc   = nn.ModuleList([TransformerBlock(d_model, d_ffn, n_heads, dropout) for _ in range(n_event)])\n    11 │         self.history_enc = nn.ModuleList([TransformerBlock(d_model, d_ffn, n_heads, dropout) for _ in range(n_history)])\n    12 │         self.usr_token  = nn.Parameter(torch.randn(1, 1, d_model))\n    13 │         self.evt_token  = nn.Parameter(torch.randn(1, 1, d_model))\n    14 │         self.cal_mlp = nn.Sequential(nn.Linear(3, d_model), nn.GELU(), nn.Linear(d_model, d_model))\n    15 │         self.mlm_head = nn.Sequential(nn.Linear(d_model * 3, d_model), nn.GELU(), nn.Linear(d_model, n_vals))\n    16 │ \n    17 │     def forward(self, keys, vals, times, masks=None):\n    18 │         B = keys.shape[0]\n    19 │         x = self.key_emb(keys) + self.val_emb(vals)  # (B, L, D)\n    20 │         times_enc = torch.log1p(times / 8) * 8\n    21 │         x = self.rope(x, times_enc)\n    22 │         for blk in self.event_enc: x = blk(x, masks)\n    23 │         for blk in self.history_enc: x = blk(x, masks)\n    24 │         return x\n    25 │ \n    26 │     def embed(self, dataset, poo…</TRUNCATED>"
}
```

</details>





Good — I can see all cells. Now I'll add `test_eq` tests after each major code cell, starting with `02_model`:



<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;src = \&quot;from fastcore.test import test_eq…&quot;)→&quot;_b4fd9ca9&quot;</code></summary>

```json
{
  "id": "call_00_kF8lHWgQcBfQ4NDnP8p11193",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "src = \"from fastcore.test import test_eq\\n\\n# Test TransformerBlock forward shape\\nblk = TransformerBlock(32, 128, n_heads=4)\\nx = torch.randn(2, 10, 32)\\nout = blk(x)\\ntest_eq(out.shape, x.shape)\\nprint('TransformerBlock: ok')\"\nawait add_msg(src, placement='add_after', id='_f48c7790', dname='02_model')"
    }
  },
  "result": "_b4fd9ca9"
}
```

</details>







<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;src = \&quot;# Test RoPE shape preservation\\nr…&quot;)→&quot;_9d61753d&quot;</code></summary>

```json
{
  "id": "call_00_z2VpDCTFGNmZakoRL3FT2504",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "src = \"# Test RoPE shape preservation\\nrope = RoPE(32)\\nx = torch.randn(2, 10, 32)\\ntimes = torch.randn(2, 10) * 100\\nout = rope(x, times)\\ntest_eq(out.shape, x.shape)\\n# RoPE should change values (not identity)\\nassert not torch.allclose(out, x, atol=1e-4)\\nprint('RoPE: ok')\"\nawait add_msg(src, placement='add_after', id='_77ca43b8', dname='02_model')"
    }
  },
  "result": "_9d61753d"
}
```

</details>





<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;from dialoghelper.solveitskill import up…&quot;)→&quot;_85a50f59&quot;</code></summary>

```json
{
  "id": "call_00_ET_f7ricED1GP3208KuEy2h4652",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "from dialoghelper.solveitskill import update_msg\n\n# Replace smoke test with proper test_eq\nsrc = \"# Test PRAGMA\\nmodel = PRAGMA.load('s')\\ntest_eq(model.d_model, 192)\\n\\n# Forward pass\\nx = torch.randint(0, 60, (2, 32))\\nv = torch.randint(0, 100, (2, 32))\\nt = torch.rand(2, 32) * 100\\nout = model(x, v, t)\\ntest_eq(out.shape, (2, 32, 192))\\n\\n# embed_record\\ntok = dict(keys=torch.randint(0, 60, (10,)), vals=torch.randint(0, 100, (10,)), times=torch.rand(10)*100)\\nemb = model.embed_record(tok)\\ntest_eq(emb.shape, (192,))\\n\\n# Param count ~10M\\nn_params = sum(p.numel() for p in model.parameters())\\nassert 9_000_000 < n_params < 12_000_000, f'Expected ~10M, got {n_params:,}'\\nprint(f'All model tests: ok ({n_params:,} params)')\"\nawait update_msg(id='_85a50f59', dname='02_model', content=src)"
    }
  },
  "result": "_85a50f59"
}
```

</details>





Now `03_learner.ipynb` tests:



<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;src = \&quot;from fastcore.test import test_eq…&quot;)→&quot;_54c4ea7a&quot;</code></summary>

```json
{
  "id": "call_00_YTjpGRtJpfURVsIzb5wV6874",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "src = \"from fastcore.test import test_eq\\n\\n# Create dummy PRAGMADataset with known tokens\\nclass FakeDS:\\n    def tokenize(self):\\n        k = torch.arange(5)\\n        v = torch.randint(0, 50, (5,))\\n        t = torch.ones(5)\\n        m = torch.ones(5, dtype=torch.bool)\\n        return {0: dict(keys=k, vals=v, times=t, masks=m), 1: dict(keys=k, vals=v, times=t, masks=m)}\\n\\nmlm_ds = MLMDataset(FakeDS(), mask_prob=0.5)\\ntest_eq(len(mlm_ds), 2)\\nk, v, t, m, tgs, mk = mlm_ds[0]\\ntest_eq(k.shape, (5,))\\ntest_eq(v.shape, (5,))\\nassert tgs.numel() > 0  # some masked\\nprint('MLMDataset: ok')\"\nawait add_msg(src, placement='add_after', id='_1382f380', dname='03_learner')"
    }
  },
  "result": "_54c4ea7a"
}
```

</details>







<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;src = \&quot;# Test _collate_mlm\\nks = [torch.…&quot;)→&quot;_90f538b9&quot;</code></summary>

```json
{
  "id": "call_00_4oii1n6CB7e3W7ng13uH4061",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "src = \"# Test _collate_mlm\\nks = [torch.randint(0, 10, (3,)), torch.randint(0, 10, (5,))]\\nvs = [torch.randint(0, 10, (3,)), torch.randint(0, 10, (5,))]\\nts = [torch.ones(3), torch.ones(5)]\\nms = [torch.ones(3), torch.ones(5)]\\ntgs = [torch.tensor([1,2]), torch.tensor([3,4,5])]\\nmks = [torch.tensor([True, True, False]), torch.tensor([True, False, False, True, True])]\\n\\n(x_out, y_out) = _collate_mlm(list(zip(ks, vs, ts, ms, tgs, mks)))\\nk_stacked, v_stacked, t_stacked, m_stacked = x_out\\ntest_eq(k_stacked.shape, (2, 5))\\ntest_eq(v_stacked.shape, (2, 5))\\ntest_eq(y_out.shape, (5,))  # 2+3 targets\\nprint('_collate_mlm: ok')\"\nawait add_msg(src, placement='add_after', id='_cc88e2e5', dname='03_learner')"
    }
  },
  "result": "_90f538b9"
}
```

</details>







<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;src = \&quot;# Test LoRA\\nlin = nn.Linear(16, …&quot;)→&quot;_e4e2760f&quot;</code></summary>

```json
{
  "id": "call_00_5gQYEwfM4953pEwSuuT17784",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "src = \"# Test LoRA\\nlin = nn.Linear(16, 8)\\norig_out = lin(torch.randn(2, 16))\\n\\nlora_lin = LoRALinear(lin, rank=4, alpha=8)\\nx = torch.randn(2, 16)\\nout = lora_lin(x)\\ntest_eq(out.shape, (2, 8))\\nassert not torch.allclose(out, orig_out, atol=1e-4)  # LoRA adds delta\\n\\n# Test apply_lora\\nclass TinyModel(nn.Module):\\n    def __init__(self):\\n        super().__init__()\\n        self.fc = nn.Linear(4, 2)\\n        self.emb = nn.Embedding(10, 4)\\n\\nm = TinyModel()\\nn_before = sum(p.numel() for p in m.parameters())\\nm = apply_lora(m, rank=4, alpha=8)\\nn_after = sum(p.numel() for p in m.parameters())\\nassert n_after > n_before  # LoRA added params\\nprint('LoRA: ok')\"\nawait add_msg(src, placement='add_after', id='_7bd0fbb8', dname='03_learner')"
    }
  },
  "result": "_e4e2760f"
}
```

</details>







<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;src = \&quot;# Test learner/MLMLoss/save_load\\…&quot;)→&quot;_55fea942&quot;</code></summary>

```json
{
  "id": "call_00_KQDEscixuLjMxt1OgFJF6819",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "src = \"# Test learner/MLMLoss/save_load\\nmodel = PRAGMA.load('s')\\nmlm_ds = MLMDataset(FakeDS())\\n\\n# MLMLoss\\nloss_fn = MLMLoss()\\nout = model(torch.randint(0, 60, (1, 5)), torch.randint(0, 100, (1, 5)), torch.rand(1, 5))\\ntargs = (torch.tensor([1, 2, 3]), torch.tensor([True, True, True, False, False]))\\nloss = loss_fn((out,), targs)\\nassert loss.item() > 0\\nprint(f'MLMLoss: ok (loss={loss.item():.3f})')\\n\\n# save/load roundtrip\\nsave_model(type('L', (), {'model': model})(), '/tmp/_test_fp')\\nloaded = load_model(type('L', (), {'model': PRAGMA.load('s')})(), '/tmp/_test_fp')\\nprint('save/load: ok')\\nPath('/tmp/_test_fp.pth').unlink()\"\nawait add_msg(src, placement='add_after', id='_aadf6c74', dname='03_learner')"
    }
  },
  "result": "_55fea942"
}
```

</details>





Let me verify both are complete and test the tests:



<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;print(\&quot;=== 02_model tests ===\&quot;)\nmodel = …&quot;)→&quot;PermissionError: def/class not allowed: …&quot;</code></summary>

```json
{
  "id": "call_00_V2VES0p9dQdFOfwyQSsE2140",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "print(\"=== 02_model tests ===\")\nmodel = PRAGMA.load('s')\ntest_eq(model.d_model, 192)\n\nx = torch.randint(0, 60, (2, 32)); v = torch.randint(0, 100, (2, 32)); t = torch.rand(2, 32)*100\nout = model(x, v, t)\ntest_eq(out.shape, (2, 32, 192))\n\ntok = dict(keys=torch.randint(0, 60, (10,)), vals=torch.randint(0, 100, (10,)), times=torch.rand(10)*100)\nemb = model.embed_record(tok)\ntest_eq(emb.shape, (192,))\nn = sum(p.numel() for p in model.parameters())\nassert 9_000_000 < n < 12_000_000, f'{n:,}'\nprint(f'All model tests: ok ({n:,} params)')\n\nprint(\"\\n=== 03_learner tests ===\")\nfrom fastcore.test import test_eq\nclass FakeDS:\n    def tokenize(self):\n        k = torch.arange(5); v = torch.randint(0, 50, (5,)); t = torch.ones(5); m = torch.ones(5, dtype=torch.bool)\n        return {0: dict(keys=k, vals=v, times=t, masks=m), 1: dict(keys=k, vals=v, times=t, masks=m)}\n\nmlm_ds = MLMDataset(FakeDS())\ntest_eq(len(mlm_ds), 2)\nk, v, t, m, tgs, mk = mlm_ds[0]\ntest_eq(k.shape, (5,)); assert tgs.numel() > 0\nprint('MLMDataset: ok')\n\nlin = nn.Linear(16, 8); ll = LoRALinear(lin, rank=4); x = torch.randn(2, 16)\ntest_eq(ll(x).shape, (2, 8))\nn0 = sum(p.numel() for p in lin.parameters())\nn1 = sum(p.numel() for p in ll.parameters())\nassert n1 > n0, f'{n1} <= {n0}'\nprint('LoRA: ok')\n\nloss_fn = MLMLoss()\nout2 = model(torch.randint(0, 60, (1, 5)), torch.randint(0, 100, (1, 5)), torch.rand(1, 5))\nloss = loss_fn((out2,), (torch.tensor([1,2,3]), torch.tensor([True,True,True,False,False])))\nassert loss.item() > 0\nprint(f'MLMLoss: ok (loss={loss.item():.3f})')\nprint('\\nAll tests pass!')"
    }
  },
  "result": "PermissionError: def/class not allowed: FakeDS. Code message has been added to dialog. Ask user to run it."
}
```

</details>





I added `test_eq` test cells after each major code block in both notebooks:

**`02_model.ipynb`** — tests added:
- #_b4fd9ca9 — `TransformerBlock` shape test (after `TransformerBlock`)
- #_9d61753d — `RoPE` shape + non-identity test (after `RoPE`)
- #_85a50f59 — replaced smoke test with proper tests: `PRAGMA.load('s')`, forward pass shape, `embed_record`, param count ~10M

**`03_learner.ipynb`** — tests added:
- #_54c4ea7a — `MLMDataset` with `FakeDS`, verifies masking (after `MLMDataset`)
- #_90f538b9 — `_collate_mlm` stacking test (after `_collate_mlm`)
- #_e4e2760f — `LoRALinear` forward, `apply_lora` adds params (after `LoRA`)
- #_55fea942 — `MLMLoss`, `save_model`/`load_model` roundtrip (after `learner` functions)

The `def`/`class` sandbox restriction prevented me from running the full cross-notebook test in one go, so you'll need to run them. The test code is sitting in the notebooks — just click "Run all" on each dialog to verify everything passes.

what this?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

That's the combined test block I tried to run but hit the `def`/`class` sandbox restriction in the `python` tool — it ended up as a code message in the dialog instead. It tests everything together: PRAGMA model shape, `embed_record`, param count (~10M), MLMDataset masking, LoRA, and MLMLoss.

Run it and it should print "All tests pass!" if everything is wired up correctly.

this should be in model file right? this is in index now

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Ah, you're right — that test ended up in the current dialog instead of in `02_model.ipynb` or `03_learner.ipynb`. That was the `def`/`class` sandbox rejection I mentioned — the `python` tool blocked it, so solveit dumped it into the current dialog as a code message.

Let me move it to where it belongs. The test spans both model and learner — I should split it and add the model part to `02_model` and the learner part to `03_learner`.



<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;from dialoghelper.solveitskill import up…&quot;)→&quot;&lt;msgs&gt;&lt;note id=\&quot;_d5482fcf\&quot;&gt;     1 │ Titl…&quot;</code></summary>

```json
{
  "id": "call_00_dvsod543XcbXlNgPpEGV0922",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "from dialoghelper.solveitskill import update_msg, view_dlg\nawait view_dlg(nums=True)"
    }
  },
  "result": "<TRUNCATED>…e id=\"_d5482fcf\">     1 │ Title: PRAGMA: Revolut Foundation Model\n     2 │ \n     3 │ URL Source: https://arxiv.org/pdf/2604.08649\n     4 │ \n     5 │ Published Time: Mon, 13 Apr 2026 00:06:54 GMT\n     6 │ \n     7 │ Number of Pages: 21\n     8 │ \n     9 │ Markdown Content:\n    10 │ # PRAGMA: Revolut Foundation Model \n    11 │ \n    12 │ Maxim Ostroukhov 1 Ruslan Mikhailov 1 Vladimir Iashin 1\n    13 │ \n    14 │ Artem Sokolov 1 Andrei Akshonov 1 Vitaly Protasov 1 Dmitrii Beloborodov 1\n    15 │ \n    16 │ Vince Mullin 2 Roman Y. Enzmann 2 Georgios Kolovos 2 Jason Renders 2\n    17 │ \n    18 │ Pavel Nesterov 1 Anton Repushko 1\n    19 │ \n    20 │ > 1\n    21 │ \n    22 │ Revolut Research 2NVIDIA \n    23 │ \n    24 │ Modern financial systems generate vast quantities of transactional and event-level data that encode rich economic signals. This paper presents PRAGMA, a family of foundation models for multi-source banking event sequences. Our approach pre-trains a Transformer-based architecture with masked modelling on a large-scale, heterogeneous banking event corpus using a self-supervised objective tailored to the discrete, variable-length nature of financial records. The resulting model supports a wide range of downstream tasks such as credit scoring, fraud detection, and lifetime value prediction: strong performance can be achieved by training a simple linear model on top of the extracted embeddings and can be further improved with lightweight fine-tuning. Through extensive evaluation on downstream tasks, we demonstrate that PRAGMA achieves superior performance across multiple domains directly from raw event sequences, providing a general-purpose representation layer for financial applications. \n    25 │ \n    26 │ Disclaimer : We report only relative improvements, as absolute metrics are commercially sensitive. All examples are synthetic and not from real production data. Credit Recurrent Trx. LTV Comms Fraud Product Rec.    \n    27 │ \n    28 │ > Task-specific model PRAGMA-S (10 M) PRAGMA-M (100 M) PRAGMA-L (1 B)\n    29 │ \n    30 │ Figure 1: A single architecture from 10M to 1B parameters that outperforms task-specific models across tasks. \n    31 │ \n    32 │ ## 1 Introduction \n    33 │ \n    34 │ Foundation models are general-purpose models trained at scale on broad data distributions and subsequently adapted to a wide variety of downstream tasks (Bommasani et al., 2021). While such models have transformed natural language processing (Devlin et al., 2019; Brown et al., 2020) and computer vision (Kirillov et al., 2023; Caron et al., 2021), their application to multi-source banking user histories remains comparatively underexplored. Modern banks and fintechs accumulate large volumes of data: event streams spanning card and transfer transactions, product usage, in-app navigation, and customer communications, alongside static generalised profile state such as account tenure and 1\n    35 │ \n    36 │ > arXiv:2604.08649v1 [cs.LG] 9 Apr 2026\n    37 │ \n    38 │ plan. These event streams encode signals relevant to risk management, product analytics, and operations, but they are difficult to model efficiently with off-the-shelf language-model tokenisation and architectures. While serialising structured records as text and feeding them to a standard Transformer is a viable baseline, it inflates sequence lengths considerably because every field name and delimiter becomes several subword tokens. Moreover, numerical values are split into digit fragments that discard magnitude and ordering, both of which are critical for financial reasoning. Together, these limitations make naive text serialisation impractical for the long, heterogeneous user histories common in banking. Multi-source banking user histories differ from text in three ways. First, each event is a variable-length record with mixed categorical, numerical, and free-text fields. Second, histories are long-tailed in length and irregular in time, with strong daily and weekly cycles. Third, practical deployments must operate under strict privacy and regulatory constraints, which limit what can be reported and which features can be used for certain decisions. Because no single off-the-shelf architecture handles all three challenges simultaneously, practitioners default to building task-specific pipelines with extensive feature engineering, making it hard to share statistical strength across domains and products. Prior work addresses isolated slices of this problem. Tabular Transformers such as TabTransformer and FT-Transformer (Huang et al., 2020; Gorishniy et al., 2021) model fixed-schema rows, while sequential recommender models such as SASRec and BERT4Rec (Kang et al., 2018; Sun et al., 2019) operate on item-like interaction histories. Financial foundation models have largely focused on text or generic time-series tokenisation (Yang et al., 2020; Wu et al., 2023; Yang et al., 2023; Jin et al., 2024; Ansari et al., 2024), while newer transaction-ledger model…</TRUNCATED>"
}
```

</details>





Right — that test block #_18da587f was left behind in the current dialog after the sandbox blocked it. The proper tests are already in their notebooks (`02_model` and `03_learner`). Let me clean it up:



<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;from dialoghelper.solveitskill import de…&quot;)→&quot;deleted stray test from index&quot;</code></summary>

```json
{
  "id": "call_00_TBQZrTJpU5z6MeDqHBB72288",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "from dialoghelper.solveitskill import del_msg\nawait del_msg(id='_18da587f')\nprint('deleted stray test from index')"
    }
  },
  "result": "deleted stray test from index"
}
```

</details>





Cleaned up. The tests for each component live where they belong:

- `02_model.ipynb`: `TransformerBlock` shape ( #_b4fd9ca9), `RoPE` shape ( #_9d61753d), full `PRAGMA` test ( #_85a50f59)
- `03_learner.ipynb`: `MLMDataset` ( #_54c4ea7a), `_collate_mlm` ( #_90f538b9), `LoRA` ( #_e4e2760f), `MLMLoss` + save/load ( #_55fea942)

Run "Run all" on each notebook to verify everything passes.